In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Phase 1 - Vietnamese Clinical NER (ViHealthBERT)
=================================================

Trains the five entity types required by the medical-text task:
    TRIỆU_CHỨNG
    TÊN_XÉT_NGHIỆM
    KẾT_QUẢ_XÉT_NGHIỆM
    CHẨN_ĐOÁN
    THUỐC

Expected JSONL record:
{
  "id": "...",
  "text": "raw text",
  "entities": [
    {"text": "...", "type": "THUỐC", "position": [start, end]}
  ]
}

Positions are [start, end), end-exclusive.

Important implementation detail
-------------------------------
ViHealthBERT uses the PhoBERT tokenizer family and does not require a fast
Tokenizer in this script. We preserve raw character positions by:
  1) splitting raw text into Unicode surface pieces with exact char offsets,
  2) assigning BIO labels to those pieces,
  3) tokenizing each piece into model subwords,
  4) propagating the piece BIO label to subwords.

Thus no VnCoreNLP re-segmentation is required for this Phase-1 baseline and
character spans remain tied to the original raw text.

Best checkpoint selection:
    exact-span macro entity F1 on valid_concept.

Example:
    python train_phase1_ner.py

CLI paths can override the constants below.
"""

import os
import gc
import re
import json
import math
import time
import random
import argparse
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)


# ============================================================
# 0. PATHS - EDIT THIS BLOCK
# ============================================================

# Change DATA_ROOT to the Kaggle dataset directory containing the JSONL files.
DATA_ROOT = "/content/drive/MyDrive/data"

TRAIN_JSONL = f"{DATA_ROOT}/ner_train.jsonl"
VALID_TEMPLATE_JSONL = f"{DATA_ROOT}/ner_valid_template.jsonl"
VALID_CONCEPT_JSONL = f"{DATA_ROOT}/ner_valid_concept.jsonl"

# Kaggle Internet ON:
MODEL_NAME_OR_PATH = "demdecuong/vihealthbert-base-syllable"
LOCAL_FILES_ONLY = False

# Kaggle Internet OFF: add the model as a Kaggle Input, then use e.g.
# MODEL_NAME_OR_PATH = "/kaggle/input/vihealthbert-base-syllable"
# LOCAL_FILES_ONLY = True

OUTPUT_DIR = "/content/drive/MyDrive/output"


# ============================================================
# 1. TRAIN CONFIG
# ============================================================

SEED = 42

# ViHealthBERT/PhoBERT-family checkpoints have a short context window.
# The script additionally caps this against the loaded model config.
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 8
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

NUM_WORKERS = 2
PIN_MEMORY = True
USE_AMP = True

EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-4


# ============================================================
# 2. LABELS
# ============================================================

ENTITY_TYPES = [
    "TRIỆU_CHỨNG",
    "TÊN_XÉT_NGHIỆM",
    "KẾT_QUẢ_XÉT_NGHIỆM",
    "CHẨN_ĐOÁN",
    "THUỐC",
]

TYPE_TO_TAG = {
    "TRIỆU_CHỨNG": "TRIEU_CHUNG",
    "TÊN_XÉT_NGHIỆM": "TEN_XET_NGHIEM",
    "KẾT_QUẢ_XÉT_NGHIỆM": "KET_QUA_XET_NGHIEM",
    "CHẨN_ĐOÁN": "CHAN_DOAN",
    "THUỐC": "THUOC",
}
TAG_TO_TYPE = {v: k for k, v in TYPE_TO_TAG.items()}

LABEL_LIST = ["O"]
for _etype in ENTITY_TYPES:
    _tag = TYPE_TO_TAG[_etype]
    LABEL_LIST += [f"B-{_tag}", f"I-{_tag}"]

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
IGNORE_INDEX = -100


# ============================================================
# 3. UTILITIES
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--train_jsonl", type=str, default=TRAIN_JSONL)
    p.add_argument(
        "--valid_template_jsonl",
        type=str,
        default=VALID_TEMPLATE_JSONL,
    )
    p.add_argument(
        "--valid_concept_jsonl",
        type=str,
        default=VALID_CONCEPT_JSONL,
    )
    p.add_argument(
        "--model_name_or_path",
        type=str,
        default=MODEL_NAME_OR_PATH,
    )
    p.add_argument("--output_dir", type=str, default=OUTPUT_DIR)

    p.add_argument("--max_length", type=int, default=MAX_LENGTH)
    p.add_argument("--train_batch_size", type=int, default=TRAIN_BATCH_SIZE)
    p.add_argument("--eval_batch_size", type=int, default=EVAL_BATCH_SIZE)
    p.add_argument("--grad_accum_steps", type=int, default=GRAD_ACCUM_STEPS)

    p.add_argument("--lr", type=float, default=LEARNING_RATE)
    p.add_argument("--weight_decay", type=float, default=WEIGHT_DECAY)
    p.add_argument("--epochs", type=int, default=NUM_EPOCHS)
    p.add_argument("--warmup_ratio", type=float, default=WARMUP_RATIO)
    p.add_argument("--seed", type=int, default=SEED)

    # CLI --local_files_only can force offline loading.
    p.add_argument(
        "--local_files_only",
        action="store_true",
        default=LOCAL_FILES_ONLY,
    )

    # Colab/Jupyter injects arguments such as:
    #   -f /root/.local/share/jupyter/runtime/kernel-xxxx.json
    # parse_known_args() safely ignores those notebook-only arguments while
    # preserving all CLI arguments defined above.
    args, unknown = p.parse_known_args()

    if unknown:
        print(
            "[INFO] Ignoring unknown Jupyter/Colab arguments:",
            unknown,
        )

    return args


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Fast enough for Kaggle while maintaining repeatable seeds.
    torch.backends.cudnn.benchmark = True


def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing file: {p}\n"
            "Edit DATA_ROOT/PATHS at the top of the script or use CLI overrides."
        )

    rows = []
    with p.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise ValueError(f"Invalid JSON at {p}:{line_no}: {exc}")
    return rows


def validate_record(row: Dict[str, Any], source: str, idx: int):
    if "text" not in row or "entities" not in row:
        raise ValueError(f"{source}[{idx}] must contain text + entities")

    text = row["text"]
    spans = []

    for ent in row["entities"]:
        etype = ent["type"]
        if etype not in ENTITY_TYPES:
            raise ValueError(f"Unknown type {etype!r} at {source}[{idx}]")

        start, end = map(int, ent["position"])
        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Invalid span {ent['position']} at {source}[{idx}] len={len(text)}"
            )

        sliced = text[start:end]
        if sliced != ent["text"]:
            raise ValueError(
                f"Span mismatch at {source}[{idx}]\n"
                f"position=[{start},{end})\n"
                f"entity={ent['text']!r}\n"
                f"slice ={sliced!r}"
            )

        spans.append((start, end, etype))

    spans.sort()
    for a, b in zip(spans, spans[1:]):
        if a[1] > b[0]:
            raise ValueError(f"Overlapping gold entities: {a} vs {b}")


# ============================================================
# 4. RAW-TEXT SURFACE SEGMENTATION
# ============================================================

# Unicode-aware: \w includes Vietnamese letters and digits.
# This deliberately splits punctuation so strings such as WBC:14,43 become
# pieces WBC / : / 14 / , / 43, allowing separate test-name/result entities.
SURFACE_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def surface_pieces(text: str) -> List[Dict[str, Any]]:
    pieces = []
    for m in SURFACE_RE.finditer(text):
        pieces.append(
            {
                "text": m.group(0),
                "start": int(m.start()),
                "end": int(m.end()),
            }
        )
    return pieces


def assign_piece_bio_labels(
    pieces: List[Dict[str, Any]],
    entities: List[Dict[str, Any]],
) -> List[str]:
    """Assign one BIO label to each surface piece via character overlap."""

    entities = sorted(
        entities,
        key=lambda x: (int(x["position"][0]), int(x["position"][1])),
    )

    labels = []
    previous_entity_idx = None

    for piece in pieces:
        ps, pe = piece["start"], piece["end"]
        matched_idx = None

        for ent_idx, ent in enumerate(entities):
            es, ee = map(int, ent["position"])
            if ps < ee and pe > es:
                matched_idx = ent_idx
                break

        if matched_idx is None:
            labels.append("O")
            previous_entity_idx = None
            continue

        ent = entities[matched_idx]
        tag = TYPE_TO_TAG[ent["type"]]

        prefix = "I" if previous_entity_idx == matched_idx else "B"
        labels.append(f"{prefix}-{tag}")
        previous_entity_idx = matched_idx

    return labels


# ============================================================
# 5. TOKENIZATION WITH MANUAL PIECE -> SUBWORD ALIGNMENT
# ============================================================

def tokenize_piece(tokenizer, piece: str) -> List[int]:
    ids = tokenizer.encode(piece, add_special_tokens=False)
    if not ids:
        if tokenizer.unk_token_id is None:
            raise RuntimeError(f"Tokenizer produced no IDs for piece {piece!r}")
        ids = [int(tokenizer.unk_token_id)]
    return [int(x) for x in ids]


def inside_label(label: str) -> str:
    if label == "O":
        return "O"
    _, tag = label.split("-", 1)
    return f"I-{tag}"


class ClinicalNERDataset(Dataset):
    """
    Entity-aware chunking dataset.

    One original clinical note may produce multiple model features because
    ViHealthBERT has a ~256-token context window. Chunks keep GLOBAL raw-text
    character offsets, so exact-span evaluation remains valid.

    Important:
    - We never intentionally split a gold entity across two chunks.
    - No validation entity is silently discarded.
    - There is no overlap between chunks, so entities are not double-counted.
    """

    def __init__(
        self,
        path: str,
        tokenizer,
        max_length: int,
        name: str,
    ):
        self.path = path
        self.name = name
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.records = read_jsonl(path)

        print(f"[{name}] validating {len(self.records):,} records...")
        for i, row in enumerate(self.records):
            validate_record(row, name, i)

        self.features = []
        self.chunked_doc_count = 0
        self.max_chunks_per_doc = 1

        total_gold = sum(len(r["entities"]) for r in self.records)
        chunk_gold = 0

        for doc_idx, row in enumerate(self.records):
            chunks = self._encode_record_chunks(row, doc_idx)

            if len(chunks) > 1:
                self.chunked_doc_count += 1
                self.max_chunks_per_doc = max(
                    self.max_chunks_per_doc,
                    len(chunks),
                )

            self.features.extend(chunks)
            chunk_gold += sum(
                len(x["gold_entities"])
                for x in chunks
            )

        # This must be exact: every original gold entity must occur in exactly
        # one chunk.
        if chunk_gold != total_gold:
            raise RuntimeError(
                f"[{name}] chunking integrity error: "
                f"original gold={total_gold:,}, "
                f"chunk gold={chunk_gold:,}"
            )

        print(
            f"[{name}] original_docs={len(self.records):,} | "
            f"model_chunks={len(self.features):,} | "
            f"chunked_docs={self.chunked_doc_count:,} | "
            f"max_chunks/doc={self.max_chunks_per_doc} | "
            f"gold_entities_preserved={chunk_gold:,}/{total_gold:,}"
        )

    def _piece_entity_indices(self, pieces, entities):
        entities = sorted(
            entities,
            key=lambda x: (
                int(x["position"][0]),
                int(x["position"][1]),
            ),
        )

        out = []
        for piece in pieces:
            ps, pe = piece["start"], piece["end"]
            matched = -1

            for ent_idx, ent in enumerate(entities):
                es, ee = map(int, ent["position"])
                if ps < ee and pe > es:
                    matched = ent_idx
                    break

            out.append(matched)

        return entities, out

    def _encode_record_chunks(self, row, doc_idx):
        text = row["text"]
        pieces = surface_pieces(text)

        if not pieces:
            # Empty/no-token note: one minimal feature.
            bos_id = self.tokenizer.bos_token_id
            eos_id = self.tokenizer.eos_token_id
            if bos_id is None:
                bos_id = self.tokenizer.cls_token_id
            if eos_id is None:
                eos_id = self.tokenizer.sep_token_id

            return [{
                "input_ids": [int(bos_id), int(eos_id)],
                "attention_mask": [1, 1],
                "labels": [IGNORE_INDEX, IGNORE_INDEX],
                "piece_ids": [-1, -1],
                "piece_offsets": [],
                "doc_idx": int(doc_idx),
                "chunk_idx": 0,
                "gold_entities": [],
            }]

        entities, piece_ent_idx = self._piece_entity_indices(
            pieces,
            row["entities"],
        )

        # BIO label per surface piece.
        piece_labels = []
        previous_entity_idx = None

        for ent_idx in piece_ent_idx:
            if ent_idx < 0:
                piece_labels.append("O")
                previous_entity_idx = None
                continue

            tag = TYPE_TO_TAG[
                entities[ent_idx]["type"]
            ]

            prefix = (
                "I"
                if previous_entity_idx == ent_idx
                else "B"
            )

            piece_labels.append(
                f"{prefix}-{tag}"
            )
            previous_entity_idx = ent_idx

        # Tokenize every piece once.
        piece_sub_ids = [
            tokenize_piece(
                self.tokenizer,
                p["text"],
            )
            for p in pieces
        ]

        content_limit = self.max_length - 2
        if content_limit <= 0:
            raise RuntimeError(
                f"max_length={self.max_length} is too small."
            )

        # Verify no single gold entity itself exceeds model capacity.
        ent_token_counts = defaultdict(int)

        for sub_ids, ent_idx in zip(
            piece_sub_ids,
            piece_ent_idx,
        ):
            if ent_idx >= 0:
                ent_token_counts[ent_idx] += len(sub_ids)

        too_long = [
            (idx, n)
            for idx, n in ent_token_counts.items()
            if n > content_limit
        ]

        if too_long:
            idx, n = too_long[0]
            ent = entities[idx]
            raise RuntimeError(
                f"[{self.name}] A single gold entity is longer than "
                f"the model context: tokens={n}, limit={content_limit}, "
                f"entity={ent['text']!r}"
            )

        # --------------------------------------------------------
        # Build non-overlapping, entity-safe piece ranges.
        # --------------------------------------------------------
        ranges = []
        start_piece = 0
        n_pieces = len(pieces)

        while start_piece < n_pieces:
            token_count = 0
            end_piece = start_piece

            while end_piece < n_pieces:
                need = len(
                    piece_sub_ids[end_piece]
                )

                if (
                    token_count + need
                    > content_limit
                ):
                    break

                token_count += need
                end_piece += 1

            # At least one piece must fit.
            if end_piece == start_piece:
                raise RuntimeError(
                    f"[{self.name}] Surface piece exceeds context limit: "
                    f"{pieces[start_piece]['text']!r}"
                )

            # If the tentative boundary cuts an entity, move the boundary
            # backwards to the first piece of that entity.
            if end_piece < n_pieces:
                left_ent = piece_ent_idx[
                    end_piece - 1
                ]
                right_ent = piece_ent_idx[
                    end_piece
                ]

                if (
                    left_ent >= 0
                    and left_ent == right_ent
                ):
                    entity_start_piece = (
                        end_piece - 1
                    )

                    while (
                        entity_start_piece
                        > start_piece
                        and piece_ent_idx[
                            entity_start_piece - 1
                        ] == left_ent
                    ):
                        entity_start_piece -= 1

                    # If the current chunk consists only of this entity,
                    # the earlier capacity check guarantees the entity fits
                    # from start_piece, so this branch should not fail.
                    if entity_start_piece == start_piece:
                        # Rebuild this chunk to include the entire entity.
                        entity_end_piece = end_piece

                        while (
                            entity_end_piece
                            < n_pieces
                            and piece_ent_idx[
                                entity_end_piece
                            ] == left_ent
                        ):
                            entity_end_piece += 1

                        total = sum(
                            len(piece_sub_ids[j])
                            for j in range(
                                start_piece,
                                entity_end_piece,
                            )
                        )

                        if total > content_limit:
                            raise RuntimeError(
                                f"[{self.name}] Entity-safe chunking failed "
                                f"for {entities[left_ent]['text']!r}"
                            )

                        end_piece = entity_end_piece
                    else:
                        end_piece = entity_start_piece

            ranges.append(
                (start_piece, end_piece)
            )
            start_piece = end_piece

        # --------------------------------------------------------
        # Encode every chunk with GLOBAL char offsets.
        # --------------------------------------------------------
        bos_id = self.tokenizer.bos_token_id
        eos_id = self.tokenizer.eos_token_id

        if bos_id is None:
            bos_id = self.tokenizer.cls_token_id
        if eos_id is None:
            eos_id = self.tokenizer.sep_token_id

        if bos_id is None or eos_id is None:
            raise RuntimeError(
                "Tokenizer must provide BOS/CLS and EOS/SEP tokens"
            )

        chunks = []

        for chunk_idx, (
            p_start,
            p_end,
        ) in enumerate(ranges):

            content_ids = []
            token_labels = []
            token_piece_ids = []

            local_piece_offsets = [
                (
                    int(pieces[j]["start"]),
                    int(pieces[j]["end"]),
                )
                for j in range(
                    p_start,
                    p_end,
                )
            ]

            for local_piece_idx, j in enumerate(
                range(p_start, p_end)
            ):
                sub_ids = piece_sub_ids[j]
                label = piece_labels[j]

                first_label_id = (
                    LABEL2ID[label]
                )
                later_label_id = (
                    LABEL2ID[
                        inside_label(label)
                    ]
                )

                for sub_idx, sub_id in enumerate(
                    sub_ids
                ):
                    content_ids.append(
                        int(sub_id)
                    )
                    token_labels.append(
                        first_label_id
                        if sub_idx == 0
                        else later_label_id
                    )
                    token_piece_ids.append(
                        local_piece_idx
                    )

            input_ids = (
                [int(bos_id)]
                + content_ids
                + [int(eos_id)]
            )

            attention_mask = [
                1
            ] * len(input_ids)

            token_labels = (
                [IGNORE_INDEX]
                + token_labels
                + [IGNORE_INDEX]
            )

            token_piece_ids = (
                [-1]
                + token_piece_ids
                + [-1]
            )

            char_start = (
                local_piece_offsets[0][0]
                if local_piece_offsets
                else 0
            )
            char_end = (
                local_piece_offsets[-1][1]
                if local_piece_offsets
                else 0
            )

            gold_entities = []

            for ent in row["entities"]:
                es, ee = map(
                    int,
                    ent["position"],
                )

                if (
                    es >= char_start
                    and ee <= char_end
                ):
                    gold_entities.append(
                        (
                            ent["type"],
                            es,
                            ee,
                        )
                    )

                # Any overlap without full containment would indicate that
                # a gold entity was split across chunks.
                elif (
                    es < char_end
                    and ee > char_start
                ):
                    raise RuntimeError(
                        f"[{self.name}] Gold entity was split across chunks: "
                        f"{ent}"
                    )

            chunks.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": token_labels,
                "piece_ids": token_piece_ids,
                "piece_offsets": local_piece_offsets,
                "doc_idx": int(doc_idx),
                "chunk_idx": int(chunk_idx),
                "gold_entities": gold_entities,
            })

        return chunks

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]


class NERCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(
            len(x["input_ids"])
            for x in features
        )

        pad_id = int(
            self.tokenizer.pad_token_id
        )

        input_ids = []
        attention_mask = []
        labels = []
        piece_ids = []
        doc_idx = []
        chunk_idx = []
        piece_offsets = []
        gold_entities = []

        for x in features:
            n = len(x["input_ids"])
            pad_n = max_len - n

            input_ids.append(
                x["input_ids"]
                + [pad_id] * pad_n
            )
            attention_mask.append(
                x["attention_mask"]
                + [0] * pad_n
            )
            labels.append(
                x["labels"]
                + [IGNORE_INDEX] * pad_n
            )
            piece_ids.append(
                x["piece_ids"]
                + [-1] * pad_n
            )
            doc_idx.append(
                x["doc_idx"]
            )
            chunk_idx.append(
                x["chunk_idx"]
            )
            piece_offsets.append(
                x["piece_offsets"]
            )
            gold_entities.append(
                x["gold_entities"]
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long,
            ),
            "piece_ids": torch.tensor(
                piece_ids,
                dtype=torch.long,
            ),
            "doc_idx": torch.tensor(
                doc_idx,
                dtype=torch.long,
            ),
            "chunk_idx": torch.tensor(
                chunk_idx,
                dtype=torch.long,
            ),
            # Python metadata because lengths differ.
            "piece_offsets": piece_offsets,
            "gold_entities": gold_entities,
        }


# ============================================================
# 6. DECODE SUBWORD PREDICTIONS -> RAW CHARACTER SPANS
# ============================================================

def first_subword_predictions(
    token_pred_ids: List[int],
    token_piece_ids: List[int],
    num_pieces: int,
) -> List[int]:
    """Use the first subword prediction as the prediction for each piece."""

    result = [LABEL2ID["O"]] * num_pieces
    seen = set()

    for pred_id, piece_id in zip(token_pred_ids, token_piece_ids):
        if piece_id < 0 or piece_id >= num_pieces:
            continue
        if piece_id in seen:
            continue
        result[piece_id] = int(pred_id)
        seen.add(piece_id)

    return result


def decode_piece_predictions(
    piece_pred_ids: List[int],
    piece_offsets: List[Tuple[int, int]],
) -> List[Tuple[str, int, int]]:
    entities = []

    current_type = None
    current_start = None
    current_end = None

    def close_current():
        nonlocal current_type, current_start, current_end
        if current_type is not None:
            entities.append(
                (current_type, int(current_start), int(current_end))
            )
        current_type = None
        current_start = None
        current_end = None

    for pred_id, (start, end) in zip(piece_pred_ids, piece_offsets):
        label = ID2LABEL[int(pred_id)]

        if label == "O":
            close_current()
            continue

        prefix, tag = label.split("-", 1)
        etype = TAG_TO_TYPE[tag]

        if prefix == "B":
            close_current()
            current_type = etype
            current_start = start
            current_end = end
        else:
            # Legal continuation.
            if current_type == etype:
                current_end = end
            else:
                # Recover from an illegal I-tag by starting a new entity.
                close_current()
                current_type = etype
                current_start = start
                current_end = end

    close_current()
    return entities


def gold_entity_set(row: Dict[str, Any], visible_end: int = None) -> set:
    out = set()
    for ent in row["entities"]:
        s, e = map(int, ent["position"])
        if visible_end is not None and e > visible_end:
            continue
        out.add((ent["type"], s, e))
    return out


# ============================================================
# 7. EXACT-SPAN ENTITY METRICS
# ============================================================

def calculate_entity_metrics(all_gold, all_pred):
    stats = {
        etype: {"tp": 0, "fp": 0, "fn": 0}
        for etype in ENTITY_TYPES
    }

    for gold_set, pred_set in zip(all_gold, all_pred):
        for etype in ENTITY_TYPES:
            g = {x for x in gold_set if x[0] == etype}
            p = {x for x in pred_set if x[0] == etype}
            stats[etype]["tp"] += len(g & p)
            stats[etype]["fp"] += len(p - g)
            stats[etype]["fn"] += len(g - p)

    per_type = {}
    f1s = []
    total_tp = total_fp = total_fn = 0

    for etype in ENTITY_TYPES:
        tp = stats[etype]["tp"]
        fp = stats[etype]["fp"]
        fn = stats[etype]["fn"]

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        per_type[etype] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }
        f1s.append(f1)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_p = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_r = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    micro_f1 = (
        2 * micro_p * micro_r / (micro_p + micro_r)
        if micro_p + micro_r
        else 0.0
    )

    return {
        "macro_f1": float(np.mean(f1s)),
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "per_type": per_type,
    }


def print_metrics(name, metrics):
    print(
        f"\n[{name}] loss={metrics['loss']:.6f} | "
        f"macro_F1={metrics['macro_f1']:.6f} | "
        f"micro_F1={metrics['micro_f1']:.6f}"
    )
    for etype in ENTITY_TYPES:
        m = metrics["per_type"][etype]
        print(
            f"  {etype:<24} "
            f"P={m['precision']:.4f} "
            f"R={m['recall']:.4f} "
            f"F1={m['f1']:.4f} "
            f"TP={m['tp']} FP={m['fp']} FN={m['fn']}"
        )


# ============================================================
# 8. EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    dataset,
    loader,
    device,
    use_amp=True,
):
    model.eval()

    all_gold = []
    all_pred = []
    losses = []

    amp_enabled = bool(
        use_amp
        and device.type == "cuda"
    )

    for batch in loader:
        piece_offsets_batch = batch.pop(
            "piece_offsets"
        )
        gold_entities_batch = batch.pop(
            "gold_entities"
        )

        piece_ids = batch.pop(
            "piece_ids"
        ).cpu().numpy()

        # Metadata only.
        batch.pop("doc_idx")
        batch.pop("chunk_idx")

        batch = {
            k: v.to(
                device,
                non_blocking=True,
            )
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            outputs = model(**batch)

        losses.append(
            float(
                outputs.loss
                .detach()
                .cpu()
            )
        )

        pred_ids = (
            outputs.logits
            .argmax(-1)
            .detach()
            .cpu()
            .numpy()
        )

        attention = (
            batch["attention_mask"]
            .detach()
            .cpu()
            .numpy()
        )

        batch_size = len(
            piece_offsets_batch
        )

        for i in range(batch_size):
            valid_len = int(
                attention[i].sum()
            )

            token_preds = (
                pred_ids[i][:valid_len]
                .tolist()
            )

            token_piece_ids = (
                piece_ids[i][:valid_len]
                .tolist()
            )

            piece_offsets = (
                piece_offsets_batch[i]
            )

            piece_preds = (
                first_subword_predictions(
                    token_preds,
                    token_piece_ids,
                    len(piece_offsets),
                )
            )

            pred_entities = set(
                decode_piece_predictions(
                    piece_preds,
                    piece_offsets,
                )
            )

            gold_entities = set(
                tuple(x)
                for x in gold_entities_batch[i]
            )

            all_pred.append(
                pred_entities
            )
            all_gold.append(
                gold_entities
            )

    metrics = calculate_entity_metrics(
        all_gold,
        all_pred,
    )

    metrics["loss"] = (
        float(np.mean(losses))
        if losses
        else math.nan
    )

    return metrics


# ============================================================
# 9. SAVE CHECKPOINT
# ============================================================

def save_checkpoint(model, tokenizer, directory: Path, epoch, metrics, args):
    directory.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(directory)
    tokenizer.save_pretrained(directory)
    save_json(
        {
            "epoch": int(epoch),
            "metrics": metrics,
            "label_list": LABEL_LIST,
            "entity_types": ENTITY_TYPES,
            "args": vars(args),
        },
        directory / "training_meta.json",
    )


# ============================================================
# 10. MAIN
# ============================================================

def main():
    args = parse_args()
    set_seed(args.seed)

    output_root = Path(args.output_dir)
    output_root.mkdir(parents=True, exist_ok=True)
    save_json(vars(args), output_root / "run_config.json")

    print("=" * 78)
    print("PHASE 1 - VIETNAMESE CLINICAL NER")
    print("=" * 78)
    print(f"train           : {args.train_jsonl}")
    print(f"valid_template  : {args.valid_template_jsonl}")
    print(f"valid_concept   : {args.valid_concept_jsonl}")
    print(f"model           : {args.model_name_or_path}")
    print(f"output          : {args.output_dir}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device          : {device}")
    if device.type == "cuda":
        print(f"GPU             : {torch.cuda.get_device_name(0)}")

    print("\nLoading model config/tokenizer...")
    config = AutoConfig.from_pretrained(
        args.model_name_or_path,
        local_files_only=args.local_files_only,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name_or_path,
        use_fast=False,
        local_files_only=args.local_files_only,
    )

    # RoBERTa uses reserved position IDs; config=258 corresponds to ~256 tokens.
    config_cap = int(getattr(config, "max_position_embeddings", args.max_length)) - 2
    effective_max_length = min(args.max_length, config_cap)
    if effective_max_length < args.max_length:
        print(
            f"[INFO] max_length capped from {args.max_length} "
            f"to {effective_max_length} by model config."
        )
    args.max_length = effective_max_length

    print("\nLoading datasets...")
    train_ds = ClinicalNERDataset(
        args.train_jsonl,
        tokenizer,
        args.max_length,
        "train",
    )
    valid_template_ds = ClinicalNERDataset(
        args.valid_template_jsonl,
        tokenizer,
        args.max_length,
        "valid_template",
    )
    valid_concept_ds = ClinicalNERDataset(
        args.valid_concept_jsonl,
        tokenizer,
        args.max_length,
        "valid_concept",
    )

    collator = NERCollator(tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=args.train_batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collator,
    )
    valid_template_loader = DataLoader(
        valid_template_ds,
        batch_size=args.eval_batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collator,
    )
    valid_concept_loader = DataLoader(
        valid_concept_ds,
        batch_size=args.eval_batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collator,
    )

    print("\nLoading token-classification model...")
    model = AutoModelForTokenClassification.from_pretrained(
        args.model_name_or_path,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
        local_files_only=args.local_files_only,
    )
    model.to(device)

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    optimizer_groups = [
        {
            "params": [
                p for n, p in model.named_parameters()
                if not any(x in n for x in no_decay)
            ],
            "weight_decay": args.weight_decay,
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if any(x in n for x in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = AdamW(optimizer_groups, lr=args.lr)

    updates_per_epoch = math.ceil(len(train_loader) / args.grad_accum_steps)
    total_steps = updates_per_epoch * args.epochs
    warmup_steps = int(total_steps * args.warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    amp_enabled = bool(USE_AMP and device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    print("\nTraining setup")
    print(f"  labels               : {len(LABEL_LIST)}")
    print(f"  max_length           : {args.max_length}")
    print(f"  train model chunks   : {len(train_ds):,}")
    print(f"  valid_template chunks: {len(valid_template_ds):,}")
    print(f"  valid_concept chunks : {len(valid_concept_ds):,}")
    print(f"  batch                : {args.train_batch_size}")
    print(f"  grad_accum           : {args.grad_accum_steps}")
    print(f"  effective batch      : {args.train_batch_size * args.grad_accum_steps}")
    print(f"  epochs               : {args.epochs}")
    print(f"  LR                   : {args.lr}")
    print(f"  optimizer steps      : {total_steps}")
    print(f"  warmup steps         : {warmup_steps}")
    print(f"  AMP                  : {amp_enabled}")

    history = []
    best_score = -1.0
    best_epoch = None
    no_improvement = 0

    for epoch in range(1, args.epochs + 1):
        epoch_start = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        for step, batch in enumerate(train_loader, 1):
            # Remove metadata before model forward.
            batch.pop("piece_offsets")
            batch.pop("gold_entities")
            batch.pop("piece_ids")
            batch.pop("doc_idx")
            batch.pop("chunk_idx")

            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                outputs = model(**batch)
                full_loss = outputs.loss
                loss = full_loss / args.grad_accum_steps

            scaler.scale(loss).backward()
            running_loss += float(full_loss.detach().cpu())

            should_update = (
                step % args.grad_accum_steps == 0
                or step == len(train_loader)
            )

            if should_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if step % 100 == 0 or step == len(train_loader):
                print(
                    f"Epoch {epoch:02d} | "
                    f"step {step:04d}/{len(train_loader):04d} | "
                    f"loss={running_loss / step:.5f} | "
                    f"lr={scheduler.get_last_lr()[0]:.3e}"
                )

        train_loss = running_loss / max(1, len(train_loader))

        template_metrics = evaluate(
            model,
            valid_template_ds,
            valid_template_loader,
            device,
            USE_AMP,
        )
        concept_metrics = evaluate(
            model,
            valid_concept_ds,
            valid_concept_loader,
            device,
            USE_AMP,
        )

        print_metrics("VALID_TEMPLATE", template_metrics)
        print_metrics("VALID_CONCEPT", concept_metrics)

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_template": template_metrics,
            "valid_concept": concept_metrics,
            "seconds": time.time() - epoch_start,
        }
        history.append(epoch_record)
        save_json(history, output_root / "training_history.json")

        # Always save last checkpoint.
        save_checkpoint(
            model,
            tokenizer,
            output_root / "last",
            epoch,
            {
                "valid_template": template_metrics,
                "valid_concept": concept_metrics,
            },
            args,
        )

        # Primary checkpoint selection = strict held-out-concept macro F1.
        score = float(concept_metrics["macro_f1"])
        if score > best_score + MIN_DELTA:
            best_score = score
            best_epoch = epoch
            no_improvement = 0

            print(
                f"\n*** NEW BEST: epoch={epoch} "
                f"valid_concept_macro_F1={score:.6f} ***"
            )

            save_checkpoint(
                model,
                tokenizer,
                output_root / "best",
                epoch,
                {
                    "valid_template": template_metrics,
                    "valid_concept": concept_metrics,
                },
                args,
            )
        else:
            no_improvement += 1
            print(
                f"\nNo improvement: {no_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        print(
            f"Epoch {epoch} done | train_loss={train_loss:.6f} | "
            f"best_epoch={best_epoch} | best_macro_F1={best_score:.6f}\n"
        )

        if no_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    print("=" * 78)
    print("TRAINING COMPLETE")
    print("=" * 78)
    print(f"Best epoch               : {best_epoch}")
    print(f"Best valid_concept macro : {best_score:.6f}")
    print(f"Best checkpoint          : {output_root / 'best'}")
    print(f"Last checkpoint          : {output_root / 'last'}")


if __name__ == "__main__":
    main()


[INFO] Ignoring unknown Jupyter/Colab arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-44d944da-ee30-424f-b634-5ac2dd394768.json']
PHASE 1 - VIETNAMESE CLINICAL NER
train           : /content/drive/MyDrive/data/ner_train.jsonl
valid_template  : /content/drive/MyDrive/data/ner_valid_template.jsonl
valid_concept   : /content/drive/MyDrive/data/ner_valid_concept.jsonl
model           : demdecuong/vihealthbert-base-syllable
output          : /content/drive/MyDrive/output
device          : cuda
GPU             : Tesla T4

Loading model config/tokenizer...

Loading datasets...
[train] validating 12,000 records...
[train] original_docs=12,000 | model_chunks=12,001 | chunked_docs=1 | max_chunks/doc=2 | gold_entities_preserved=62,885/62,885
[valid_template] validating 1,500 records...
[valid_template] original_docs=1,500 | model_chunks=1,502 | chunked_docs=2 | max_chunks/doc=2 | gold_entities_preserved=7,805/7,805
[valid_concept] validating 1,500 records...
[valid_concept] original_

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: demdecuong/vihealthbert-base-syllable
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


No improvement: 1/3
Epoch 4 done | train_loss=0.007181 | best_epoch=3 | best_macro_F1=0.952535

Epoch 05 | step 0100/0751 | loss=0.00594 | lr=1.074e-05
Epoch 05 | step 0200/0751 | loss=0.00582 | lr=1.037e-05
Epoch 05 | step 0300/0751 | loss=0.00570 | lr=1.000e-05
Epoch 05 | step 0400/0751 | loss=0.00562 | lr=9.631e-06
Epoch 05 | step 0500/0751 | loss=0.00554 | lr=9.261e-06
Epoch 05 | step 0600/0751 | loss=0.00546 | lr=8.892e-06
Epoch 05 | step 0700/0751 | loss=0.00538 | lr=8.523e-06
Epoch 05 | step 0751/0751 | loss=0.00534 | lr=8.331e-06

[VALID_TEMPLATE] loss=0.014935 | macro_F1=0.952442 | micro_F1=0.961561
  TRIỆU_CHỨNG              P=0.9544 R=0.9774 F1=0.9657 TP=2029 FP=97 FN=47
  TÊN_XÉT_NGHIỆM           P=0.9041 R=0.9049 F1=0.9045 TP=971 FP=103 FN=102
  KẾT_QUẢ_XÉT_NGHIỆM       P=0.9041 R=0.9049 F1=0.9045 TP=971 FP=103 FN=102
  CHẨN_ĐOÁN                P=0.9946 R=0.9809 F1=0.9877 TP=1853 FP=10 FN=36
  THUỐC                    P=1.0000 R=0.9994 F1=0.9997 TP=1693 FP=0 FN=1

[VALID_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


No improvement: 2/3
Epoch 5 done | train_loss=0.005339 | best_epoch=3 | best_macro_F1=0.952535

Epoch 06 | step 0100/0751 | loss=0.00472 | lr=7.962e-06
Epoch 06 | step 0200/0751 | loss=0.00465 | lr=7.592e-06
Epoch 06 | step 0300/0751 | loss=0.00462 | lr=7.223e-06
Epoch 06 | step 0400/0751 | loss=0.00456 | lr=6.854e-06
Epoch 06 | step 0500/0751 | loss=0.00450 | lr=6.484e-06
Epoch 06 | step 0600/0751 | loss=0.00445 | lr=6.115e-06
Epoch 06 | step 0700/0751 | loss=0.00441 | lr=5.746e-06
Epoch 06 | step 0751/0751 | loss=0.00439 | lr=5.554e-06

[VALID_TEMPLATE] loss=0.013905 | macro_F1=0.954195 | micro_F1=0.963731
  TRIỆU_CHỨNG              P=0.9637 R=0.9860 F1=0.9748 TP=2047 FP=77 FN=29
  TÊN_XÉT_NGHIỆM           P=0.9050 R=0.9059 F1=0.9054 TP=972 FP=102 FN=101
  KẾT_QUẢ_XÉT_NGHIỆM       P=0.9050 R=0.9059 F1=0.9054 TP=972 FP=102 FN=101
  CHẨN_ĐOÁN                P=0.9925 R=0.9788 F1=0.9856 TP=1849 FP=14 FN=40
  THUỐC                    P=1.0000 R=0.9994 F1=0.9997 TP=1693 FP=0 FN=1

[VALID_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


No improvement: 3/3
Epoch 6 done | train_loss=0.004391 | best_epoch=3 | best_macro_F1=0.952535

Early stopping triggered.
TRAINING COMPLETE
Best epoch               : 3
Best valid_concept macro : 0.952535
Best checkpoint          : /content/drive/MyDrive/output/best
Last checkpoint          : /content/drive/MyDrive/output/last


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
PHASE 2 — Vietnamese Clinical Assertion Classification
======================================================

Task:
For each already-detected entity of type:
    - TRIỆU_CHỨNG
    - CHẨN_ĐOÁN
    - THUỐC

predict a MULTI-LABEL assertion set:
    - isNegated
    - isFamily
    - isHistorical

This script is designed for the Phase-2 JSONL files created earlier:
    assertion_train_balanced.jsonl
    assertion_valid_template.jsonl
    assertion_valid_concept.jsonl

Expected row:
{
    "id": "...",
    "context": "...",
    "entity_text": "hen phế quản",
    "entity_position": [start, end],
    "entity_type": "CHẨN_ĐOÁN",
    "labels": ["isHistorical"]
}

Key implementation decisions
----------------------------
1) Entity-centered context:
   ViHealthBERT has a short (~256-token) context window. We tokenize prefix/entity/
   suffix separately and ALWAYS preserve the complete target entity plus nearby
   left/right context. The target can therefore never be silently truncated.

2) Explicit special markers:
   <TYPE_TRIEU_CHUNG>, <TYPE_CHAN_DOAN>, <TYPE_THUOC>
   <ENT> ... </ENT>

3) Multi-label head:
   3 independent sigmoid outputs + BCEWithLogitsLoss.

4) Competition-oriented model selection:
   Every epoch:
     - evaluate VALID_TEMPLATE
     - evaluate VALID_CONCEPT at threshold 0.5
     - tune 3 independent thresholds on VALID_CONCEPT
     - select BEST checkpoint by tuned mean sample-level Jaccard

5) Colab/Jupyter safe:
   parse_known_args() ignores notebook's injected "-f kernel-xxxx.json".

Recommended initialization:
    /content/drive/MyDrive/output/best
(the BEST Phase-1 NER checkpoint; encoder weights are reused)

Clean ablation:
    demdecuong/vihealthbert-base-syllable

Run:
    python train_phase2_assertion.py

or paste the whole script into a Colab cell and run it directly.
"""

import os
import gc
import json
import math
import time
import random
import argparse
from pathlib import Path
from typing import Dict, List, Any, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)


# ============================================================
# 0. PATHS — EDIT THIS BLOCK
# ============================================================

DATA_ROOT = "/content/drive/MyDrive/data"

TRAIN_JSONL = (
    f"{DATA_ROOT}/assertion_train_balanced.jsonl"
)
VALID_TEMPLATE_JSONL = (
    f"{DATA_ROOT}/assertion_valid_template.jsonl"
)
VALID_CONCEPT_JSONL = (
    f"{DATA_ROOT}/assertion_valid_concept.jsonl"
)

# Recommended: initialize Phase 2 from the BEST Phase-1 NER encoder.
# The token-classification head is discarded/replaced automatically; the
# ViHealthBERT encoder weights are reused.
PHASE1_BEST_DIR = "/content/drive/MyDrive/output/best"

MODEL_NAME_OR_PATH = PHASE1_BEST_DIR
LOCAL_FILES_ONLY = True

# Alternative clean ablation: initialize directly from base ViHealthBERT.
# MODEL_NAME_OR_PATH = "demdecuong/vihealthbert-base-syllable"
# LOCAL_FILES_ONLY = False

OUTPUT_DIR = (
    "/content/drive/MyDrive/output_phase2_assertion"
)


# ============================================================
# 1. TRAIN CONFIG
# ============================================================

SEED = 42

MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 6
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

NUM_WORKERS = 2
PIN_MEMORY = True
USE_AMP = True

EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-5

# Threshold grid for competition Jaccard optimization.
THRESHOLD_MIN = 0.10
THRESHOLD_MAX = 0.90
THRESHOLD_STEP = 0.05


# ============================================================
# 2. LABELS / ENTITY TYPES
# ============================================================

ASSERTION_LABELS = [
    "isNegated",
    "isFamily",
    "isHistorical",
]

LABEL2ID = {
    name: i
    for i, name in enumerate(ASSERTION_LABELS)
}
ID2LABEL = {
    i: name
    for name, i in LABEL2ID.items()
}

ALLOWED_ENTITY_TYPES = [
    "TRIỆU_CHỨNG",
    "CHẨN_ĐOÁN",
    "THUỐC",
]

TYPE_TOKEN = {
    "TRIỆU_CHỨNG": "<TYPE_TRIEU_CHUNG>",
    "CHẨN_ĐOÁN": "<TYPE_CHAN_DOAN>",
    "THUỐC": "<TYPE_THUOC>",
}

ENT_START = "<ENT>"
ENT_END = "</ENT>"

ADDITIONAL_SPECIAL_TOKENS = [
    ENT_START,
    ENT_END,
    TYPE_TOKEN["TRIỆU_CHỨNG"],
    TYPE_TOKEN["CHẨN_ĐOÁN"],
    TYPE_TOKEN["THUỐC"],
]


# ============================================================
# 3. ARGUMENTS / UTILITIES
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument(
        "--train_jsonl",
        type=str,
        default=TRAIN_JSONL,
    )
    p.add_argument(
        "--valid_template_jsonl",
        type=str,
        default=VALID_TEMPLATE_JSONL,
    )
    p.add_argument(
        "--valid_concept_jsonl",
        type=str,
        default=VALID_CONCEPT_JSONL,
    )
    p.add_argument(
        "--model_name_or_path",
        type=str,
        default=MODEL_NAME_OR_PATH,
    )
    p.add_argument(
        "--output_dir",
        type=str,
        default=OUTPUT_DIR,
    )

    p.add_argument(
        "--max_length",
        type=int,
        default=MAX_LENGTH,
    )
    p.add_argument(
        "--train_batch_size",
        type=int,
        default=TRAIN_BATCH_SIZE,
    )
    p.add_argument(
        "--eval_batch_size",
        type=int,
        default=EVAL_BATCH_SIZE,
    )
    p.add_argument(
        "--grad_accum_steps",
        type=int,
        default=GRAD_ACCUM_STEPS,
    )

    p.add_argument(
        "--lr",
        type=float,
        default=LEARNING_RATE,
    )
    p.add_argument(
        "--weight_decay",
        type=float,
        default=WEIGHT_DECAY,
    )
    p.add_argument(
        "--epochs",
        type=int,
        default=NUM_EPOCHS,
    )
    p.add_argument(
        "--warmup_ratio",
        type=float,
        default=WARMUP_RATIO,
    )
    p.add_argument(
        "--seed",
        type=int,
        default=SEED,
    )

    p.add_argument(
        "--local_files_only",
        action="store_true",
        default=LOCAL_FILES_ONLY,
    )

    # Colab/Jupyter injects e.g. "-f kernel-xxxx.json".
    args, unknown = p.parse_known_args()

    if unknown:
        print(
            "[INFO] Ignoring unknown Jupyter/Colab args:",
            unknown,
        )

    return args


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Fast training; fixed Python/NumPy/PyTorch seeds are still used.
    torch.backends.cudnn.benchmark = True


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing file:\n  {p}\n\n"
            "Edit DATA_ROOT at the top of this script "
            "or override the path by CLI."
        )

    rows = []

    with p.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                row = json.loads(line)
            except Exception as exc:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{p}:{line_no}: {exc}"
                )

            rows.append(row)

    return rows


def save_json(obj: Any, path: Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
        )


def validate_row(
    row: Dict[str, Any],
    source: str,
    idx: int,
):
    required = {
        "context",
        "entity_text",
        "entity_position",
        "entity_type",
        "labels",
    }

    missing = required - set(row)

    if missing:
        raise ValueError(
            f"{source}[{idx}] missing fields: "
            f"{sorted(missing)}"
        )

    context = row["context"]
    entity_text = row["entity_text"]

    start, end = map(
        int,
        row["entity_position"],
    )

    if not (
        0 <= start < end <= len(context)
    ):
        raise ValueError(
            f"{source}[{idx}] invalid entity_position "
            f"{row['entity_position']} "
            f"for context length {len(context)}"
        )

    sliced = context[start:end]

    if sliced != entity_text:
        raise ValueError(
            f"{source}[{idx}] span mismatch\n"
            f"entity_text={entity_text!r}\n"
            f"context[{start}:{end}]={sliced!r}"
        )

    if (
        row["entity_type"]
        not in ALLOWED_ENTITY_TYPES
    ):
        raise ValueError(
            f"{source}[{idx}] invalid entity_type: "
            f"{row['entity_type']!r}"
        )

    labels = row["labels"]

    if not isinstance(labels, list):
        raise ValueError(
            f"{source}[{idx}] labels must be list."
        )

    unknown = (
        set(labels)
        - set(ASSERTION_LABELS)
    )

    if unknown:
        raise ValueError(
            f"{source}[{idx}] unknown assertion labels: "
            f"{sorted(unknown)}"
        )


# ============================================================
# 4. ENTITY-CENTERED TOKENIZATION
# ============================================================

def encode_no_special(
    tokenizer,
    text: str,
) -> List[int]:
    if not text:
        return []

    ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    return [
        int(x)
        for x in ids
    ]


def allocate_context_budget(
    prefix_len: int,
    suffix_len: int,
    budget: int,
) -> Tuple[int, int]:
    """
    Allocate available tokens around the entity.

    Returns:
        n_prefix_tokens, n_suffix_tokens

    Prefix tokens are later taken from the RIGHT (nearest entity).
    Suffix tokens are taken from the LEFT (nearest entity).
    """

    if budget <= 0:
        return 0, 0

    left_target = budget // 2
    right_target = budget - left_target

    n_prefix = min(
        prefix_len,
        left_target,
    )
    n_suffix = min(
        suffix_len,
        right_target,
    )

    remaining = (
        budget
        - n_prefix
        - n_suffix
    )

    # Give unused budget to the longer remaining side.
    extra_prefix_available = (
        prefix_len - n_prefix
    )
    extra_suffix_available = (
        suffix_len - n_suffix
    )

    if (
        extra_prefix_available
        >= extra_suffix_available
    ):
        extra = min(
            extra_prefix_available,
            remaining,
        )
        n_prefix += extra
        remaining -= extra

        extra = min(
            extra_suffix_available,
            remaining,
        )
        n_suffix += extra
        remaining -= extra

    else:
        extra = min(
            extra_suffix_available,
            remaining,
        )
        n_suffix += extra
        remaining -= extra

        extra = min(
            extra_prefix_available,
            remaining,
        )
        n_prefix += extra
        remaining -= extra

    return n_prefix, n_suffix


def build_entity_centered_encoding(
    tokenizer,
    row: Dict[str, Any],
    max_length: int,
) -> Dict[str, Any]:

    context = row["context"]
    start, end = map(
        int,
        row["entity_position"],
    )

    prefix_text = context[:start]
    entity_text = context[start:end]
    suffix_text = context[end:]

    prefix_ids = encode_no_special(
        tokenizer,
        prefix_text,
    )
    entity_ids = encode_no_special(
        tokenizer,
        entity_text,
    )
    suffix_ids = encode_no_special(
        tokenizer,
        suffix_text,
    )

    type_ids = encode_no_special(
        tokenizer,
        TYPE_TOKEN[
            row["entity_type"]
        ],
    )
    ent_start_ids = encode_no_special(
        tokenizer,
        ENT_START,
    )
    ent_end_ids = encode_no_special(
        tokenizer,
        ENT_END,
    )

    # Number of BOS/EOS/etc tokens model will add.
    n_outer_special = (
        tokenizer.num_special_tokens_to_add(
            pair=False
        )
    )

    content_limit = (
        max_length
        - n_outer_special
    )

    mandatory_len = (
        len(type_ids)
        + len(ent_start_ids)
        + len(entity_ids)
        + len(ent_end_ids)
    )

    if mandatory_len > content_limit:
        raise RuntimeError(
            "Target entity itself cannot fit in the "
            "model context window.\n"
            f"entity={entity_text!r}\n"
            f"mandatory_tokens={mandatory_len}\n"
            f"content_limit={content_limit}"
        )

    budget = (
        content_limit
        - mandatory_len
    )

    n_prefix, n_suffix = (
        allocate_context_budget(
            prefix_len=len(prefix_ids),
            suffix_len=len(suffix_ids),
            budget=budget,
        )
    )

    prefix_keep = (
        prefix_ids[-n_prefix:]
        if n_prefix > 0
        else []
    )

    suffix_keep = (
        suffix_ids[:n_suffix]
        if n_suffix > 0
        else []
    )

    content_ids = (
        type_ids
        + prefix_keep
        + ent_start_ids
        + entity_ids
        + ent_end_ids
        + suffix_keep
    )

    input_ids = (
        tokenizer.build_inputs_with_special_tokens(
            content_ids
        )
    )

    if len(input_ids) > max_length:
        raise RuntimeError(
            "Internal token-budget error: "
            f"len(input_ids)={len(input_ids)} "
            f"> max_length={max_length}"
        )

    attention_mask = [
        1
    ] * len(input_ids)

    label_vec = np.zeros(
        len(ASSERTION_LABELS),
        dtype=np.float32,
    )

    for label in row["labels"]:
        label_vec[
            LABEL2ID[label]
        ] = 1.0

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_vec,
        "sample_id": row.get(
            "id",
            "",
        ),
        "entity_type": row["entity_type"],
        "entity_text": entity_text,
        "prefix_tokens_kept": n_prefix,
        "suffix_tokens_kept": n_suffix,
        "prefix_tokens_total": len(
            prefix_ids
        ),
        "suffix_tokens_total": len(
            suffix_ids
        ),
    }


# ============================================================
# 5. DATASET / COLLATOR
# ============================================================

class AssertionDataset(Dataset):
    def __init__(
        self,
        path: str,
        tokenizer,
        max_length: int,
        name: str,
    ):
        self.path = path
        self.name = name
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.records = read_jsonl(path)

        print(
            f"[{name}] validating "
            f"{len(self.records):,} rows..."
        )

        for i, row in enumerate(
            self.records
        ):
            validate_row(
                row,
                name,
                i,
            )

        self.features = []

        cropped_left = 0
        cropped_right = 0

        for row in self.records:
            feat = (
                build_entity_centered_encoding(
                    tokenizer=tokenizer,
                    row=row,
                    max_length=max_length,
                )
            )

            if (
                feat["prefix_tokens_kept"]
                < feat["prefix_tokens_total"]
            ):
                cropped_left += 1

            if (
                feat["suffix_tokens_kept"]
                < feat["suffix_tokens_total"]
            ):
                cropped_right += 1

            self.features.append(feat)

        print(
            f"[{name}] encoded="
            f"{len(self.features):,} | "
            f"left_context_cropped="
            f"{cropped_left:,} | "
            f"right_context_cropped="
            f"{cropped_right:,}"
        )

        self._print_label_stats()

    def _print_label_stats(self):
        counts = {
            label: 0
            for label in ASSERTION_LABELS
        }

        empty = 0

        for row in self.records:
            if not row["labels"]:
                empty += 1

            for label in row["labels"]:
                counts[label] += 1

        print(
            f"[{self.name}] no_assertion="
            f"{empty:,}"
        )

        for label in ASSERTION_LABELS:
            print(
                f"[{self.name}] "
                f"{label}={counts[label]:,}"
            )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]


class AssertionCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        model_items = [
            {
                "input_ids": x["input_ids"],
                "attention_mask":
                    x["attention_mask"],
            }
            for x in features
        ]

        batch = self.tokenizer.pad(
            model_items,
            padding=True,
            return_tensors="pt",
        )

        batch["labels"] = torch.tensor(
            np.stack(
                [
                    x["labels"]
                    for x in features
                ]
            ),
            dtype=torch.float32,
        )

        batch["sample_ids"] = [
            x["sample_id"]
            for x in features
        ]

        batch["entity_types"] = [
            x["entity_type"]
            for x in features
        ]

        return batch


# ============================================================
# 6. METRICS
# ============================================================

def sigmoid_np(logits: np.ndarray):
    logits = np.clip(
        logits,
        -50.0,
        50.0,
    )

    return (
        1.0
        / (
            1.0
            + np.exp(-logits)
        )
    )


def jaccard_per_sample(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> np.ndarray:
    """
    Jaccard for each entity's assertion set.

    If both gold and prediction are empty:
        Jaccard = 1.0
    """

    intersection = np.logical_and(
        y_true,
        y_pred,
    ).sum(axis=1)

    union = np.logical_or(
        y_true,
        y_pred,
    ).sum(axis=1)

    scores = np.ones(
        len(y_true),
        dtype=np.float32,
    )

    non_empty = union > 0

    scores[non_empty] = (
        intersection[non_empty]
        / union[non_empty]
    )

    return scores


def calculate_metrics(
    probs: np.ndarray,
    y_true: np.ndarray,
    thresholds: np.ndarray,
) -> Dict[str, Any]:

    y_true_bool = (
        y_true >= 0.5
    )

    y_pred = (
        probs
        >= thresholds.reshape(1, -1)
    )

    per_label = {}

    tp_total = 0
    fp_total = 0
    fn_total = 0

    label_f1s = []

    for i, label in enumerate(
        ASSERTION_LABELS
    ):
        yt = y_true_bool[:, i]
        yp = y_pred[:, i]

        tp = int(
            np.logical_and(
                yt,
                yp,
            ).sum()
        )

        fp = int(
            np.logical_and(
                ~yt,
                yp,
            ).sum()
        )

        fn = int(
            np.logical_and(
                yt,
                ~yp,
            ).sum()
        )

        tn = int(
            np.logical_and(
                ~yt,
                ~yp,
            ).sum()
        )

        precision = (
            tp / (tp + fp)
            if tp + fp
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn
            else 0.0
        )

        f1 = (
            2.0
            * precision
            * recall
            / (precision + recall)
            if precision + recall
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "threshold": float(
                thresholds[i]
            ),
        }

        label_f1s.append(f1)

        tp_total += tp
        fp_total += fp
        fn_total += fn

    micro_p = (
        tp_total
        / (tp_total + fp_total)
        if tp_total + fp_total
        else 0.0
    )

    micro_r = (
        tp_total
        / (tp_total + fn_total)
        if tp_total + fn_total
        else 0.0
    )

    micro_f1 = (
        2.0
        * micro_p
        * micro_r
        / (micro_p + micro_r)
        if micro_p + micro_r
        else 0.0
    )

    sample_jaccard = (
        jaccard_per_sample(
            y_true_bool,
            y_pred,
        )
    )

    exact_match = np.all(
        y_true_bool == y_pred,
        axis=1,
    ).mean()

    return {
        "mean_jaccard": float(
            sample_jaccard.mean()
        ),
        "exact_set_match": float(
            exact_match
        ),
        "macro_f1": float(
            np.mean(label_f1s)
        ),
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "per_label": per_label,
        "thresholds": {
            label: float(
                thresholds[i]
            )
            for i, label
            in enumerate(
                ASSERTION_LABELS
            )
        },
    }


def tune_thresholds(
    probs: np.ndarray,
    y_true: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Exhaustive 3D threshold search for the exact competition-oriented
    sample-level mean Jaccard.

    17 values (0.10..0.90 step 0.05)^3 = 4913 combinations.
    """

    grid = np.round(
        np.arange(
            THRESHOLD_MIN,
            THRESHOLD_MAX
            + 1e-9,
            THRESHOLD_STEP,
        ),
        4,
    )

    y_true_bool = (
        y_true >= 0.5
    )

    best_thresholds = np.array(
        [0.5, 0.5, 0.5],
        dtype=np.float32,
    )

    best_jaccard = -1.0
    best_exact = -1.0

    # Precompute predictions for each label/threshold.
    pred_cache = [
        {
            float(t): (
                probs[:, label_idx] >= t
            )
            for t in grid
        }
        for label_idx in range(
            len(ASSERTION_LABELS)
        )
    ]

    for t0 in grid:
        p0 = pred_cache[0][float(t0)]

        for t1 in grid:
            p1 = pred_cache[1][float(t1)]

            for t2 in grid:
                p2 = pred_cache[2][float(t2)]

                pred = np.stack(
                    [p0, p1, p2],
                    axis=1,
                )

                j = float(
                    jaccard_per_sample(
                        y_true_bool,
                        pred,
                    ).mean()
                )

                # Tie-breaker:
                # prefer higher exact set match,
                # then thresholds closer to 0.5.
                if (
                    j > best_jaccard
                    + 1e-12
                ):
                    better = True
                elif abs(
                    j - best_jaccard
                ) <= 1e-12:
                    exact = float(
                        np.all(
                            y_true_bool == pred,
                            axis=1,
                        ).mean()
                    )

                    if exact > best_exact:
                        better = True
                    elif abs(
                        exact - best_exact
                    ) <= 1e-12:
                        current_dist = (
                            abs(t0 - 0.5)
                            + abs(t1 - 0.5)
                            + abs(t2 - 0.5)
                        )

                        best_dist = float(
                            np.abs(
                                best_thresholds
                                - 0.5
                            ).sum()
                        )

                        better = (
                            current_dist
                            < best_dist
                        )
                    else:
                        better = False
                else:
                    better = False

                if better:
                    best_jaccard = j
                    best_thresholds = (
                        np.array(
                            [t0, t1, t2],
                            dtype=np.float32,
                        )
                    )

                    best_exact = float(
                        np.all(
                            y_true_bool == pred,
                            axis=1,
                        ).mean()
                    )

    metrics = calculate_metrics(
        probs=probs,
        y_true=y_true,
        thresholds=best_thresholds,
    )

    return best_thresholds, metrics


def calculate_metrics_by_entity_type(
    probs: np.ndarray,
    y_true: np.ndarray,
    entity_types: List[str],
    thresholds: np.ndarray,
) -> Dict[str, Any]:

    out = {}

    entity_types_np = np.array(
        entity_types,
        dtype=object,
    )

    for entity_type in ALLOWED_ENTITY_TYPES:
        mask = (
            entity_types_np
            == entity_type
        )

        if not mask.any():
            continue

        out[entity_type] = (
            calculate_metrics(
                probs=probs[mask],
                y_true=y_true[mask],
                thresholds=thresholds,
            )
        )

    return out


def print_metrics(
    split_name: str,
    metrics: Dict[str, Any],
):
    print(
        f"\n[{split_name}] "
        f"Jaccard={metrics['mean_jaccard']:.6f} | "
        f"ExactSet={metrics['exact_set_match']:.6f} | "
        f"macro_F1={metrics['macro_f1']:.6f} | "
        f"micro_F1={metrics['micro_f1']:.6f}"
    )

    print(
        "  thresholds: "
        + ", ".join(
            f"{label}="
            f"{metrics['thresholds'][label]:.2f}"
            for label in ASSERTION_LABELS
        )
    )

    for label in ASSERTION_LABELS:
        m = metrics["per_label"][label]

        print(
            f"  {label:<15} "
            f"P={m['precision']:.4f} "
            f"R={m['recall']:.4f} "
            f"F1={m['f1']:.4f} "
            f"TP={m['tp']} "
            f"FP={m['fp']} "
            f"FN={m['fn']} "
            f"TN={m['tn']}"
        )


# ============================================================
# 7. EVALUATION
# ============================================================

@torch.no_grad()
def collect_predictions(
    model,
    loader,
    device,
    use_amp=True,
):
    model.eval()

    logits_all = []
    labels_all = []
    entity_types_all = []
    losses = []

    amp_enabled = bool(
        use_amp
        and device.type == "cuda"
    )

    for batch in loader:
        entity_types = batch.pop(
            "entity_types"
        )
        batch.pop("sample_ids")

        batch = {
            k: v.to(
                device,
                non_blocking=True,
            )
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            outputs = model(**batch)

        losses.append(
            float(
                outputs.loss
                .detach()
                .cpu()
            )
        )

        logits_all.append(
            outputs.logits
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        labels_all.append(
            batch["labels"]
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        entity_types_all.extend(
            entity_types
        )

    logits = np.concatenate(
        logits_all,
        axis=0,
    )

    y_true = np.concatenate(
        labels_all,
        axis=0,
    )

    probs = sigmoid_np(logits)

    return {
        "probs": probs,
        "y_true": y_true,
        "entity_types": entity_types_all,
        "loss": float(
            np.mean(losses)
            if losses
            else math.nan
        ),
    }


# ============================================================
# 8. CHECKPOINT
# ============================================================

def save_checkpoint(
    model,
    tokenizer,
    directory: Path,
    epoch: int,
    metrics: Dict[str, Any],
    thresholds: np.ndarray,
    args,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        directory
    )

    tokenizer.save_pretrained(
        directory
    )

    threshold_dict = {
        label: float(
            thresholds[i]
        )
        for i, label
        in enumerate(
            ASSERTION_LABELS
        )
    }

    save_json(
        threshold_dict,
        directory
        / "thresholds.json",
    )

    save_json(
        {
            "epoch": int(epoch),
            "selection_metric":
                "valid_concept_tuned_mean_jaccard",
            "metrics": metrics,
            "assertion_labels":
                ASSERTION_LABELS,
            "entity_types":
                ALLOWED_ENTITY_TYPES,
            "thresholds":
                threshold_dict,
            "training_args":
                vars(args),
        },
        directory
        / "training_meta.json",
    )


# ============================================================
# 9. MAIN
# ============================================================

def main():
    args = parse_args()
    set_seed(args.seed)

    output_root = Path(
        args.output_dir
    )

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    save_json(
        vars(args),
        output_root
        / "run_config.json",
    )

    print("=" * 78)
    print(
        "PHASE 2 - CLINICAL ASSERTION CLASSIFICATION"
    )
    print("=" * 78)

    print(
        f"train          : "
        f"{args.train_jsonl}"
    )
    print(
        f"valid_template : "
        f"{args.valid_template_jsonl}"
    )
    print(
        f"valid_concept  : "
        f"{args.valid_concept_jsonl}"
    )
    print(
        f"model          : "
        f"{args.model_name_or_path}"
    )
    print(
        f"output         : "
        f"{args.output_dir}"
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"device         : "
        f"{device}"
    )

    if device.type == "cuda":
        print(
            f"GPU            : "
            f"{torch.cuda.get_device_name(0)}"
        )

    # --------------------------------------------------------
    # Load config/tokenizer
    # --------------------------------------------------------

    print(
        "\nLoading config/tokenizer..."
    )

    config = AutoConfig.from_pretrained(
        args.model_name_or_path,
        local_files_only=
            args.local_files_only,
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            args.model_name_or_path,
            use_fast=False,
            local_files_only=
                args.local_files_only,
        )
    )

    # Add explicit task markers.
    num_added_tokens = (
        tokenizer.add_special_tokens(
            {
                "additional_special_tokens":
                    ADDITIONAL_SPECIAL_TOKENS
            }
        )
    )

    print(
        f"Added special tokens: "
        f"{num_added_tokens}"
    )

    # ViHealthBERT/PhoBERT RoBERTa-style:
    # max_position_embeddings=258 -> usable ~256 input tokens.
    model_pos_cap = int(
        getattr(
            config,
            "max_position_embeddings",
            args.max_length + 2,
        )
    )

    config_cap = (
        model_pos_cap - 2
    )

    effective_max_length = min(
        args.max_length,
        config_cap,
    )

    if (
        effective_max_length
        < args.max_length
    ):
        print(
            f"[INFO] max_length capped "
            f"from {args.max_length} "
            f"to {effective_max_length} "
            f"by model config."
        )

    args.max_length = (
        effective_max_length
    )

    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    print(
        "\nLoading datasets..."
    )

    train_ds = AssertionDataset(
        path=args.train_jsonl,
        tokenizer=tokenizer,
        max_length=args.max_length,
        name="train",
    )

    valid_template_ds = (
        AssertionDataset(
            path=args.valid_template_jsonl,
            tokenizer=tokenizer,
            max_length=args.max_length,
            name="valid_template",
        )
    )

    valid_concept_ds = (
        AssertionDataset(
            path=args.valid_concept_jsonl,
            tokenizer=tokenizer,
            max_length=args.max_length,
            name="valid_concept",
        )
    )

    collator = AssertionCollator(
        tokenizer
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            args.train_batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collator,
    )

    valid_template_loader = (
        DataLoader(
            valid_template_ds,
            batch_size=
                args.eval_batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            collate_fn=collator,
        )
    )

    valid_concept_loader = (
        DataLoader(
            valid_concept_ds,
            batch_size=
                args.eval_batch_size,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            collate_fn=collator,
        )
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    print(
        "\nLoading multi-label model..."
    )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            args.model_name_or_path,
            num_labels=
                len(ASSERTION_LABELS),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            problem_type=
                "multi_label_classification",
            ignore_mismatched_sizes=True,
            local_files_only=
                args.local_files_only,
        )
    )

    # IMPORTANT:
    # tokenizer gained ENT/TYPE special tokens.
    model.resize_token_embeddings(
        len(tokenizer)
    )

    model.to(device)

    # --------------------------------------------------------
    # Optimizer / scheduler
    # --------------------------------------------------------

    no_decay = (
        "bias",
        "LayerNorm.weight",
        "layer_norm.weight",
    )

    optimizer_groups = [
        {
            "params": [
                p
                for n, p
                in model.named_parameters()
                if not any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay":
                args.weight_decay,
        },
        {
            "params": [
                p
                for n, p
                in model.named_parameters()
                if any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = AdamW(
        optimizer_groups,
        lr=args.lr,
    )

    updates_per_epoch = math.ceil(
        len(train_loader)
        / args.grad_accum_steps
    )

    total_steps = (
        updates_per_epoch
        * args.epochs
    )

    warmup_steps = int(
        total_steps
        * args.warmup_ratio
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=
                warmup_steps,
            num_training_steps=
                total_steps,
        )
    )

    amp_enabled = bool(
        USE_AMP
        and device.type == "cuda"
    )

    scaler = (
        torch.cuda.amp.GradScaler(
            enabled=amp_enabled
        )
    )

    print(
        "\nTraining setup"
    )
    print(
        f"  assertion labels    : "
        f"{ASSERTION_LABELS}"
    )
    print(
        f"  max_length          : "
        f"{args.max_length}"
    )
    print(
        f"  train rows          : "
        f"{len(train_ds):,}"
    )
    print(
        f"  valid_template rows : "
        f"{len(valid_template_ds):,}"
    )
    print(
        f"  valid_concept rows  : "
        f"{len(valid_concept_ds):,}"
    )
    print(
        f"  batch               : "
        f"{args.train_batch_size}"
    )
    print(
        f"  grad_accum          : "
        f"{args.grad_accum_steps}"
    )
    print(
        f"  effective batch     : "
        f"{args.train_batch_size * args.grad_accum_steps}"
    )
    print(
        f"  epochs              : "
        f"{args.epochs}"
    )
    print(
        f"  LR                  : "
        f"{args.lr}"
    )
    print(
        f"  optimizer steps     : "
        f"{total_steps}"
    )
    print(
        f"  warmup steps        : "
        f"{warmup_steps}"
    )
    print(
        f"  AMP                 : "
        f"{amp_enabled}"
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    history = []

    best_score = -1.0
    best_epoch = None
    best_thresholds = np.array(
        [0.5, 0.5, 0.5],
        dtype=np.float32,
    )

    no_improvement = 0

    for epoch in range(
        1,
        args.epochs + 1,
    ):
        epoch_start = time.time()

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        running_loss = 0.0

        for step, batch in enumerate(
            train_loader,
            1,
        ):
            # Non-model metadata.
            batch.pop(
                "sample_ids"
            )
            batch.pop(
                "entity_types"
            )

            batch = {
                k: v.to(
                    device,
                    non_blocking=True,
                )
                for k, v
                in batch.items()
            }

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                outputs = model(**batch)

                full_loss = outputs.loss

                loss = (
                    full_loss
                    / args.grad_accum_steps
                )

            scaler.scale(
                loss
            ).backward()

            running_loss += float(
                full_loss
                .detach()
                .cpu()
            )

            should_update = (
                step
                % args.grad_accum_steps
                == 0
                or step
                == len(train_loader)
            )

            if should_update:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                scaler.step(
                    optimizer
                )
                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

            if (
                step % 100 == 0
                or step
                == len(train_loader)
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"step "
                    f"{step:04d}/"
                    f"{len(train_loader):04d} | "
                    f"loss="
                    f"{running_loss / step:.5f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.3e}"
                )

        train_loss = (
            running_loss
            / max(
                1,
                len(train_loader),
            )
        )

        # ----------------------------------------------------
        # Validation: collect once
        # ----------------------------------------------------

        template_out = (
            collect_predictions(
                model=model,
                loader=
                    valid_template_loader,
                device=device,
                use_amp=USE_AMP,
            )
        )

        concept_out = (
            collect_predictions(
                model=model,
                loader=
                    valid_concept_loader,
                device=device,
                use_amp=USE_AMP,
            )
        )

        # Default threshold 0.5
        default_thresholds = (
            np.array(
                [0.5, 0.5, 0.5],
                dtype=np.float32,
            )
        )

        template_default = (
            calculate_metrics(
                probs=
                    template_out["probs"],
                y_true=
                    template_out["y_true"],
                thresholds=
                    default_thresholds,
            )
        )

        concept_default = (
            calculate_metrics(
                probs=
                    concept_out["probs"],
                y_true=
                    concept_out["y_true"],
                thresholds=
                    default_thresholds,
            )
        )

        template_default[
            "loss"
        ] = template_out["loss"]

        concept_default[
            "loss"
        ] = concept_out["loss"]

        # Tune the 3 thresholds on strict held-out concept validation.
        tuned_thresholds, (
            concept_tuned
        ) = tune_thresholds(
            probs=
                concept_out["probs"],
            y_true=
                concept_out["y_true"],
        )

        concept_tuned[
            "loss"
        ] = concept_out["loss"]

        # Apply the SAME tuned thresholds to valid-template.
        template_tuned = (
            calculate_metrics(
                probs=
                    template_out["probs"],
                y_true=
                    template_out["y_true"],
                thresholds=
                    tuned_thresholds,
            )
        )

        template_tuned[
            "loss"
        ] = template_out["loss"]

        # Entity-type breakdown using tuned thresholds.
        concept_by_type = (
            calculate_metrics_by_entity_type(
                probs=
                    concept_out["probs"],
                y_true=
                    concept_out["y_true"],
                entity_types=
                    concept_out[
                        "entity_types"
                    ],
                thresholds=
                    tuned_thresholds,
            )
        )

        print(
            "\n--- threshold=0.50 ---"
        )
        print_metrics(
            "VALID_TEMPLATE_DEFAULT",
            template_default,
        )
        print_metrics(
            "VALID_CONCEPT_DEFAULT",
            concept_default,
        )

        print(
            "\n--- tuned thresholds "
            "on VALID_CONCEPT ---"
        )
        print_metrics(
            "VALID_TEMPLATE_TUNED",
            template_tuned,
        )
        print_metrics(
            "VALID_CONCEPT_TUNED",
            concept_tuned,
        )

        print(
            "\n[VALID_CONCEPT by entity type]"
        )

        for entity_type in (
            ALLOWED_ENTITY_TYPES
        ):
            if (
                entity_type
                not in concept_by_type
            ):
                continue

            m = (
                concept_by_type[
                    entity_type
                ]
            )

            print(
                f"  {entity_type:<18} "
                f"Jaccard="
                f"{m['mean_jaccard']:.4f} | "
                f"ExactSet="
                f"{m['exact_set_match']:.4f} | "
                f"micro_F1="
                f"{m['micro_f1']:.4f}"
            )

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_template_default":
                template_default,
            "valid_concept_default":
                concept_default,
            "valid_template_tuned":
                template_tuned,
            "valid_concept_tuned":
                concept_tuned,
            "valid_concept_by_entity_type":
                concept_by_type,
            "seconds":
                time.time()
                - epoch_start,
        }

        history.append(
            epoch_record
        )

        save_json(
            history,
            output_root
            / "training_history.json",
        )

        # Save last model with this epoch's tuned thresholds.
        save_checkpoint(
            model=model,
            tokenizer=tokenizer,
            directory=
                output_root / "last",
            epoch=epoch,
            metrics={
                "valid_template_default":
                    template_default,
                "valid_concept_default":
                    concept_default,
                "valid_template_tuned":
                    template_tuned,
                "valid_concept_tuned":
                    concept_tuned,
                "valid_concept_by_entity_type":
                    concept_by_type,
            },
            thresholds=
                tuned_thresholds,
            args=args,
        )

        # Competition-aligned model selection.
        score = float(
            concept_tuned[
                "mean_jaccard"
            ]
        )

        if (
            score
            > best_score
            + MIN_DELTA
        ):
            best_score = score
            best_epoch = epoch
            best_thresholds = (
                tuned_thresholds.copy()
            )
            no_improvement = 0

            print(
                "\n*** NEW BEST *** "
                f"epoch={epoch} | "
                f"valid_concept_tuned_"
                f"Jaccard={score:.6f}"
            )

            save_checkpoint(
                model=model,
                tokenizer=tokenizer,
                directory=
                    output_root / "best",
                epoch=epoch,
                metrics={
                    "valid_template_default":
                        template_default,
                    "valid_concept_default":
                        concept_default,
                    "valid_template_tuned":
                        template_tuned,
                    "valid_concept_tuned":
                        concept_tuned,
                    "valid_concept_by_entity_type":
                        concept_by_type,
                },
                thresholds=
                    tuned_thresholds,
                args=args,
            )

            save_json(
                {
                    label: float(
                        tuned_thresholds[i]
                    )
                    for i, label
                    in enumerate(
                        ASSERTION_LABELS
                    )
                },
                output_root
                / "best_thresholds.json",
            )

        else:
            no_improvement += 1

            print(
                f"\nNo improvement: "
                f"{no_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        print(
            f"\nEpoch {epoch} done | "
            f"train_loss="
            f"{train_loss:.6f} | "
            f"best_epoch="
            f"{best_epoch} | "
            f"best_Jaccard="
            f"{best_score:.6f}"
        )

        print(
            "Best thresholds: "
            + ", ".join(
                f"{label}="
                f"{best_thresholds[i]:.2f}"
                for i, label
                in enumerate(
                    ASSERTION_LABELS
                )
            )
            + "\n"
        )

        if (
            no_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "Early stopping triggered."
            )
            break

        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    print("=" * 78)
    print(
        "PHASE 2 TRAINING COMPLETE"
    )
    print("=" * 78)

    print(
        f"Best epoch              : "
        f"{best_epoch}"
    )
    print(
        f"Best valid_concept "
        f"Jaccard : "
        f"{best_score:.6f}"
    )

    print(
        "Best thresholds         : "
        + ", ".join(
            f"{label}="
            f"{best_thresholds[i]:.2f}"
            for i, label
            in enumerate(
                ASSERTION_LABELS
            )
        )
    )

    print(
        f"Best checkpoint         : "
        f"{output_root / 'best'}"
    )

    print(
        f"Last checkpoint         : "
        f"{output_root / 'last'}"
    )

    print(
        "\nUse the model in /best together "
        "with /best/thresholds.json for inference."
    )


if __name__ == "__main__":
    main()

[INFO] Ignoring unknown Jupyter/Colab args: ['-f', '/root/.local/share/jupyter/runtime/kernel-44d944da-ee30-424f-b634-5ac2dd394768.json']
PHASE 2 - CLINICAL ASSERTION CLASSIFICATION
train          : /content/drive/MyDrive/data/assertion_train_balanced.jsonl
valid_template : /content/drive/MyDrive/data/assertion_valid_template.jsonl
valid_concept  : /content/drive/MyDrive/data/assertion_valid_concept.jsonl
model          : /content/drive/MyDrive/output/best
output         : /content/drive/MyDrive/output_phase2_assertion
device         : cuda
GPU            : Tesla T4

Loading config/tokenizer...
Added special tokens: 5

Loading datasets...
[train] validating 32,000 rows...
[train] encoded=32,000 | left_context_cropped=0 | right_context_cropped=1
[train] no_assertion=8,000
[train] isNegated=8,000
[train] isFamily=8,000
[train] isHistorical=8,000
[valid_template] validating 5,659 rows...
[valid_template] encoded=5,659 | left_context_cropped=5 | right_context_cropped=6
[valid_template] no_

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `11`.


[valid_concept] encoded=5,678 | left_context_cropped=0 | right_context_cropped=0
[valid_concept] no_assertion=2,881
[valid_concept] isNegated=730
[valid_concept] isFamily=293
[valid_concept] isHistorical=1,774

Loading multi-label model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/output/best
Key                        | Status     | 
---------------------------+------------+-
classifier.weight          | UNEXPECTED | 
classifier.bias            | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


...


*** NEW BEST *** epoch=5 | valid_concept_tuned_Jaccard=0.907626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 5 done | train_loss=0.000507 | best_epoch=5 | best_Jaccard=0.907626
Best thresholds: isNegated=0.10, isFamily=0.50, isHistorical=0.30

Epoch 06 | step 0100/2000 | loss=0.00043 | lr=3.519e-06
Epoch 06 | step 0200/2000 | loss=0.00043 | lr=3.333e-06
Epoch 06 | step 0300/2000 | loss=0.00043 | lr=3.148e-06
Epoch 06 | step 0400/2000 | loss=0.00043 | lr=2.963e-06
Epoch 06 | step 0500/2000 | loss=0.00042 | lr=2.778e-06
Epoch 06 | step 0600/2000 | loss=0.00042 | lr=2.593e-06
Epoch 06 | step 0700/2000 | loss=0.00042 | lr=2.407e-06
Epoch 06 | step 0800/2000 | loss=0.00042 | lr=2.222e-06
Epoch 06 | step 0900/2000 | loss=0.00042 | lr=2.037e-06
Epoch 06 | step 1000/2000 | loss=0.00041 | lr=1.852e-06
Epoch 06 | step 1100/2000 | loss=0.00041 | lr=1.667e-06
Epoch 06 | step 1200/2000 | loss=0.00041 | lr=1.481e-06
Epoch 06 | step 1300/2000 | loss=0.00041 | lr=1.296e-06
Epoch 06 | step 1400/2000 | loss=0.00041 | lr=1.111e-06
Epoch 06 | step 1500/2000 | loss=0.00041 | lr=9.259e-07
Epoch 06 | step 16

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


No improvement: 1/3

Epoch 6 done | train_loss=0.000401 | best_epoch=5 | best_Jaccard=0.907626
Best thresholds: isNegated=0.10, isFamily=0.50, isHistorical=0.30

PHASE 2 TRAINING COMPLETE
Best epoch              : 5
Best valid_concept Jaccard : 0.907626
Best thresholds         : isNegated=0.10, isFamily=0.50, isHistorical=0.30
Best checkpoint         : /content/drive/MyDrive/output_phase2_assertion/best
Last checkpoint         : /content/drive/MyDrive/output_phase2_assertion/last

Use the model in /best together with /best/thresholds.json for inference.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
PHASE 3 — ICD-10 Cross-Lingual Retriever
========================================

Goal
----
Map a Vietnamese diagnosis mention to a small candidate set of ICD-10 codes.

Input:
    Vietnamese diagnosis mention
        e.g. "viêm phổi không xác định tác nhân"

Output:
    Top-K ICD-10 candidates
        e.g. J18.9, J18.8, J15.9, ...

This is RETRIEVAL, not final reranking.

Why multilingual E5?
---------------------
The competition mention is mainly Vietnamese while the provided ICD-10 KB is
English. We therefore use a multilingual bi-encoder rather than ViHealthBERT.

Default model:
    intfloat/multilingual-e5-base

E5 retrieval convention:
    query text   -> "query: ..."
    ICD document -> "passage: ..."

Data expected
-------------
1) icd_link_train.jsonl
   {
     "query": "...",
     "positive_codes": ["J18.9"],
     "positives": [...],
     "hard_negatives": [
       {"code": "J18.8", ...},
       ...
     ]
   }

2) icd_link_valid_concept.jsonl
   Same schema, but ICD concepts are held out from training.

3) icd10.csv
   Required columns:
       code
       title_en
       search_text_en
   Optional:
       block_title
       chapter_title

Scientific safeguards
---------------------
- The synthetic train file contains many repeated query/code pairs.
  We DEDUPLICATE (query, positive_code) pairs and merge their hard-negative pools.
- We evaluate ZERO-SHOT E5 before fine-tuning.
- Best model is selected primarily by FULL-CATALOG Recall@20 on held-out ICD
  concepts; MRR breaks ties.
- If fine-tuning hurts held-out retrieval, the epoch-0 zero-shot model can remain
  the best checkpoint.
- Full validation retrieval is against ALL codes in icd10.csv, not only the
  supplied hard negatives.
- Bottom transformer layers are frozen by default to reduce catastrophic
  forgetting from the small synthetic concept set.

Output
------
OUTPUT_DIR/
    run_config.json
    zero_shot_metrics.json
    training_history.json
    best/
        config.json / model.safetensors / tokenizer...
        training_meta.json
        retrieval_config.json
        icd_embeddings.npy
        icd_metadata.jsonl
        validation_top20.jsonl
        icd.faiss              # only if faiss is installed
    last/
        model/tokenizer files...

Colab-safe:
    parse_known_args() ignores Jupyter's injected "-f kernel-xxxx.json".

Run:
    python train_phase3_icd_retriever.py
"""

import os
import gc
import json
import math
import time
import random
import argparse
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)


# ============================================================
# 0. PATHS — EDIT THIS BLOCK
# ============================================================

DATA_ROOT = "/content/drive/MyDrive/data"

TRAIN_JSONL = (
    f"{DATA_ROOT}/icd_link_train.jsonl"
)

VALID_JSONL = (
    f"{DATA_ROOT}/icd_link_valid_concept.jsonl"
)

ICD10_CSV = (
    f"{DATA_ROOT}/icd10.csv"
)

# Internet ON
MODEL_NAME_OR_PATH = (
    "intfloat/multilingual-e5-base"
)
LOCAL_FILES_ONLY = False

# Internet OFF example:
# MODEL_NAME_OR_PATH = "/content/drive/MyDrive/models/multilingual-e5-base"
# LOCAL_FILES_ONLY = True

OUTPUT_DIR = (
    "/content/drive/MyDrive/output_phase3_icd_retriever"
)


# ============================================================
# 1. TRAIN CONFIG
# ============================================================

SEED = 42

QUERY_MAX_LENGTH = 64
DOC_MAX_LENGTH = 192

# Conservative for T4/L4.
TRAIN_BATCH_SIZE = 4
EVAL_QUERY_BATCH_SIZE = 64
ENCODE_DOC_BATCH_SIZE = 64

GRAD_ACCUM_STEPS = 4

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01

NUM_EPOCHS = 5
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

# Each unique (query, positive_code) is repeated with different hard negatives.
PAIR_REPEAT_FACTOR = 8
HARD_NEGATIVES_PER_QUERY = 4

# Multi-class contrastive temperature over:
#   1 positive + N explicit hard negatives.
TEMPERATURE = 0.05

# Preserve multilingual knowledge on tiny synthetic dataset.
FREEZE_BOTTOM_N_LAYERS = 6
FREEZE_EMBEDDINGS = True

USE_GRADIENT_CHECKPOINTING = True
USE_AMP = True

NUM_WORKERS = 2
PIN_MEMORY = True

EARLY_STOPPING_PATIENCE = 2

# Full-catalog retrieval cutoffs.
RECALL_KS = [1, 5, 10, 20]
SAVE_TOP_K = 20

# Primary selection = Recall@20.
SELECTION_K = 20
MIN_DELTA = 1e-6


# ============================================================
# 2. CLI / UTILITIES
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument(
        "--train_jsonl",
        type=str,
        default=TRAIN_JSONL,
    )
    p.add_argument(
        "--valid_jsonl",
        type=str,
        default=VALID_JSONL,
    )
    p.add_argument(
        "--icd10_csv",
        type=str,
        default=ICD10_CSV,
    )
    p.add_argument(
        "--model_name_or_path",
        type=str,
        default=MODEL_NAME_OR_PATH,
    )
    p.add_argument(
        "--output_dir",
        type=str,
        default=OUTPUT_DIR,
    )

    p.add_argument(
        "--query_max_length",
        type=int,
        default=QUERY_MAX_LENGTH,
    )
    p.add_argument(
        "--doc_max_length",
        type=int,
        default=DOC_MAX_LENGTH,
    )

    p.add_argument(
        "--train_batch_size",
        type=int,
        default=TRAIN_BATCH_SIZE,
    )
    p.add_argument(
        "--eval_query_batch_size",
        type=int,
        default=EVAL_QUERY_BATCH_SIZE,
    )
    p.add_argument(
        "--encode_doc_batch_size",
        type=int,
        default=ENCODE_DOC_BATCH_SIZE,
    )
    p.add_argument(
        "--grad_accum_steps",
        type=int,
        default=GRAD_ACCUM_STEPS,
    )

    p.add_argument(
        "--lr",
        type=float,
        default=LEARNING_RATE,
    )
    p.add_argument(
        "--weight_decay",
        type=float,
        default=WEIGHT_DECAY,
    )
    p.add_argument(
        "--epochs",
        type=int,
        default=NUM_EPOCHS,
    )
    p.add_argument(
        "--warmup_ratio",
        type=float,
        default=WARMUP_RATIO,
    )

    p.add_argument(
        "--pair_repeat_factor",
        type=int,
        default=PAIR_REPEAT_FACTOR,
    )
    p.add_argument(
        "--hard_negatives_per_query",
        type=int,
        default=HARD_NEGATIVES_PER_QUERY,
    )
    p.add_argument(
        "--temperature",
        type=float,
        default=TEMPERATURE,
    )

    p.add_argument(
        "--freeze_bottom_n_layers",
        type=int,
        default=FREEZE_BOTTOM_N_LAYERS,
    )

    p.add_argument(
        "--seed",
        type=int,
        default=SEED,
    )

    p.add_argument(
        "--local_files_only",
        action="store_true",
        default=LOCAL_FILES_ONLY,
    )

    args, unknown = p.parse_known_args()

    if unknown:
        print(
            "[INFO] Ignoring unknown Jupyter/Colab args:",
            unknown,
        )

    return args


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing JSONL:\n  {p}\n\n"
            "Edit DATA_ROOT/PATHS at the top of the script "
            "or use CLI overrides."
        )

    rows = []

    with p.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                rows.append(
                    json.loads(line)
                )
            except Exception as exc:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{p}:{line_no}: {exc}"
                )

    return rows


def write_jsonl(
    rows: List[Dict[str, Any]],
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def save_json(
    obj: Any,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
        )


# ============================================================
# 3. ICD CATALOG
# ============================================================

def clean_text(x: Any) -> str:
    if x is None:
        return ""

    s = str(x).strip()

    if s.lower() == "nan":
        return ""

    return " ".join(
        s.split()
    )


def make_icd_document(
    row: Dict[str, Any],
) -> str:
    """
    Build the English document that E5 will encode.

    We keep code as metadata, NOT as semantic text.
    """

    title = clean_text(
        row.get(
            "title_en",
            "",
        )
    )

    search_text = clean_text(
        row.get(
            "search_text_en",
            "",
        )
    )

    block_title = clean_text(
        row.get(
            "block_title",
            "",
        )
    )

    chapter_title = clean_text(
        row.get(
            "chapter_title",
            "",
        )
    )

    # search_text_en already starts with title in most rows.
    base = (
        search_text
        if search_text
        else title
    )

    extras = []

    if (
        block_title
        and block_title.lower()
        not in base.lower()
    ):
        extras.append(
            f"Category: {block_title}"
        )

    if (
        chapter_title
        and chapter_title.lower()
        not in base.lower()
    ):
        extras.append(
            f"Chapter: {chapter_title}"
        )

    if extras:
        base = (
            base
            + " || "
            + " || ".join(extras)
        )

    if not base:
        raise ValueError(
            f"Empty ICD document for code "
            f"{row.get('code')!r}"
        )

    return (
        "passage: "
        + base
    )


def load_icd_catalog(
    path: str,
) -> Tuple[
    List[Dict[str, Any]],
    Dict[str, int],
]:
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing ICD CSV:\n  {p}"
        )

    df = pd.read_csv(
        p,
        dtype=str,
    ).fillna("")

    required = {
        "code",
        "title_en",
        "search_text_en",
    }

    missing = (
        required
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            "ICD CSV missing columns: "
            + str(
                sorted(missing)
            )
        )

    df = (
        df.drop_duplicates(
            subset=["code"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    records = []

    for row in df.to_dict(
        orient="records"
    ):
        code = clean_text(
            row["code"]
        )

        if not code:
            continue

        records.append(
            {
                "code": code,
                "title_en": clean_text(
                    row.get(
                        "title_en",
                        "",
                    )
                ),
                "search_text_en": clean_text(
                    row.get(
                        "search_text_en",
                        "",
                    )
                ),
                "block_code": clean_text(
                    row.get(
                        "block_code",
                        "",
                    )
                ),
                "block_title": clean_text(
                    row.get(
                        "block_title",
                        "",
                    )
                ),
                "chapter_code": clean_text(
                    row.get(
                        "chapter_code",
                        "",
                    )
                ),
                "chapter_title": clean_text(
                    row.get(
                        "chapter_title",
                        "",
                    )
                ),
                "document": make_icd_document(
                    row
                ),
            }
        )

    code_to_index = {
        row["code"]: i
        for i, row
        in enumerate(records)
    }

    print(
        f"[ICD] catalog codes="
        f"{len(records):,}"
    )

    return (
        records,
        code_to_index,
    )


# ============================================================
# 4. TRAIN DATA — DEDUPLICATE SYNTHETIC REPEATS
# ============================================================

class ICDPairDataset(Dataset):
    """
    Deduplicates by:
        (query, positive_code)

    For each unique pair, merge all hard-negative codes observed in the
    original synthetic file.

    Dataset length:
        unique_pairs * repeat_factor

    Repeat slots rotate/shuffle through the merged hard-negative pool.
    """

    def __init__(
        self,
        path: str,
        catalog_records:
            List[Dict[str, Any]],
        code_to_index:
            Dict[str, int],
        repeat_factor: int,
        hard_negatives_per_query: int,
        seed: int,
    ):
        self.path = path
        self.catalog_records = (
            catalog_records
        )
        self.code_to_index = (
            code_to_index
        )
        self.repeat_factor = int(
            repeat_factor
        )
        self.hard_n = int(
            hard_negatives_per_query
        )
        self.seed = int(seed)

        raw_rows = read_jsonl(path)

        merged = {}

        raw_positive_pairs = 0

        for row in raw_rows:
            query = clean_text(
                row.get(
                    "query",
                    "",
                )
            )

            if not query:
                continue

            positive_codes = [
                clean_text(x)
                for x in row.get(
                    "positive_codes",
                    [],
                )
                if clean_text(x)
                in code_to_index
            ]

            hard_codes = []

            for neg in row.get(
                "hard_negatives",
                [],
            ):
                code = clean_text(
                    neg.get(
                        "code",
                        "",
                    )
                )

                if (
                    code
                    and code
                    in code_to_index
                ):
                    hard_codes.append(
                        code
                    )

            positive_set = set(
                positive_codes
            )

            for pos_code in positive_codes:
                raw_positive_pairs += 1

                key = (
                    query,
                    pos_code,
                )

                if key not in merged:
                    merged[key] = {
                        "query": query,
                        "positive_code":
                            pos_code,
                        "negative_pool": set(),
                    }

                for neg_code in hard_codes:
                    if (
                        neg_code
                        not in positive_set
                    ):
                        merged[key][
                            "negative_pool"
                        ].add(
                            neg_code
                        )

        self.unique_pairs = []

        all_codes = [
            x["code"]
            for x in catalog_records
        ]

        rng = random.Random(
            self.seed
        )

        for item in merged.values():
            negatives = sorted(
                item["negative_pool"]
            )

            if len(
                negatives
            ) < self.hard_n:
                fallback = [
                    c
                    for c in all_codes
                    if c
                    != item[
                        "positive_code"
                    ]
                    and c
                    not in negatives
                ]

                rng.shuffle(
                    fallback
                )

                negatives.extend(
                    fallback[
                        : self.hard_n
                        - len(negatives)
                    ]
                )

            self.unique_pairs.append(
                {
                    "query":
                        item["query"],
                    "positive_code":
                        item[
                            "positive_code"
                        ],
                    "negative_pool":
                        negatives,
                }
            )

        # Stable ordering, then deterministic shuffle.
        self.unique_pairs.sort(
            key=lambda x: (
                x["query"],
                x["positive_code"],
            )
        )

        rng.shuffle(
            self.unique_pairs
        )

        self.raw_rows = len(
            raw_rows
        )

        self.raw_positive_pairs = (
            raw_positive_pairs
        )

        unique_codes = {
            x["positive_code"]
            for x
            in self.unique_pairs
        }

        unique_queries = {
            x["query"]
            for x
            in self.unique_pairs
        }

        print(
            "[TRAIN DATA] "
            f"raw_rows={self.raw_rows:,} | "
            f"raw_positive_pairs="
            f"{self.raw_positive_pairs:,}"
        )

        print(
            "[TRAIN DATA] "
            f"unique_queries="
            f"{len(unique_queries):,} | "
            f"unique_query_code_pairs="
            f"{len(self.unique_pairs):,} | "
            f"unique_positive_codes="
            f"{len(unique_codes):,}"
        )

        print(
            "[TRAIN DATA] "
            f"repeat_factor="
            f"{self.repeat_factor} | "
            f"effective_samples/epoch="
            f"{len(self):,}"
        )

    def __len__(self):
        return (
            len(self.unique_pairs)
            * self.repeat_factor
        )

    def __getitem__(self, idx):
        pair_idx = (
            idx
            % len(
                self.unique_pairs
            )
        )

        repeat_slot = (
            idx
            // len(
                self.unique_pairs
            )
        )

        item = self.unique_pairs[
            pair_idx
        ]

        neg_pool = item[
            "negative_pool"
        ]

        # Deterministic negative rotation per pair/repeat slot.
        local_rng = random.Random(
            self.seed
            + pair_idx * 10007
            + repeat_slot * 1000003
        )

        if (
            len(neg_pool)
            <= self.hard_n
        ):
            negatives = list(
                neg_pool
            )
        else:
            negatives = (
                local_rng.sample(
                    neg_pool,
                    self.hard_n,
                )
            )

        return {
            "query":
                item["query"],
            "positive_code":
                item[
                    "positive_code"
                ],
            "negative_codes":
                negatives,
        }


class ICDTrainCollator:
    def __init__(
        self,
        tokenizer,
        catalog_records,
        code_to_index,
        query_max_length: int,
        doc_max_length: int,
    ):
        self.tokenizer = tokenizer
        self.catalog = (
            catalog_records
        )
        self.code_to_index = (
            code_to_index
        )
        self.query_max_length = int(
            query_max_length
        )
        self.doc_max_length = int(
            doc_max_length
        )

    def _doc_text(
        self,
        code: str,
    ) -> str:
        idx = self.code_to_index[
            code
        ]

        return self.catalog[
            idx
        ]["document"]

    def __call__(
        self,
        features,
    ):
        queries = [
            "query: "
            + x["query"]
            for x in features
        ]

        # Candidate 0 = positive.
        candidate_codes = []

        for x in features:
            codes = [
                x["positive_code"]
            ] + list(
                x["negative_codes"]
            )

            candidate_codes.append(
                codes
            )

        candidate_count = len(
            candidate_codes[0]
        )

        if not all(
            len(x)
            == candidate_count
            for x in candidate_codes
        ):
            raise RuntimeError(
                "Candidate count differs "
                "within a training batch."
            )

        flat_docs = []

        for codes in candidate_codes:
            for code in codes:
                flat_docs.append(
                    self._doc_text(
                        code
                    )
                )

        query_batch = (
            self.tokenizer(
                queries,
                padding=True,
                truncation=True,
                max_length=
                    self.query_max_length,
                return_tensors="pt",
            )
        )

        doc_batch = (
            self.tokenizer(
                flat_docs,
                padding=True,
                truncation=True,
                max_length=
                    self.doc_max_length,
                return_tensors="pt",
            )
        )

        return {
            "query_batch":
                query_batch,
            "doc_batch":
                doc_batch,
            "candidate_count":
                candidate_count,
            "positive_codes": [
                x["positive_code"]
                for x in features
            ],
            "candidate_codes":
                candidate_codes,
        }


# ============================================================
# 5. VALIDATION DATA
# ============================================================

def load_validation(
    path: str,
    code_to_index:
        Dict[str, int],
) -> List[Dict[str, Any]]:

    raw_rows = read_jsonl(path)

    # Validation file itself has synthetic repeats.
    # Deduplicate by (query, tuple(gold_codes)).
    unique = {}

    for row in raw_rows:
        query = clean_text(
            row.get(
                "query",
                "",
            )
        )

        gold = sorted(
            {
                clean_text(x)
                for x
                in row.get(
                    "positive_codes",
                    []
                )
                if clean_text(x)
                in code_to_index
            }
        )

        if (
            not query
            or not gold
        ):
            continue

        key = (
            query,
            tuple(gold),
        )

        unique[key] = {
            "query": query,
            "gold_codes": gold,
        }

    rows = list(
        unique.values()
    )

    rows.sort(
        key=lambda x: (
            x["query"],
            tuple(
                x["gold_codes"]
            ),
        )
    )

    unique_codes = {
        c
        for row in rows
        for c in row[
            "gold_codes"
        ]
    }

    print(
        "[VALID DATA] "
        f"raw_rows={len(raw_rows):,} | "
        f"unique_queries="
        f"{len(rows):,} | "
        f"heldout_gold_codes="
        f"{len(unique_codes):,}"
    )

    return rows


# ============================================================
# 6. MODEL ENCODING
# ============================================================

def average_pool(
    last_hidden_states:
        torch.Tensor,
    attention_mask:
        torch.Tensor,
) -> torch.Tensor:

    mask = (
        attention_mask[
            ...,
            None,
        ]
        .bool()
    )

    hidden = (
        last_hidden_states
        .masked_fill(
            ~mask,
            0.0,
        )
    )

    denom = (
        attention_mask
        .sum(dim=1)
        [..., None]
        .clamp(min=1)
    )

    return (
        hidden.sum(
            dim=1
        )
        / denom
    )


def encode_batch(
    model,
    batch,
) -> torch.Tensor:

    outputs = model(
        **batch
    )

    emb = average_pool(
        outputs.last_hidden_state,
        batch[
            "attention_mask"
        ],
    )

    emb = F.normalize(
        emb,
        p=2,
        dim=1,
    )

    return emb


def move_token_batch(
    batch,
    device,
):
    return {
        k: v.to(
            device,
            non_blocking=True,
        )
        for k, v
        in batch.items()
    }


# ============================================================
# 7. FREEZING / PARAMETER REPORT
# ============================================================

def freeze_bottom_layers(
    model,
    n_layers: int,
    freeze_embeddings: bool,
):
    """
    multilingual-e5-base is XLM-R based:
        model.embeddings
        model.encoder.layer
    """

    if (
        freeze_embeddings
        and hasattr(
            model,
            "embeddings",
        )
    ):
        for p in (
            model.embeddings
            .parameters()
        ):
            p.requires_grad = False

        print(
            "[FREEZE] embeddings frozen"
        )

    layers = None

    if (
        hasattr(
            model,
            "encoder",
        )
        and hasattr(
            model.encoder,
            "layer",
        )
    ):
        layers = (
            model.encoder.layer
        )

    if layers is None:
        print(
            "[WARN] Could not locate "
            "model.encoder.layer; "
            "no transformer layers frozen."
        )
        return

    n_layers = max(
        0,
        min(
            int(n_layers),
            len(layers),
        ),
    )

    for layer in layers[
        :n_layers
    ]:
        for p in (
            layer.parameters()
        ):
            p.requires_grad = False

    print(
        f"[FREEZE] frozen bottom "
        f"{n_layers}/{len(layers)} "
        f"transformer layers"
    )


def count_parameters(
    model,
):
    total = sum(
        p.numel()
        for p
        in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p
        in model.parameters()
        if p.requires_grad
    )

    print(
        f"[PARAMS] total="
        f"{total:,} | "
        f"trainable="
        f"{trainable:,} | "
        f"trainable%="
        f"{100.0 * trainable / total:.2f}%"
    )


# ============================================================
# 8. FULL ICD CATALOG ENCODING
# ============================================================

@torch.no_grad()
def encode_catalog(
    model,
    tokenizer,
    catalog_records,
    device,
    batch_size: int,
    doc_max_length: int,
    use_amp=True,
) -> np.ndarray:

    model.eval()

    amp_enabled = bool(
        use_amp
        and device.type == "cuda"
    )

    all_embeddings = []

    docs = [
        x["document"]
        for x in catalog_records
    ]

    for start in range(
        0,
        len(docs),
        batch_size,
    ):
        batch_text = docs[
            start:
            start + batch_size
        ]

        token_batch = (
            tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=
                    doc_max_length,
                return_tensors="pt",
            )
        )

        token_batch = (
            move_token_batch(
                token_batch,
                device,
            )
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            emb = encode_batch(
                model,
                token_batch,
            )

        all_embeddings.append(
            emb
            .float()
            .cpu()
            .numpy()
        )

        if (
            (start // batch_size)
            % 50
            == 0
        ):
            print(
                f"[ICD ENCODE] "
                f"{min(start + batch_size, len(docs)):,}"
                f"/{len(docs):,}"
            )

    matrix = np.concatenate(
        all_embeddings,
        axis=0,
    ).astype(
        np.float32
    )

    # Numerical safety.
    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    matrix = (
        matrix
        / np.clip(
            norms,
            1e-12,
            None,
        )
    )

    return matrix


# ============================================================
# 9. FULL-CATALOG VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_full_catalog(
    model,
    tokenizer,
    valid_rows,
    catalog_records,
    code_to_index,
    catalog_embeddings:
        np.ndarray,
    device,
    query_batch_size: int,
    query_max_length: int,
    save_top_k: int,
    use_amp=True,
):
    model.eval()

    amp_enabled = bool(
        use_amp
        and device.type == "cuda"
    )

    # ~11k x 768 is small enough to keep on GPU for evaluation.
    catalog_tensor = (
        torch.from_numpy(
            catalog_embeddings
        )
        .to(
            device=device,
            dtype=torch.float32,
        )
    )

    ranks = []
    prediction_rows = []

    hit_counts = {
        k: 0
        for k in RECALL_KS
    }

    for start in range(
        0,
        len(valid_rows),
        query_batch_size,
    ):
        rows = valid_rows[
            start:
            start + query_batch_size
        ]

        queries = [
            "query: "
            + row["query"]
            for row in rows
        ]

        token_batch = (
            tokenizer(
                queries,
                padding=True,
                truncation=True,
                max_length=
                    query_max_length,
                return_tensors="pt",
            )
        )

        token_batch = (
            move_token_batch(
                token_batch,
                device,
            )
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            q_emb = encode_batch(
                model,
                token_batch,
            )

        # Keep ranking scores in float32.
        scores = (
            q_emb.float()
            @ catalog_tensor.T
        )

        # Need top K for output, but MRR requires exact rank of best gold.
        # Catalog is only ~11k codes, so full sorting is acceptable.
        sorted_idx = torch.argsort(
            scores,
            dim=1,
            descending=True,
        )

        top_idx = sorted_idx[
            :,
            :save_top_k,
        ]

        scores_cpu = (
            scores
            .detach()
            .cpu()
            .numpy()
        )

        sorted_cpu = (
            sorted_idx
            .detach()
            .cpu()
            .numpy()
        )

        top_cpu = (
            top_idx
            .detach()
            .cpu()
            .numpy()
        )

        for i, row in enumerate(
            rows
        ):
            gold_codes = set(
                row["gold_codes"]
            )

            gold_indices = {
                code_to_index[c]
                for c in gold_codes
            }

            ranked = (
                sorted_cpu[i]
                .tolist()
            )

            best_rank = None

            for rank0, idx in enumerate(
                ranked
            ):
                if idx in gold_indices:
                    best_rank = (
                        rank0 + 1
                    )
                    break

            if best_rank is None:
                # Should be impossible because gold codes were filtered to catalog.
                best_rank = (
                    len(
                        catalog_records
                    )
                    + 1
                )

            ranks.append(
                best_rank
            )

            for k in RECALL_KS:
                if best_rank <= k:
                    hit_counts[k] += 1

            preds = []

            for idx in top_cpu[i]:
                meta = (
                    catalog_records[
                        int(idx)
                    ]
                )

                preds.append(
                    {
                        "code":
                            meta["code"],
                        "title_en":
                            meta["title_en"],
                        "score":
                            float(
                                scores_cpu[
                                    i,
                                    int(idx),
                                ]
                            ),
                    }
                )

            prediction_rows.append(
                {
                    "query":
                        row["query"],
                    "gold_codes":
                        row["gold_codes"],
                    "best_gold_rank":
                        int(best_rank),
                    "top_candidates":
                        preds,
                }
            )

    n = len(
        valid_rows
    )

    recall = {
        f"recall@{k}":
            (
                hit_counts[k] / n
                if n
                else 0.0
            )
        for k in RECALL_KS
    }

    mrr = (
        float(
            np.mean(
                [
                    1.0 / r
                    for r in ranks
                ]
            )
        )
        if ranks
        else 0.0
    )

    mean_rank = (
        float(
            np.mean(ranks)
        )
        if ranks
        else math.nan
    )

    median_rank = (
        float(
            np.median(ranks)
        )
        if ranks
        else math.nan
    )

    metrics = {
        **recall,
        "mrr": mrr,
        "mean_best_gold_rank":
            mean_rank,
        "median_best_gold_rank":
            median_rank,
        "n_queries": n,
        "catalog_size":
            len(catalog_records),
    }

    del catalog_tensor
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return (
        metrics,
        prediction_rows,
    )


def print_retrieval_metrics(
    name: str,
    metrics:
        Dict[str, Any],
):
    print(
        f"\n[{name}]"
    )

    for k in RECALL_KS:
        print(
            f"  Recall@{k:<2} = "
            f"{metrics[f'recall@{k}']:.6f}"
        )

    print(
        f"  MRR       = "
        f"{metrics['mrr']:.6f}"
    )

    print(
        f"  Mean rank = "
        f"{metrics['mean_best_gold_rank']:.3f}"
    )

    print(
        f"  Median    = "
        f"{metrics['median_best_gold_rank']:.3f}"
    )


# ============================================================
# 10. SAVE RETRIEVAL ARTIFACTS
# ============================================================

def maybe_save_faiss(
    embeddings:
        np.ndarray,
    path: Path,
):
    try:
        import faiss

        index = (
            faiss.IndexFlatIP(
                embeddings.shape[1]
            )
        )

        index.add(
            embeddings.astype(
                np.float32
            )
        )

        faiss.write_index(
            index,
            str(path),
        )

        print(
            f"[FAISS] saved: {path}"
        )

        return True

    except Exception as exc:
        print(
            "[FAISS] not saved "
            "(optional). "
            f"Reason: {exc}"
        )

        return False


def save_retrieval_bundle(
    model,
    tokenizer,
    directory: Path,
    epoch: int,
    metrics:
        Dict[str, Any],
    predictions:
        List[Dict[str, Any]],
    catalog_records:
        List[Dict[str, Any]],
    catalog_embeddings:
        np.ndarray,
    args,
    baseline_metrics=None,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        directory
    )

    tokenizer.save_pretrained(
        directory
    )

    np.save(
        directory
        / "icd_embeddings.npy",
        catalog_embeddings.astype(
            np.float32
        ),
    )

    metadata_rows = [
        {
            "index": i,
            "code": x["code"],
            "title_en":
                x["title_en"],
            "search_text_en":
                x[
                    "search_text_en"
                ],
            "block_code":
                x["block_code"],
            "block_title":
                x["block_title"],
            "chapter_code":
                x["chapter_code"],
            "chapter_title":
                x["chapter_title"],
        }
        for i, x
        in enumerate(
            catalog_records
        )
    ]

    write_jsonl(
        metadata_rows,
        directory
        / "icd_metadata.jsonl",
    )

    write_jsonl(
        predictions,
        directory
        / "validation_top20.jsonl",
    )

    save_json(
        {
            "model_prefixes": {
                "query":
                    "query: ",
                "passage":
                    "passage: ",
            },
            "query_max_length":
                args.query_max_length,
            "doc_max_length":
                args.doc_max_length,
            "embedding_normalization":
                "L2",
            "similarity":
                "inner_product == cosine after L2 normalization",
            "top_k":
                SAVE_TOP_K,
            "catalog_size":
                len(
                    catalog_records
                ),
        },
        directory
        / "retrieval_config.json",
    )

    save_json(
        {
            "epoch": int(epoch),
            "selection_metric":
                f"recall@{SELECTION_K}",
            "tie_break_metric":
                "mrr",
            "metrics":
                metrics,
            "zero_shot_metrics":
                baseline_metrics,
            "training_args":
                vars(args),
        },
        directory
        / "training_meta.json",
    )

    maybe_save_faiss(
        catalog_embeddings,
        directory
        / "icd.faiss",
    )


def save_model_only(
    model,
    tokenizer,
    directory: Path,
    epoch: int,
    args,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        directory
    )

    tokenizer.save_pretrained(
        directory
    )

    save_json(
        {
            "epoch": int(epoch),
            "training_args":
                vars(args),
        },
        directory
        / "training_meta.json",
    )


# ============================================================
# 11. TRAIN LOSS
# ============================================================

def contrastive_hard_negative_loss(
    query_embeddings:
        torch.Tensor,
    doc_embeddings:
        torch.Tensor,
    batch_size: int,
    candidate_count: int,
    temperature: float,
) -> Tuple[
    torch.Tensor,
    torch.Tensor,
]:

    docs = doc_embeddings.view(
        batch_size,
        candidate_count,
        -1,
    )

    # [B, C]
    scores = torch.einsum(
        "bd,bcd->bc",
        query_embeddings,
        docs,
    )

    scores = (
        scores
        / float(
            temperature
        )
    )

    # Candidate 0 is always positive.
    targets = torch.zeros(
        batch_size,
        dtype=torch.long,
        device=scores.device,
    )

    loss = F.cross_entropy(
        scores,
        targets,
    )

    accuracy = (
        scores.argmax(
            dim=1
        )
        == targets
    ).float().mean()

    return (
        loss,
        accuracy,
    )


# ============================================================
# 12. MAIN
# ============================================================

def main():
    args = parse_args()
    set_seed(args.seed)

    output_root = Path(
        args.output_dir
    )

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    save_json(
        vars(args),
        output_root
        / "run_config.json",
    )

    print("=" * 78)
    print(
        "PHASE 3 - ICD-10 CROSS-LINGUAL RETRIEVER"
    )
    print("=" * 78)

    print(
        f"train       : "
        f"{args.train_jsonl}"
    )
    print(
        f"valid       : "
        f"{args.valid_jsonl}"
    )
    print(
        f"ICD KB      : "
        f"{args.icd10_csv}"
    )
    print(
        f"model       : "
        f"{args.model_name_or_path}"
    )
    print(
        f"output      : "
        f"{args.output_dir}"
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"device      : "
        f"{device}"
    )

    if device.type == "cuda":
        print(
            f"GPU         : "
            f"{torch.cuda.get_device_name(0)}"
        )

    # --------------------------------------------------------
    # ICD catalog
    # --------------------------------------------------------

    catalog_records, (
        code_to_index
    ) = load_icd_catalog(
        args.icd10_csv
    )

    # --------------------------------------------------------
    # Model/tokenizer
    # --------------------------------------------------------

    print(
        "\nLoading multilingual E5..."
    )

    config = (
        AutoConfig
        .from_pretrained(
            args.model_name_or_path,
            local_files_only=
                args.local_files_only,
        )
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            args.model_name_or_path,
            local_files_only=
                args.local_files_only,
        )
    )

    model = (
        AutoModel
        .from_pretrained(
            args.model_name_or_path,
            local_files_only=
                args.local_files_only,
        )
    )

    # Respect actual model max context.
    tokenizer_cap = int(
        getattr(
            tokenizer,
            "model_max_length",
            512,
        )
    )

    # Some tokenizers expose an absurd sentinel max length.
    if tokenizer_cap > 100000:
        tokenizer_cap = 512

    args.query_max_length = min(
        args.query_max_length,
        tokenizer_cap,
    )

    args.doc_max_length = min(
        args.doc_max_length,
        tokenizer_cap,
    )

    freeze_bottom_layers(
        model,
        n_layers=
            args.freeze_bottom_n_layers,
        freeze_embeddings=
            FREEZE_EMBEDDINGS,
    )

    if (
        USE_GRADIENT_CHECKPOINTING
        and hasattr(
            model,
            "gradient_checkpointing_enable",
        )
    ):
        model.gradient_checkpointing_enable()

        print(
            "[MODEL] gradient checkpointing enabled"
        )

    model.to(device)
    count_parameters(model)

    # --------------------------------------------------------
    # Train/valid data
    # --------------------------------------------------------

    train_ds = ICDPairDataset(
        path=args.train_jsonl,
        catalog_records=
            catalog_records,
        code_to_index=
            code_to_index,
        repeat_factor=
            args.pair_repeat_factor,
        hard_negatives_per_query=
            args.hard_negatives_per_query,
        seed=args.seed,
    )

    valid_rows = (
        load_validation(
            path=args.valid_jsonl,
            code_to_index=
                code_to_index,
        )
    )

    train_collator = (
        ICDTrainCollator(
            tokenizer=tokenizer,
            catalog_records=
                catalog_records,
            code_to_index=
                code_to_index,
            query_max_length=
                args.query_max_length,
            doc_max_length=
                args.doc_max_length,
        )
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            args.train_batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=
            train_collator,
    )

    # --------------------------------------------------------
    # ZERO-SHOT gate
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 78
    )
    print(
        "ZERO-SHOT FULL-CATALOG EVALUATION"
    )
    print(
        "=" * 78
    )

    zero_catalog_emb = (
        encode_catalog(
            model=model,
            tokenizer=tokenizer,
            catalog_records=
                catalog_records,
            device=device,
            batch_size=
                args.encode_doc_batch_size,
            doc_max_length=
                args.doc_max_length,
            use_amp=USE_AMP,
        )
    )

    zero_metrics, (
        zero_predictions
    ) = evaluate_full_catalog(
        model=model,
        tokenizer=tokenizer,
        valid_rows=valid_rows,
        catalog_records=
            catalog_records,
        code_to_index=
            code_to_index,
        catalog_embeddings=
            zero_catalog_emb,
        device=device,
        query_batch_size=
            args.eval_query_batch_size,
        query_max_length=
            args.query_max_length,
        save_top_k=
            SAVE_TOP_K,
        use_amp=USE_AMP,
    )

    print_retrieval_metrics(
        "ZERO_SHOT",
        zero_metrics,
    )

    save_json(
        zero_metrics,
        output_root
        / "zero_shot_metrics.json",
    )

    write_jsonl(
        zero_predictions,
        output_root
        / "zero_shot_top20.jsonl",
    )

    # Epoch 0 is a legitimate BEST checkpoint.
    best_recall = float(
        zero_metrics[
            f"recall@{SELECTION_K}"
        ]
    )

    best_mrr = float(
        zero_metrics["mrr"]
    )

    best_epoch = 0
    no_improvement = 0

    print(
        "\n[BASELINE GATE] "
        f"epoch=0 is initial best | "
        f"Recall@{SELECTION_K}="
        f"{best_recall:.6f} | "
        f"MRR={best_mrr:.6f}"
    )

    save_retrieval_bundle(
        model=model,
        tokenizer=tokenizer,
        directory=
            output_root / "best",
        epoch=0,
        metrics=zero_metrics,
        predictions=
            zero_predictions,
        catalog_records=
            catalog_records,
        catalog_embeddings=
            zero_catalog_emb,
        args=args,
        baseline_metrics=
            zero_metrics,
    )

    # Free CPU array before training if desired;
    # embeddings are saved in /best.
    del zero_catalog_emb
    gc.collect()

    # --------------------------------------------------------
    # Optimizer / scheduler
    # --------------------------------------------------------

    no_decay = (
        "bias",
        "LayerNorm.weight",
        "layer_norm.weight",
    )

    trainable_named = [
        (n, p)
        for n, p
        in model.named_parameters()
        if p.requires_grad
    ]

    optimizer_groups = [
        {
            "params": [
                p
                for n, p
                in trainable_named
                if not any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay":
                args.weight_decay,
        },
        {
            "params": [
                p
                for n, p
                in trainable_named
                if any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = AdamW(
        optimizer_groups,
        lr=args.lr,
    )

    updates_per_epoch = math.ceil(
        len(train_loader)
        / args.grad_accum_steps
    )

    total_steps = (
        updates_per_epoch
        * args.epochs
    )

    warmup_steps = int(
        total_steps
        * args.warmup_ratio
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=
                warmup_steps,
            num_training_steps=
                total_steps,
        )
    )

    amp_enabled = bool(
        USE_AMP
        and device.type == "cuda"
    )

    try:
        scaler = (
            torch.amp.GradScaler(
                "cuda",
                enabled=amp_enabled,
            )
        )
    except Exception:
        scaler = (
            torch.cuda.amp.GradScaler(
                enabled=amp_enabled
            )
        )

    print(
        "\nTraining setup"
    )
    print(
        f"  unique pairs          : "
        f"{len(train_ds.unique_pairs):,}"
    )
    print(
        f"  effective samples     : "
        f"{len(train_ds):,}"
    )
    print(
        f"  query batch           : "
        f"{args.train_batch_size}"
    )
    print(
        f"  explicit hard neg/q   : "
        f"{args.hard_negatives_per_query}"
    )
    print(
        f"  grad accum            : "
        f"{args.grad_accum_steps}"
    )
    print(
        f"  effective q batch     : "
        f"{args.train_batch_size * args.grad_accum_steps}"
    )
    print(
        f"  query max len         : "
        f"{args.query_max_length}"
    )
    print(
        f"  doc max len           : "
        f"{args.doc_max_length}"
    )
    print(
        f"  temperature           : "
        f"{args.temperature}"
    )
    print(
        f"  LR                    : "
        f"{args.lr}"
    )
    print(
        f"  epochs                : "
        f"{args.epochs}"
    )
    print(
        f"  optimizer steps       : "
        f"{total_steps}"
    )
    print(
        f"  warmup steps          : "
        f"{warmup_steps}"
    )
    print(
        f"  selection             : "
        f"Recall@{SELECTION_K}, MRR tie-break"
    )
    print(
        f"  AMP                   : "
        f"{amp_enabled}"
    )

    history = []

    # --------------------------------------------------------
    # Fine-tune
    # --------------------------------------------------------

    for epoch in range(
        1,
        args.epochs + 1,
    ):
        epoch_start = time.time()

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        running_loss = 0.0
        running_acc = 0.0

        for step, batch in enumerate(
            train_loader,
            1,
        ):
            query_batch = (
                move_token_batch(
                    batch[
                        "query_batch"
                    ],
                    device,
                )
            )

            doc_batch = (
                move_token_batch(
                    batch[
                        "doc_batch"
                    ],
                    device,
                )
            )

            candidate_count = int(
                batch[
                    "candidate_count"
                ]
            )

            batch_size = (
                query_batch[
                    "input_ids"
                ].shape[0]
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                q_emb = encode_batch(
                    model,
                    query_batch,
                )

                d_emb = encode_batch(
                    model,
                    doc_batch,
                )

                full_loss, (
                    train_acc
                ) = (
                    contrastive_hard_negative_loss(
                        query_embeddings=
                            q_emb,
                        doc_embeddings=
                            d_emb,
                        batch_size=
                            batch_size,
                        candidate_count=
                            candidate_count,
                        temperature=
                            args.temperature,
                    )
                )

                loss = (
                    full_loss
                    / args.grad_accum_steps
                )

            scaler.scale(
                loss
            ).backward()

            running_loss += float(
                full_loss
                .detach()
                .cpu()
            )

            running_acc += float(
                train_acc
                .detach()
                .cpu()
            )

            should_update = (
                step
                % args.grad_accum_steps
                == 0
                or step
                == len(train_loader)
            )

            if should_update:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    [
                        p
                        for p
                        in model.parameters()
                        if p.requires_grad
                    ],
                    MAX_GRAD_NORM,
                )

                scaler.step(
                    optimizer
                )
                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

            if (
                step % 50 == 0
                or step
                == len(train_loader)
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"step "
                    f"{step:04d}/"
                    f"{len(train_loader):04d} | "
                    f"loss="
                    f"{running_loss / step:.5f} | "
                    f"hardneg_acc="
                    f"{running_acc / step:.4f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.3e}"
                )

        train_loss = (
            running_loss
            / max(
                1,
                len(train_loader),
            )
        )

        train_acc = (
            running_acc
            / max(
                1,
                len(train_loader),
            )
        )

        # ----------------------------------------------------
        # Full-catalog validation
        # ----------------------------------------------------

        print(
            "\nEncoding full ICD catalog "
            f"for epoch {epoch}..."
        )

        catalog_emb = (
            encode_catalog(
                model=model,
                tokenizer=tokenizer,
                catalog_records=
                    catalog_records,
                device=device,
                batch_size=
                    args.encode_doc_batch_size,
                doc_max_length=
                    args.doc_max_length,
                use_amp=USE_AMP,
            )
        )

        metrics, predictions = (
            evaluate_full_catalog(
                model=model,
                tokenizer=tokenizer,
                valid_rows=
                    valid_rows,
                catalog_records=
                    catalog_records,
                code_to_index=
                    code_to_index,
                catalog_embeddings=
                    catalog_emb,
                device=device,
                query_batch_size=
                    args.eval_query_batch_size,
                query_max_length=
                    args.query_max_length,
                save_top_k=
                    SAVE_TOP_K,
                use_amp=USE_AMP,
            )
        )

        print_retrieval_metrics(
            f"VALID_CONCEPT_EPOCH_{epoch}",
            metrics,
        )

        epoch_record = {
            "epoch": epoch,
            "train_loss":
                train_loss,
            "train_hardneg_accuracy":
                train_acc,
            "valid": metrics,
            "seconds":
                time.time()
                - epoch_start,
        }

        history.append(
            epoch_record
        )

        save_json(
            history,
            output_root
            / "training_history.json",
        )

        save_model_only(
            model=model,
            tokenizer=tokenizer,
            directory=
                output_root / "last",
            epoch=epoch,
            args=args,
        )

        current_recall = float(
            metrics[
                f"recall@{SELECTION_K}"
            ]
        )

        current_mrr = float(
            metrics["mrr"]
        )

        recall_improved = (
            current_recall
            > best_recall
            + MIN_DELTA
        )

        recall_tied = (
            abs(
                current_recall
                - best_recall
            )
            <= MIN_DELTA
        )

        mrr_improved = (
            current_mrr
            > best_mrr
            + MIN_DELTA
        )

        improved = (
            recall_improved
            or (
                recall_tied
                and mrr_improved
            )
        )

        if improved:
            best_recall = (
                current_recall
            )
            best_mrr = (
                current_mrr
            )
            best_epoch = epoch
            no_improvement = 0

            print(
                "\n*** NEW BEST *** "
                f"epoch={epoch} | "
                f"Recall@{SELECTION_K}="
                f"{best_recall:.6f} | "
                f"MRR={best_mrr:.6f}"
            )

            save_retrieval_bundle(
                model=model,
                tokenizer=tokenizer,
                directory=
                    output_root
                    / "best",
                epoch=epoch,
                metrics=metrics,
                predictions=
                    predictions,
                catalog_records=
                    catalog_records,
                catalog_embeddings=
                    catalog_emb,
                args=args,
                baseline_metrics=
                    zero_metrics,
            )

        else:
            no_improvement += 1

            print(
                f"\nNo improvement: "
                f"{no_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        print(
            f"\nEpoch {epoch} done | "
            f"train_loss="
            f"{train_loss:.6f} | "
            f"best_epoch="
            f"{best_epoch} | "
            f"best_Recall@{SELECTION_K}="
            f"{best_recall:.6f} | "
            f"best_MRR="
            f"{best_mrr:.6f}\n"
        )

        del catalog_emb
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (
            no_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "Early stopping triggered."
            )
            break

    print("=" * 78)
    print(
        "PHASE 3 TRAINING COMPLETE"
    )
    print("=" * 78)

    print(
        f"Best epoch          : "
        f"{best_epoch}"
    )

    if best_epoch == 0:
        print(
            "Best model source   : "
            "ZERO-SHOT multilingual E5 "
            "(fine-tuning did not improve the gate)"
        )
    else:
        print(
            "Best model source   : "
            "fine-tuned multilingual E5"
        )

    print(
        f"Best Recall@{SELECTION_K:<2}     : "
        f"{best_recall:.6f}"
    )

    print(
        f"Best MRR            : "
        f"{best_mrr:.6f}"
    )

    print(
        f"Best checkpoint     : "
        f"{output_root / 'best'}"
    )

    print(
        "\nUse these together at inference:"
    )
    print(
        f"  model/tokenizer : "
        f"{output_root / 'best'}"
    )
    print(
        f"  embeddings      : "
        f"{output_root / 'best' / 'icd_embeddings.npy'}"
    )
    print(
        f"  metadata        : "
        f"{output_root / 'best' / 'icd_metadata.jsonl'}"
    )
    print(
        f"  config          : "
        f"{output_root / 'best' / 'retrieval_config.json'}"
    )


if __name__ == "__main__":
    main()

[INFO] Ignoring unknown Jupyter/Colab args: ['-f', '/root/.local/share/jupyter/runtime/kernel-d05d07bd-8c74-49a0-903b-0a89a05fe25b.json']
PHASE 3 - ICD-10 CROSS-LINGUAL RETRIEVER
train       : /content/drive/MyDrive/data/icd_link_train.jsonl
valid       : /content/drive/MyDrive/data/icd_link_valid_concept.jsonl
ICD KB      : /content/drive/MyDrive/data/icd10.csv
model       : intfloat/multilingual-e5-base
output      : /content/drive/MyDrive/output_phase3_icd_retriever
device      : cuda
GPU         : Tesla T4
[ICD] catalog codes=11,243

Loading multilingual E5...


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

[FAISS] not saved (optional). Reason: No module named 'faiss'

Epoch 1 done | train_loss=0.948706 | best_epoch=1 | best_Recall@20=0.700000 | best_MRR=0.243684

Epoch 02 | step 0050/0242 | loss=0.27739 | hardneg_acc=0.9200 | lr=8.436e-06
Epoch 02 | step 0100/0242 | loss=0.23145 | hardneg_acc=0.9350 | lr=7.964e-06
Epoch 02 | step 0150/0242 | loss=0.19922 | hardneg_acc=0.9467 | lr=7.527e-06
Epoch 02 | step 0200/0242 | loss=0.18055 | hardneg_acc=0.9475 | lr=7.055e-06
Epoch 02 | step 0242/0242 | loss=0.17301 | hardneg_acc=0.9473 | lr=6.655e-06

Encoding full ICD catalog for epoch 2...
[ICD ENCODE] 64/11,243
[ICD ENCODE] 3,264/11,243
[ICD ENCODE] 6,464/11,243
[ICD ENCODE] 9,664/11,243

[VALID_CONCEPT_EPOCH_2]
  Recall@1  = 0.200000
  Recall@5  = 0.400000
  Recall@10 = 0.533333
  Recall@20 = 0.600000
  MRR       = 0.311880
  Mean rank = 62.767
  Median    = 8.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


No improvement: 1/2

Epoch 2 done | train_loss=0.173012 | best_epoch=1 | best_Recall@20=0.700000 | best_MRR=0.243684

Epoch 03 | step 0050/0242 | loss=0.06634 | hardneg_acc=0.9850 | lr=6.218e-06
Epoch 03 | step 0100/0242 | loss=0.06704 | hardneg_acc=0.9825 | lr=5.745e-06
Epoch 03 | step 0150/0242 | loss=0.06011 | hardneg_acc=0.9867 | lr=5.309e-06
Epoch 03 | step 0200/0242 | loss=0.06275 | hardneg_acc=0.9812 | lr=4.836e-06
Epoch 03 | step 0242/0242 | loss=0.06520 | hardneg_acc=0.9793 | lr=4.436e-06

Encoding full ICD catalog for epoch 3...
[ICD ENCODE] 64/11,243
[ICD ENCODE] 3,264/11,243
[ICD ENCODE] 6,464/11,243
[ICD ENCODE] 9,664/11,243

[VALID_CONCEPT_EPOCH_3]
  Recall@1  = 0.133333
  Recall@5  = 0.333333
  Recall@10 = 0.433333
  Recall@20 = 0.533333
  MRR       = 0.230561
  Mean rank = 66.000
  Median    = 15.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


No improvement: 2/2

Epoch 3 done | train_loss=0.065203 | best_epoch=1 | best_Recall@20=0.700000 | best_MRR=0.243684

Early stopping triggered.
PHASE 3 TRAINING COMPLETE
Best epoch          : 1
Best model source   : fine-tuned multilingual E5
Best Recall@20     : 0.700000
Best MRR            : 0.243684
Best checkpoint     : /content/drive/MyDrive/output_phase3_icd_retriever/best

Use these together at inference:
  model/tokenizer : /content/drive/MyDrive/output_phase3_icd_retriever/best
  embeddings      : /content/drive/MyDrive/output_phase3_icd_retriever/best/icd_embeddings.npy
  metadata        : /content/drive/MyDrive/output_phase3_icd_retriever/best/icd_metadata.jsonl
  config          : /content/drive/MyDrive/output_phase3_icd_retriever/best/retrieval_config.json


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
PHASE 4 — RxNorm Drug Linker (Hybrid Lexical + Dense)
=====================================================

Goal
----
Map an extracted THUỐC mention to RxNorm RXCUI candidates.

Example:
    "amlodipine 10 mg po daily"
        -> RXCUI 308135

System
------
A) Lexical retrieval:
       normalized exact match
       + character TF-IDF retrieval over RxNorm terms

B) Dense retrieval:
       multilingual-e5-base
       query: drug mention
       passage: canonical RxNorm concept text

C) Hybrid:
       union lexical Top-N + dense Top-N
       weighted score:
           alpha * dense_score + (1-alpha) * lexical_score
       alpha is tuned on VALID_CONCEPT.

D) Organizer overrides:
       organizer_gold_linking_seeds.jsonl is loaded separately and saved as
       organizer_overrides.json. It is NOT used to inflate validation metrics.

Training data
-------------
rxnorm_link_train.jsonl:
{
  "id": "rx-000001",
  "query": "esomeprazole ... po q6h:prn",
  "positive_rxcui": "1810789",
  "positive": {
      "rxcui": "1810789",
      "text": "esomeprazole strontium 24.65 MG Delayed Release Oral Capsule",
      "tty": "SCD"
  },
  "hard_negatives": [
      {"rxcui": "...", "text": "...", "tty": "..."}
  ]
}

Validation:
    rxnorm_link_valid_concept.jsonl

Knowledge base:
    rxnorm.csv
Columns:
    normalized_name, rxcui, display_name, tty, sab

Model
-----
Default:
    intfloat/multilingual-e5-base

E5 retrieval format:
    query: ...
    passage: ...

Loss
----
Each training sample:
    1 positive + N explicit hard negatives

Loss =
    0.70 * explicit-hard-negative CE
  + 0.30 * masked in-batch CE

Same-RXCUI examples in the batch are masked from in-batch negatives to avoid
false negatives.

Scientific safeguards
---------------------
- Deduplicate repeated (query, positive_rxcui) pairs.
- Full-catalog validation against all RxNorm concepts.
- Benchmark lexical and zero-shot dense before fine-tuning.
- Epoch 0 can remain BEST if fine-tuning hurts held-out RXCUI retrieval.
- Hybrid alpha is tuned independently each epoch on valid_concept.
- Checkpoint selection prioritizes hybrid Recall@20, then hybrid MRR.
- Organizer example mappings are preserved as overrides but excluded from model
  selection metrics.

Outputs
-------
OUTPUT_DIR/
    run_config.json
    lexical_metrics.json
    lexical_valid_top100.jsonl
    zero_shot_dense_metrics.json
    training_history.json
    organizer_overrides.json
    best/
        model/tokenizer
        rxnorm_concept_embeddings.npy
        rxnorm_concept_metadata.jsonl
        hybrid_config.json
        validation_top20.jsonl
        training_meta.json
        rxnorm.faiss       # optional
    last/
        model/tokenizer

Colab/Jupyter-safe via parse_known_args().
"""

import os
import gc
import re
import json
import math
import time
import random
import argparse
import unicodedata
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)


# ============================================================
# 0. PATHS — EDIT THIS BLOCK
# ============================================================

DATA_ROOT = "/content/drive/MyDrive/data"

TRAIN_JSONL = (
    f"{DATA_ROOT}/rxnorm_link_train.jsonl"
)

VALID_JSONL = (
    f"{DATA_ROOT}/rxnorm_link_valid_concept.jsonl"
)

RXNORM_CSV = (
    f"{DATA_ROOT}/rxnorm.csv"
)

ORGANIZER_SEEDS_JSONL = (
    f"{DATA_ROOT}/organizer_gold_linking_seeds.jsonl"
)

# Internet ON
MODEL_NAME_OR_PATH = (
    "intfloat/multilingual-e5-base"
)
LOCAL_FILES_ONLY = False

# Internet OFF example:
# MODEL_NAME_OR_PATH = "/content/drive/MyDrive/models/multilingual-e5-base"
# LOCAL_FILES_ONLY = True

OUTPUT_DIR = (
    "/content/drive/MyDrive/output_phase4_rxnorm"
)


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42

QUERY_MAX_LENGTH = 96
DOC_MAX_LENGTH = 128

TRAIN_BATCH_SIZE = 64
EVAL_QUERY_BATCH_SIZE = 64
ENCODE_DOC_BATCH_SIZE = 96

GRAD_ACCUM_STEPS = 2

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

HARD_NEGATIVES_PER_QUERY = 4
TEMPERATURE = 0.05

EXPLICIT_LOSS_WEIGHT = 0.70
INBATCH_LOSS_WEIGHT = 0.30

# Preserve some pretrained multilingual knowledge.
FREEZE_EMBEDDINGS = False
FREEZE_BOTTOM_N_LAYERS = 3

USE_GRADIENT_CHECKPOINTING = True
USE_AMP = True

NUM_WORKERS = 2
PIN_MEMORY = True

EARLY_STOPPING_PATIENCE = 2
MIN_DELTA = 1e-6

RECALL_KS = [1, 5, 10, 20]
SELECTION_K = 20

DENSE_SAVE_TOP_K = 100
LEXICAL_SAVE_TOP_K = 100
FINAL_SAVE_TOP_K = 20

# Hybrid alpha:
# final_score = alpha*dense + (1-alpha)*lexical
HYBRID_ALPHA_GRID = [
    0.0, 0.1, 0.2, 0.3, 0.4,
    0.5, 0.6, 0.7, 0.8, 0.9, 1.0
]

# Character TF-IDF lexical retrieval.
TFIDF_NGRAM_RANGE = (3, 5)
TFIDF_MIN_DF = 1
TFIDF_QUERY_BATCH = 16

# Preferred RxNorm concept term selection.
TTY_PRIORITY = {
    "SCD": 0,   # Semantic Clinical Drug
    "SBD": 1,   # Semantic Branded Drug
    "PSN": 2,   # Prescribable Name
    "PIN": 3,   # Precise Ingredient
    "MIN": 4,   # Multiple Ingredients
    "IN": 5,    # Ingredient
    "BN": 6,    # Brand Name
    "GPCK": 7,
    "BPCK": 8,
    "SCDC": 9,
    "SBDC": 10,
    "SCDF": 11,
    "SBDF": 12,
}


# ============================================================
# 2. CLI / BASIC UTILS
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--train_jsonl", type=str, default=TRAIN_JSONL)
    p.add_argument("--valid_jsonl", type=str, default=VALID_JSONL)
    p.add_argument("--rxnorm_csv", type=str, default=RXNORM_CSV)
    p.add_argument(
        "--organizer_seeds_jsonl",
        type=str,
        default=ORGANIZER_SEEDS_JSONL,
    )
    p.add_argument(
        "--model_name_or_path",
        type=str,
        default=MODEL_NAME_OR_PATH,
    )
    p.add_argument("--output_dir", type=str, default=OUTPUT_DIR)

    p.add_argument(
        "--query_max_length",
        type=int,
        default=QUERY_MAX_LENGTH,
    )
    p.add_argument(
        "--doc_max_length",
        type=int,
        default=DOC_MAX_LENGTH,
    )

    p.add_argument(
        "--train_batch_size",
        type=int,
        default=TRAIN_BATCH_SIZE,
    )
    p.add_argument(
        "--eval_query_batch_size",
        type=int,
        default=EVAL_QUERY_BATCH_SIZE,
    )
    p.add_argument(
        "--encode_doc_batch_size",
        type=int,
        default=ENCODE_DOC_BATCH_SIZE,
    )
    p.add_argument(
        "--grad_accum_steps",
        type=int,
        default=GRAD_ACCUM_STEPS,
    )

    p.add_argument("--lr", type=float, default=LEARNING_RATE)
    p.add_argument(
        "--weight_decay",
        type=float,
        default=WEIGHT_DECAY,
    )
    p.add_argument("--epochs", type=int, default=NUM_EPOCHS)
    p.add_argument(
        "--warmup_ratio",
        type=float,
        default=WARMUP_RATIO,
    )

    p.add_argument(
        "--hard_negatives_per_query",
        type=int,
        default=HARD_NEGATIVES_PER_QUERY,
    )
    p.add_argument(
        "--temperature",
        type=float,
        default=TEMPERATURE,
    )
    p.add_argument(
        "--freeze_bottom_n_layers",
        type=int,
        default=FREEZE_BOTTOM_N_LAYERS,
    )

    p.add_argument("--seed", type=int, default=SEED)

    p.add_argument(
        "--local_files_only",
        action="store_true",
        default=LOCAL_FILES_ONLY,
    )

    args, unknown = p.parse_known_args()

    if unknown:
        print(
            "[INFO] Ignoring unknown Jupyter/Colab args:",
            unknown,
        )

    return args


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


def clean_text(x: Any) -> str:
    if x is None:
        return ""

    s = str(x).strip()

    if s.lower() == "nan":
        return ""

    return " ".join(s.split())


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing JSONL:\n  {p}\n"
            "Edit the PATHS block or use CLI overrides."
        )

    rows = []

    with p.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise ValueError(
                    f"Invalid JSON at {p}:{line_no}: {exc}"
                )

    return rows


def write_jsonl(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
        )


# ============================================================
# 3. DRUG TEXT NORMALIZATION
# ============================================================

ROUTE_FREQ_PATTERNS = [
    r"\bpo\b",
    r"\boral(?:ly)?\b",
    r"\biv\b",
    r"\bim\b",
    r"\bsc\b",
    r"\bsq\b",
    r"\bsubq\b",
    r"\bprn\b",
    r"\bdaily\b",
    r"\bonce daily\b",
    r"\bbid\b",
    r"\btid\b",
    r"\bqid\b",
    r"\bqam\b",
    r"\bqpm\b",
    r"\bqhs\b",
    r"\bq\d+h\b",
    r"\bevery \d+ hours?\b",
    r"\bat bedtime\b",
    r"\bas needed\b",
]


def normalize_unicode(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()

    s = s.replace("μg", "mcg")
    s = s.replace("µg", "mcg")

    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_rx_text(s: str) -> str:
    """
    Conservative normalization:
    keep ingredient, dose, unit, dosage form.
    """

    s = normalize_unicode(s)

    # Standardize common units.
    s = re.sub(r"\bmilligrams?\b", "mg", s)
    s = re.sub(r"\bmilligram\b", "mg", s)
    s = re.sub(r"\bmicrograms?\b", "mcg", s)
    s = re.sub(r"\bgrams?\b", "g", s)
    s = re.sub(r"\bmilliliters?\b", "ml", s)

    # Normalize spaces around slash and decimal punctuation.
    s = re.sub(r"\s*/\s*", "/", s)
    s = re.sub(r"\s+", " ", s)

    # Keep alphanumerics + clinically useful punctuation.
    s = re.sub(
        r"[^a-z0-9\.\-/\s\[\]\(\)%]+",
        " ",
        s,
    )

    s = re.sub(r"\s+", " ", s)

    return s.strip()


def strip_schedule_route(s: str) -> str:
    """
    Remove prescribing schedule/route cues that generally do not define RXCUI.
    Dose and formulation are kept.
    """

    s = normalize_rx_text(s)

    for pat in ROUTE_FREQ_PATTERNS:
        s = re.sub(
            pat,
            " ",
            s,
            flags=re.IGNORECASE,
        )

    s = re.sub(
        r"[:;,]+$",
        " ",
        s,
    )

    s = re.sub(r"\s+", " ", s)

    return s.strip()


def lexical_query_variants(s: str) -> List[str]:
    variants = [
        normalize_rx_text(s),
        strip_schedule_route(s),
    ]

    out = []
    seen = set()

    for x in variants:
        x = x.strip()

        if x and x not in seen:
            out.append(x)
            seen.add(x)

    return out


# ============================================================
# 4. RXNORM KB
# ============================================================

def load_rxnorm_terms(path: str):
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"\nMissing RxNorm CSV:\n  {p}"
        )

    df = pd.read_csv(
        p,
        dtype=str,
    ).fillna("")

    required = {
        "normalized_name",
        "rxcui",
        "display_name",
        "tty",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            "rxnorm.csv missing columns: "
            + str(sorted(missing))
        )

    df["rxcui"] = (
        df["rxcui"]
        .astype(str)
        .str.strip()
    )

    df["display_name"] = (
        df["display_name"]
        .astype(str)
        .str.strip()
    )

    df["tty"] = (
        df["tty"]
        .astype(str)
        .str.strip()
    )

    # Preserve provided normalized_name but also compute our query-side style.
    df["lexical_text"] = [
        normalize_rx_text(
            x if clean_text(x) else y
        )
        for x, y in zip(
            df["normalized_name"],
            df["display_name"],
        )
    ]

    df = (
        df[
            (df["rxcui"] != "")
            & (df["display_name"] != "")
            & (df["lexical_text"] != "")
        ]
        .drop_duplicates(
            subset=[
                "rxcui",
                "lexical_text",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

    print(
        f"[RXNORM] unique term rows="
        f"{len(df):,}"
    )

    return df


def tty_priority(tty: str) -> int:
    return TTY_PRIORITY.get(
        clean_text(tty),
        999,
    )


def build_concept_catalog(
    term_df: pd.DataFrame,
):
    """
    One row per RXCUI for dense retrieval.

    Dense document:
        preferred_name || aliases: alias1 ; alias2 ; alias3
    """

    grouped = defaultdict(list)

    for row in term_df.to_dict(
        orient="records"
    ):
        grouped[
            row["rxcui"]
        ].append(row)

    concepts = []

    for rxcui, rows in grouped.items():
        rows = sorted(
            rows,
            key=lambda x: (
                tty_priority(
                    x.get("tty", "")
                ),
                -len(
                    clean_text(
                        x.get(
                            "display_name",
                            "",
                        )
                    )
                ),
            ),
        )

        preferred = rows[0]

        aliases = []
        seen = {
            normalize_rx_text(
                preferred[
                    "display_name"
                ]
            )
        }

        for row in rows[1:]:
            name = clean_text(
                row["display_name"]
            )

            norm = normalize_rx_text(name)

            if (
                norm
                and norm not in seen
            ):
                aliases.append(name)
                seen.add(norm)

            if len(aliases) >= 3:
                break

        body = preferred[
            "display_name"
        ]

        if aliases:
            body += (
                " || aliases: "
                + " ; ".join(aliases)
            )

        concepts.append(
            {
                "rxcui": str(rxcui),
                "display_name":
                    preferred[
                        "display_name"
                    ],
                "tty":
                    preferred[
                        "tty"
                    ],
                "aliases":
                    aliases,
                "document":
                    "passage: " + body,
            }
        )

    concepts.sort(
        key=lambda x: (
            int(x["rxcui"])
            if x["rxcui"].isdigit()
            else x["rxcui"]
        )
    )

    rxcui_to_index = {
        x["rxcui"]: i
        for i, x
        in enumerate(concepts)
    }

    print(
        f"[RXNORM] unique concepts/RXCUI="
        f"{len(concepts):,}"
    )

    return concepts, rxcui_to_index


def build_term_to_concept_index(
    term_df,
    rxcui_to_index,
):
    term_to_concept = np.array(
        [
            rxcui_to_index[
                str(x)
            ]
            for x in term_df["rxcui"]
        ],
        dtype=np.int32,
    )

    return term_to_concept


# ============================================================
# 5. ORGANIZER OVERRIDES
# ============================================================

def load_organizer_overrides(
    path: str,
):
    p = Path(path)

    if not p.exists():
        print(
            "[OVERRIDES] organizer seed file not found; "
            "continuing without overrides."
        )
        return {}

    rows = read_jsonl(path)

    out = {}

    for row in rows:
        if row.get("type") != "THUỐC":
            continue

        mention = clean_text(
            row.get(
                "mention",
                "",
            )
        )

        candidates = [
            str(x)
            for x
            in row.get(
                "candidates",
                [],
            )
        ]

        if not mention or not candidates:
            continue

        for variant in lexical_query_variants(
            mention
        ):
            out[variant] = {
                "mention": mention,
                "candidates": candidates,
                "source": row.get(
                    "source",
                    "organizer_example",
                ),
            }

    print(
        f"[OVERRIDES] drug mention variants="
        f"{len(out):,}"
    )

    return out


# ============================================================
# 6. TRAIN / VALID DATA
# ============================================================

class RxNormTrainDataset(Dataset):
    def __init__(
        self,
        path: str,
        rxcui_to_index:
            Dict[str, int],
        hard_negatives_per_query: int,
        seed: int,
    ):
        self.path = path
        self.rxcui_to_index = (
            rxcui_to_index
        )
        self.hard_n = int(
            hard_negatives_per_query
        )
        self.seed = int(seed)

        raw = read_jsonl(path)

        # Deduplicate (query, positive_rxcui) and merge hard-negative pools.
        merged = {}

        for row in raw:
            query = clean_text(
                row.get("query", "")
            )

            pos = str(
                row.get(
                    "positive_rxcui",
                    "",
                )
            ).strip()

            if (
                not query
                or pos
                not in rxcui_to_index
            ):
                continue

            key = (
                query,
                pos,
            )

            if key not in merged:
                merged[key] = {
                    "query": query,
                    "positive_rxcui": pos,
                    "negative_pool": set(),
                }

            for neg in row.get(
                "hard_negatives",
                [],
            ):
                n = str(
                    neg.get(
                        "rxcui",
                        "",
                    )
                ).strip()

                if (
                    n
                    and n != pos
                    and n
                    in rxcui_to_index
                ):
                    merged[key][
                        "negative_pool"
                    ].add(n)

        self.samples = []

        all_rxcuis = list(
            rxcui_to_index.keys()
        )

        rng = random.Random(
            self.seed
        )

        for item in merged.values():
            pool = sorted(
                item[
                    "negative_pool"
                ]
            )

            if len(pool) < self.hard_n:
                fallback = [
                    x
                    for x in all_rxcuis
                    if (
                        x
                        != item[
                            "positive_rxcui"
                        ]
                        and x not in pool
                    )
                ]

                rng.shuffle(fallback)

                pool.extend(
                    fallback[
                        : self.hard_n
                        - len(pool)
                    ]
                )

            self.samples.append(
                {
                    "query":
                        item["query"],
                    "positive_rxcui":
                        item[
                            "positive_rxcui"
                        ],
                    "negative_pool":
                        pool,
                }
            )

        self.samples.sort(
            key=lambda x: (
                x["query"],
                x["positive_rxcui"],
            )
        )

        rng.shuffle(self.samples)

        unique_cui = {
            x["positive_rxcui"]
            for x in self.samples
        }

        unique_q = {
            x["query"]
            for x in self.samples
        }

        print(
            "[TRAIN] "
            f"raw_rows={len(raw):,} | "
            f"unique_queries={len(unique_q):,} | "
            f"unique_pairs={len(self.samples):,} | "
            f"positive_RXCUI={len(unique_cui):,}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        pool = item["negative_pool"]

        local_rng = random.Random(
            self.seed
            + idx * 10007
            + random.randint(
                0,
                10**6,
            )
        )

        if len(pool) <= self.hard_n:
            negatives = list(pool)
        else:
            negatives = local_rng.sample(
                pool,
                self.hard_n,
            )

        return {
            "query":
                item["query"],
            "positive_rxcui":
                item[
                    "positive_rxcui"
                ],
            "negative_rxcuis":
                negatives,
        }


def load_validation(
    path: str,
    rxcui_to_index:
        Dict[str, int],
):
    raw = read_jsonl(path)

    unique = {}

    for row in raw:
        query = clean_text(
            row.get("query", "")
        )

        pos = str(
            row.get(
                "positive_rxcui",
                "",
            )
        ).strip()

        if (
            not query
            or pos
            not in rxcui_to_index
        ):
            continue

        key = (
            query,
            pos,
        )

        unique[key] = {
            "query": query,
            "gold_rxcuis": [pos],
        }

    rows = list(
        unique.values()
    )

    rows.sort(
        key=lambda x: (
            x["query"],
            tuple(
                x["gold_rxcuis"]
            ),
        )
    )

    unique_cui = {
        c
        for row in rows
        for c in row[
            "gold_rxcuis"
        ]
    }

    print(
        "[VALID] "
        f"raw_rows={len(raw):,} | "
        f"unique_pairs={len(rows):,} | "
        f"heldout_RXCUI={len(unique_cui):,}"
    )

    return rows


# ============================================================
# 7. LEXICAL RETRIEVER
# ============================================================

class LexicalRetriever:
    def __init__(
        self,
        term_df: pd.DataFrame,
        concepts,
        rxcui_to_index,
        term_to_concept:
            np.ndarray,
    ):
        self.term_df = term_df
        self.concepts = concepts
        self.rxcui_to_index = (
            rxcui_to_index
        )
        self.term_to_concept = (
            term_to_concept
        )

        # Exact map.
        exact = defaultdict(set)

        for row in term_df.to_dict(
            orient="records"
        ):
            rxcui = str(
                row["rxcui"]
            )

            for key in {
                normalize_rx_text(
                    row[
                        "display_name"
                    ]
                ),
                normalize_rx_text(
                    row[
                        "lexical_text"
                    ]
                ),
                strip_schedule_route(
                    row[
                        "display_name"
                    ]
                ),
            }:
                if key:
                    exact[key].add(
                        rxcui
                    )

        self.exact_map = {
            k: sorted(v)
            for k, v in exact.items()
        }

        print(
            f"[LEXICAL] exact keys="
            f"{len(self.exact_map):,}"
        )

        try:
            from sklearn.feature_extraction.text import (
                TfidfVectorizer,
            )
        except Exception as exc:
            raise RuntimeError(
                "scikit-learn is required for Phase 4 lexical retrieval.\n"
                "On Colab/Kaggle it is normally preinstalled."
            ) from exc

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=
                TFIDF_NGRAM_RANGE,
            min_df=TFIDF_MIN_DF,
            lowercase=False,
            dtype=np.float32,
            norm="l2",
        )

        corpus = (
            term_df[
                "lexical_text"
            ]
            .astype(str)
            .tolist()
        )

        print(
            "[LEXICAL] fitting char-TFIDF "
            f"on {len(corpus):,} RxNorm terms..."
        )

        self.term_matrix = (
            self.vectorizer
            .fit_transform(corpus)
            .tocsr()
        )

        print(
            "[LEXICAL] TFIDF matrix="
            f"{self.term_matrix.shape} | "
            f"nnz={self.term_matrix.nnz:,}"
        )

    def retrieve_one(
        self,
        query: str,
        top_k: int,
    ):
        """
        Return concept-level lexical candidates:
            [{rxcui, score, display_name, exact}, ...]
        """

        variants = (
            lexical_query_variants(
                query
            )
        )

        exact_rxcuis = set()

        for v in variants:
            exact_rxcuis.update(
                self.exact_map.get(
                    v,
                    [],
                )
            )

        # Use schedule-stripped query for TF-IDF.
        q_text = (
            variants[-1]
            if variants
            else normalize_rx_text(query)
        )

        q_vec = (
            self.vectorizer
            .transform([q_text])
        )

        term_scores = (
            q_vec
            @ self.term_matrix.T
        ).toarray()[0]

        # Get more term candidates because multiple terms map to same RXCUI.
        term_top_n = min(
            max(
                top_k * 8,
                100,
            ),
            len(term_scores),
        )

        if term_top_n < len(
            term_scores
        ):
            idx = np.argpartition(
                -term_scores,
                term_top_n - 1,
            )[:term_top_n]

            idx = idx[
                np.argsort(
                    -term_scores[idx]
                )
            ]
        else:
            idx = np.argsort(
                -term_scores
            )

        concept_score = {}

        for term_idx in idx:
            cidx = int(
                self.term_to_concept[
                    int(term_idx)
                ]
            )

            score = float(
                term_scores[
                    int(term_idx)
                ]
            )

            if (
                cidx not in concept_score
                or score
                > concept_score[cidx]
            ):
                concept_score[cidx] = (
                    score
                )

        # Exact matches receive 1.05 to force them above ordinary cosine TF-IDF.
        for rxcui in exact_rxcuis:
            cidx = (
                self.rxcui_to_index[
                    rxcui
                ]
            )

            concept_score[cidx] = max(
                concept_score.get(
                    cidx,
                    0.0,
                ),
                1.05,
            )

        ranked = sorted(
            concept_score.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]

        out = []

        for cidx, score in ranked:
            meta = self.concepts[
                cidx
            ]

            out.append(
                {
                    "rxcui":
                        meta["rxcui"],
                    "display_name":
                        meta[
                            "display_name"
                        ],
                    "tty":
                        meta["tty"],
                    "score":
                        float(score),
                    "exact":
                        bool(
                            meta["rxcui"]
                            in exact_rxcuis
                        ),
                }
            )

        return out

    def retrieve_many(
        self,
        rows,
        top_k: int,
    ):
        outputs = []

        for i, row in enumerate(rows):
            candidates = (
                self.retrieve_one(
                    row["query"],
                    top_k,
                )
            )

            outputs.append(
                {
                    "query":
                        row["query"],
                    "gold_rxcuis":
                        row.get(
                            "gold_rxcuis",
                            [],
                        ),
                    "top_candidates":
                        candidates,
                }
            )

            if (
                (i + 1) % 250 == 0
                or i + 1 == len(rows)
            ):
                print(
                    f"[LEXICAL] "
                    f"{i+1:,}/{len(rows):,}"
                )

        return outputs


# ============================================================
# 8. RETRIEVAL METRICS
# ============================================================

def metrics_from_ranked_rows(
    ranked_rows,
):
    hit = {
        k: 0
        for k in RECALL_KS
    }

    ranks = []

    for row in ranked_rows:
        gold = set(
            str(x)
            for x in row[
                "gold_rxcuis"
            ]
        )

        rank = None

        for i, cand in enumerate(
            row[
                "top_candidates"
            ],
            1,
        ):
            if str(
                cand["rxcui"]
            ) in gold:
                rank = i
                break

        if rank is None:
            rank = 10**9

        ranks.append(rank)

        for k in RECALL_KS:
            if rank <= k:
                hit[k] += 1

    n = len(ranked_rows)

    metrics = {
        f"recall@{k}":
            (
                hit[k] / n
                if n
                else 0.0
            )
        for k in RECALL_KS
    }

    valid_ranks = [
        r
        for r in ranks
        if r < 10**9
    ]

    metrics["mrr"] = (
        float(
            np.mean(
                [
                    1.0 / r
                    for r in ranks
                    if r < 10**9
                ]
                + [
                    0.0
                    for r in ranks
                    if r >= 10**9
                ]
            )
        )
        if ranks
        else 0.0
    )

    metrics[
        "mean_rank_retrieved"
    ] = (
        float(
            np.mean(
                valid_ranks
            )
        )
        if valid_ranks
        else math.nan
    )

    metrics[
        "n_queries"
    ] = n

    return metrics


def print_metrics(
    name,
    metrics,
):
    print(
        f"\n[{name}]"
    )

    for k in RECALL_KS:
        print(
            f"  Recall@{k:<2} = "
            f"{metrics[f'recall@{k}']:.6f}"
        )

    print(
        f"  MRR       = "
        f"{metrics['mrr']:.6f}"
    )


# ============================================================
# 9. TRAIN COLLATOR
# ============================================================

class RxNormTrainCollator:
    def __init__(
        self,
        tokenizer,
        concepts,
        rxcui_to_index,
        query_max_length,
        doc_max_length,
    ):
        self.tokenizer = tokenizer
        self.concepts = concepts
        self.rxcui_to_index = (
            rxcui_to_index
        )
        self.query_max_length = int(
            query_max_length
        )
        self.doc_max_length = int(
            doc_max_length
        )

    def doc_text(self, rxcui):
        return self.concepts[
            self.rxcui_to_index[
                str(rxcui)
            ]
        ]["document"]

    def __call__(self, features):
        queries = [
            "query: "
            + x["query"]
            for x in features
        ]

        candidate_rxcuis = []

        for x in features:
            candidate_rxcuis.append(
                [
                    x[
                        "positive_rxcui"
                    ]
                ]
                + list(
                    x[
                        "negative_rxcuis"
                    ]
                )
            )

        candidate_count = len(
            candidate_rxcuis[0]
        )

        if not all(
            len(x)
            == candidate_count
            for x in candidate_rxcuis
        ):
            raise RuntimeError(
                "Candidate count differs inside batch."
            )

        docs = []

        for codes in candidate_rxcuis:
            for code in codes:
                docs.append(
                    self.doc_text(code)
                )

        q_batch = self.tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=
                self.query_max_length,
            return_tensors="pt",
        )

        d_batch = self.tokenizer(
            docs,
            padding=True,
            truncation=True,
            max_length=
                self.doc_max_length,
            return_tensors="pt",
        )

        return {
            "query_batch":
                q_batch,
            "doc_batch":
                d_batch,
            "candidate_count":
                candidate_count,
            "positive_rxcuis": [
                x[
                    "positive_rxcui"
                ]
                for x in features
            ],
        }


# ============================================================
# 10. E5 ENCODING
# ============================================================

def average_pool(
    hidden,
    attention_mask,
):
    mask = (
        attention_mask[
            ...,
            None,
        ]
        .bool()
    )

    hidden = hidden.masked_fill(
        ~mask,
        0.0,
    )

    denom = (
        attention_mask
        .sum(dim=1)
        [..., None]
        .clamp(min=1)
    )

    return (
        hidden.sum(dim=1)
        / denom
    )


def encode_batch(
    model,
    token_batch,
):
    out = model(
        **token_batch
    )

    emb = average_pool(
        out.last_hidden_state,
        token_batch[
            "attention_mask"
        ],
    )

    return F.normalize(
        emb,
        p=2,
        dim=1,
    )


def move_batch(
    batch,
    device,
):
    return {
        k: v.to(
            device,
            non_blocking=True,
        )
        for k, v
        in batch.items()
    }


def freeze_model(
    model,
    bottom_n: int,
):
    if (
        FREEZE_EMBEDDINGS
        and hasattr(
            model,
            "embeddings",
        )
    ):
        for p in (
            model.embeddings
            .parameters()
        ):
            p.requires_grad = False

        print(
            "[FREEZE] embeddings frozen"
        )

    layers = None

    if (
        hasattr(
            model,
            "encoder",
        )
        and hasattr(
            model.encoder,
            "layer",
        )
    ):
        layers = (
            model.encoder.layer
        )

    if layers is None:
        print(
            "[WARN] model.encoder.layer not found; "
            "no bottom transformer layers frozen."
        )
        return

    n = min(
        max(
            int(bottom_n),
            0,
        ),
        len(layers),
    )

    for layer in layers[:n]:
        for p in (
            layer.parameters()
        ):
            p.requires_grad = False

    print(
        f"[FREEZE] bottom layers "
        f"{n}/{len(layers)} frozen"
    )


def print_parameter_count(
    model,
):
    total = sum(
        p.numel()
        for p
        in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p
        in model.parameters()
        if p.requires_grad
    )

    print(
        f"[PARAMS] total={total:,} | "
        f"trainable={trainable:,} | "
        f"{100*trainable/total:.2f}%"
    )


# ============================================================
# 11. TRAIN LOSS
# ============================================================

def retrieval_loss(
    query_emb,
    doc_emb,
    candidate_count: int,
    positive_rxcuis:
        List[str],
    temperature: float,
):
    batch_size = (
        query_emb.shape[0]
    )

    docs = doc_emb.view(
        batch_size,
        candidate_count,
        -1,
    )

    # ----------------------------
    # Explicit hard-negative loss
    # ----------------------------

    explicit_scores = torch.einsum(
        "bd,bcd->bc",
        query_emb,
        docs,
    ) / float(temperature)

    explicit_targets = torch.zeros(
        batch_size,
        dtype=torch.long,
        device=query_emb.device,
    )

    explicit_loss = F.cross_entropy(
        explicit_scores,
        explicit_targets,
    )

    explicit_acc = (
        explicit_scores.argmax(
            dim=1
        )
        == explicit_targets
    ).float().mean()

    # ----------------------------
    # Masked in-batch positive docs
    # ----------------------------

    pos_docs = docs[:, 0, :]

    inbatch_scores = (
        query_emb
        @ pos_docs.T
    ) / float(temperature)

    # Mask false negatives:
    # if query i and positive document j have same RXCUI, j must not be a
    # negative for i. Keep diagonal as the target.
    for i in range(batch_size):
        for j in range(batch_size):
            if (
                i != j
                and positive_rxcuis[i]
                == positive_rxcuis[j]
            ):
                inbatch_scores[
                    i,
                    j,
                ] = -1e4

    inbatch_targets = torch.arange(
        batch_size,
        dtype=torch.long,
        device=query_emb.device,
    )

    inbatch_loss = F.cross_entropy(
        inbatch_scores,
        inbatch_targets,
    )

    inbatch_acc = (
        inbatch_scores.argmax(
            dim=1
        )
        == inbatch_targets
    ).float().mean()

    total_loss = (
        EXPLICIT_LOSS_WEIGHT
        * explicit_loss
        + INBATCH_LOSS_WEIGHT
        * inbatch_loss
    )

    return {
        "loss": total_loss,
        "explicit_loss":
            explicit_loss,
        "inbatch_loss":
            inbatch_loss,
        "explicit_acc":
            explicit_acc,
        "inbatch_acc":
            inbatch_acc,
    }


# ============================================================
# 12. CONCEPT CATALOG EMBEDDINGS
# ============================================================

@torch.no_grad()
def encode_concept_catalog(
    model,
    tokenizer,
    concepts,
    device,
    batch_size,
    doc_max_length,
):
    model.eval()

    amp_enabled = bool(
        USE_AMP
        and device.type == "cuda"
    )

    all_emb = []

    docs = [
        x["document"]
        for x in concepts
    ]

    for start in range(
        0,
        len(docs),
        batch_size,
    ):
        batch_text = docs[
            start:
            start + batch_size
        ]

        tokens = tokenizer(
            batch_text,
            padding=True,
            truncation=True,
            max_length=
                doc_max_length,
            return_tensors="pt",
        )

        tokens = move_batch(
            tokens,
            device,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            emb = encode_batch(
                model,
                tokens,
            )

        all_emb.append(
            emb.float()
            .cpu()
            .numpy()
        )

        if (
            (start // batch_size)
            % 50 == 0
        ):
            print(
                "[DENSE KB] "
                f"{min(start+batch_size, len(docs)):,}/"
                f"{len(docs):,}"
            )

    arr = np.concatenate(
        all_emb,
        axis=0,
    ).astype(np.float32)

    # Numerical re-normalization.
    arr /= np.clip(
        np.linalg.norm(
            arr,
            axis=1,
            keepdims=True,
        ),
        1e-12,
        None,
    )

    return arr


# ============================================================
# 13. DENSE RETRIEVAL
# ============================================================

@torch.no_grad()
def dense_retrieve(
    model,
    tokenizer,
    valid_rows,
    concepts,
    concept_embeddings,
    device,
    query_batch_size,
    query_max_length,
    top_k,
):
    model.eval()

    amp_enabled = bool(
        USE_AMP
        and device.type == "cuda"
    )

    catalog = (
        torch.from_numpy(
            concept_embeddings
        )
        .to(
            device=device,
            dtype=torch.float32,
        )
    )

    outputs = []

    for start in range(
        0,
        len(valid_rows),
        query_batch_size,
    ):
        rows = valid_rows[
            start:
            start + query_batch_size
        ]

        queries = [
            "query: "
            + row["query"]
            for row in rows
        ]

        tokens = tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=
                query_max_length,
            return_tensors="pt",
        )

        tokens = move_batch(
            tokens,
            device,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            q_emb = encode_batch(
                model,
                tokens,
            )

        scores = (
            q_emb.float()
            @ catalog.T
        )

        k = min(
            int(top_k),
            scores.shape[1],
        )

        vals, idx = torch.topk(
            scores,
            k=k,
            dim=1,
            largest=True,
            sorted=True,
        )

        vals = (
            vals
            .detach()
            .cpu()
            .numpy()
        )

        idx = (
            idx
            .detach()
            .cpu()
            .numpy()
        )

        for i, row in enumerate(rows):
            candidates = []

            for score, cidx in zip(
                vals[i],
                idx[i],
            ):
                meta = concepts[
                    int(cidx)
                ]

                candidates.append(
                    {
                        "rxcui":
                            meta[
                                "rxcui"
                            ],
                        "display_name":
                            meta[
                                "display_name"
                            ],
                        "tty":
                            meta["tty"],
                        "score":
                            float(score),
                    }
                )

            outputs.append(
                {
                    "query":
                        row["query"],
                    "gold_rxcuis":
                        row[
                            "gold_rxcuis"
                        ],
                    "top_candidates":
                        candidates,
                }
            )

    del catalog

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return outputs


# ============================================================
# 14. HYBRID RETRIEVAL
# ============================================================

def normalize_candidate_scores(
    candidates,
):
    if not candidates:
        return {}

    vals = np.array(
        [
            float(x["score"])
            for x in candidates
        ],
        dtype=np.float32,
    )

    lo = float(vals.min())
    hi = float(vals.max())

    if hi - lo < 1e-8:
        normalized = np.ones_like(
            vals
        )
    else:
        normalized = (
            (vals - lo)
            / (hi - lo)
        )

    return {
        str(cand["rxcui"]):
            float(score)
        for cand, score
        in zip(
            candidates,
            normalized,
        )
    }


def hybrid_rank_one(
    lexical_row,
    dense_row,
    concepts,
    rxcui_to_index,
    alpha,
    top_k,
):
    lex_map = (
        normalize_candidate_scores(
            lexical_row[
                "top_candidates"
            ]
        )
    )

    dense_map = (
        normalize_candidate_scores(
            dense_row[
                "top_candidates"
            ]
        )
    )

    union = (
        set(lex_map)
        | set(dense_map)
    )

    scored = []

    for rxcui in union:
        dense_score = (
            dense_map.get(
                rxcui,
                0.0,
            )
        )

        lex_score = (
            lex_map.get(
                rxcui,
                0.0,
            )
        )

        final = (
            float(alpha)
            * dense_score
            + (
                1.0
                - float(alpha)
            )
            * lex_score
        )

        cidx = (
            rxcui_to_index[
                rxcui
            ]
        )

        meta = concepts[
            cidx
        ]

        scored.append(
            {
                "rxcui": rxcui,
                "display_name":
                    meta[
                        "display_name"
                    ],
                "tty":
                    meta["tty"],
                "score":
                    float(final),
                "dense_score":
                    float(
                        dense_score
                    ),
                "lexical_score":
                    float(
                        lex_score
                    ),
            }
        )

    scored.sort(
        key=lambda x: (
            x["score"],
            x["dense_score"],
            x["lexical_score"],
        ),
        reverse=True,
    )

    return scored[:top_k]


def build_hybrid_rows(
    lexical_rows,
    dense_rows,
    concepts,
    rxcui_to_index,
    alpha,
    top_k,
):
    if len(
        lexical_rows
    ) != len(dense_rows):
        raise RuntimeError(
            "Lexical/dense row counts differ."
        )

    out = []

    for lex, dense in zip(
        lexical_rows,
        dense_rows,
    ):
        if (
            lex["query"]
            != dense["query"]
        ):
            raise RuntimeError(
                "Lexical/dense query order mismatch."
            )

        candidates = (
            hybrid_rank_one(
                lexical_row=lex,
                dense_row=dense,
                concepts=concepts,
                rxcui_to_index=
                    rxcui_to_index,
                alpha=alpha,
                top_k=top_k,
            )
        )

        out.append(
            {
                "query":
                    dense["query"],
                "gold_rxcuis":
                    dense[
                        "gold_rxcuis"
                    ],
                "top_candidates":
                    candidates,
            }
        )

    return out


def tune_hybrid_alpha(
    lexical_rows,
    dense_rows,
    concepts,
    rxcui_to_index,
):
    best_alpha = None
    best_metrics = None
    best_rows = None

    for alpha in (
        HYBRID_ALPHA_GRID
    ):
        rows = (
            build_hybrid_rows(
                lexical_rows=
                    lexical_rows,
                dense_rows=
                    dense_rows,
                concepts=concepts,
                rxcui_to_index=
                    rxcui_to_index,
                alpha=alpha,
                top_k=
                    FINAL_SAVE_TOP_K,
            )
        )

        metrics = (
            metrics_from_ranked_rows(
                rows
            )
        )

        print(
            f"[HYBRID alpha={alpha:.1f}] "
            f"R@20="
            f"{metrics['recall@20']:.6f} | "
            f"MRR="
            f"{metrics['mrr']:.6f}"
        )

        if best_metrics is None:
            better = True
        else:
            r = metrics[
                f"recall@{SELECTION_K}"
            ]

            br = best_metrics[
                f"recall@{SELECTION_K}"
            ]

            if r > br + MIN_DELTA:
                better = True
            elif (
                abs(r - br)
                <= MIN_DELTA
                and metrics["mrr"]
                > best_metrics["mrr"]
                + MIN_DELTA
            ):
                better = True
            else:
                better = False

        if better:
            best_alpha = (
                float(alpha)
            )
            best_metrics = metrics
            best_rows = rows

    return (
        best_alpha,
        best_metrics,
        best_rows,
    )


# ============================================================
# 15. SAVE ARTIFACTS
# ============================================================

def maybe_save_faiss(
    embeddings,
    path,
):
    try:
        import faiss

        index = faiss.IndexFlatIP(
            embeddings.shape[1]
        )

        index.add(
            embeddings.astype(
                np.float32
            )
        )

        faiss.write_index(
            index,
            str(path),
        )

        print(
            f"[FAISS] saved {path}"
        )

        return True

    except Exception as exc:
        print(
            "[FAISS] optional index not saved: "
            f"{exc}"
        )

        return False


def save_best_bundle(
    model,
    tokenizer,
    directory,
    epoch,
    dense_metrics,
    hybrid_alpha,
    hybrid_metrics,
    hybrid_rows,
    concepts,
    embeddings,
    args,
    lexical_metrics,
    zero_shot_metrics,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        directory
    )

    tokenizer.save_pretrained(
        directory
    )

    np.save(
        directory
        / "rxnorm_concept_embeddings.npy",
        embeddings.astype(
            np.float32
        ),
    )

    metadata = [
        {
            "index": i,
            "rxcui":
                x["rxcui"],
            "display_name":
                x[
                    "display_name"
                ],
            "tty":
                x["tty"],
            "aliases":
                x["aliases"],
        }
        for i, x
        in enumerate(concepts)
    ]

    write_jsonl(
        metadata,
        directory
        / "rxnorm_concept_metadata.jsonl",
    )

    write_jsonl(
        hybrid_rows,
        directory
        / "validation_top20.jsonl",
    )

    save_json(
        {
            "hybrid_alpha":
                hybrid_alpha,
            "formula":
                "alpha*dense + (1-alpha)*lexical",
            "dense_top_k":
                DENSE_SAVE_TOP_K,
            "lexical_top_k":
                LEXICAL_SAVE_TOP_K,
            "final_top_k":
                FINAL_SAVE_TOP_K,
            "query_prefix":
                "query: ",
            "passage_prefix":
                "passage: ",
            "query_max_length":
                args.query_max_length,
            "doc_max_length":
                args.doc_max_length,
            "dense_similarity":
                "cosine via L2-normalized inner product",
            "lexical":
                "normalized exact + char_wb TF-IDF",
        },
        directory
        / "hybrid_config.json",
    )

    save_json(
        {
            "epoch": int(epoch),
            "selection_metric":
                f"hybrid_recall@{SELECTION_K}",
            "tie_break_metric":
                "hybrid_mrr",
            "dense_metrics":
                dense_metrics,
            "hybrid_metrics":
                hybrid_metrics,
            "hybrid_alpha":
                hybrid_alpha,
            "lexical_baseline":
                lexical_metrics,
            "zero_shot_dense":
                zero_shot_metrics,
            "training_args":
                vars(args),
        },
        directory
        / "training_meta.json",
    )

    maybe_save_faiss(
        embeddings,
        directory
        / "rxnorm.faiss",
    )


def save_model_only(
    model,
    tokenizer,
    directory,
    epoch,
    args,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        directory
    )

    tokenizer.save_pretrained(
        directory
    )

    save_json(
        {
            "epoch": int(epoch),
            "training_args":
                vars(args),
        },
        directory
        / "training_meta.json",
    )


# ============================================================
# 16. MAIN
# ============================================================

def main():
    args = parse_args()
    set_seed(args.seed)

    output_root = Path(
        args.output_dir
    )

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    save_json(
        vars(args),
        output_root
        / "run_config.json",
    )

    print("=" * 78)
    print(
        "PHASE 4 - RXNORM HYBRID DRUG LINKER"
    )
    print("=" * 78)

    print(
        f"train       : {args.train_jsonl}"
    )
    print(
        f"valid       : {args.valid_jsonl}"
    )
    print(
        f"rxnorm.csv  : {args.rxnorm_csv}"
    )
    print(
        f"model       : {args.model_name_or_path}"
    )
    print(
        f"output      : {args.output_dir}"
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"device      : {device}"
    )

    if device.type == "cuda":
        print(
            f"GPU         : "
            f"{torch.cuda.get_device_name(0)}"
        )

    # --------------------------------------------------------
    # KB
    # --------------------------------------------------------

    term_df = load_rxnorm_terms(
        args.rxnorm_csv
    )

    concepts, (
        rxcui_to_index
    ) = build_concept_catalog(
        term_df
    )

    term_to_concept = (
        build_term_to_concept_index(
            term_df,
            rxcui_to_index,
        )
    )

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    train_ds = RxNormTrainDataset(
        path=args.train_jsonl,
        rxcui_to_index=
            rxcui_to_index,
        hard_negatives_per_query=
            args.hard_negatives_per_query,
        seed=args.seed,
    )

    valid_rows = load_validation(
        path=args.valid_jsonl,
        rxcui_to_index=
            rxcui_to_index,
    )

    # --------------------------------------------------------
    # Organizer overrides
    # --------------------------------------------------------

    overrides = (
        load_organizer_overrides(
            args.organizer_seeds_jsonl
        )
    )

    save_json(
        overrides,
        output_root
        / "organizer_overrides.json",
    )

    # --------------------------------------------------------
    # Lexical baseline
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 78
    )
    print(
        "LEXICAL BASELINE"
    )
    print(
        "=" * 78
    )

    lexical = LexicalRetriever(
        term_df=term_df,
        concepts=concepts,
        rxcui_to_index=
            rxcui_to_index,
        term_to_concept=
            term_to_concept,
    )

    lexical_valid_rows = (
        lexical.retrieve_many(
            valid_rows,
            top_k=
                LEXICAL_SAVE_TOP_K,
        )
    )

    lexical_metrics = (
        metrics_from_ranked_rows(
            lexical_valid_rows
        )
    )

    print_metrics(
        "LEXICAL_VALID",
        lexical_metrics,
    )

    save_json(
        lexical_metrics,
        output_root
        / "lexical_metrics.json",
    )

    write_jsonl(
        lexical_valid_rows,
        output_root
        / "lexical_valid_top100.jsonl",
    )

    # --------------------------------------------------------
    # Model/tokenizer
    # --------------------------------------------------------

    print(
        "\nLoading multilingual E5..."
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            args.model_name_or_path,
            local_files_only=
                args.local_files_only,
        )
    )

    model = (
        AutoModel
        .from_pretrained(
            args.model_name_or_path,
            local_files_only=
                args.local_files_only,
        )
    )

    tokenizer_cap = int(
        getattr(
            tokenizer,
            "model_max_length",
            512,
        )
    )

    if tokenizer_cap > 100000:
        tokenizer_cap = 512

    args.query_max_length = min(
        args.query_max_length,
        tokenizer_cap,
    )

    args.doc_max_length = min(
        args.doc_max_length,
        tokenizer_cap,
    )

    freeze_model(
        model,
        args.freeze_bottom_n_layers,
    )

    if (
        USE_GRADIENT_CHECKPOINTING
        and hasattr(
            model,
            "gradient_checkpointing_enable",
        )
    ):
        model.gradient_checkpointing_enable()

        print(
            "[MODEL] gradient checkpointing enabled"
        )

    model.to(device)

    print_parameter_count(
        model
    )

    # --------------------------------------------------------
    # Train loader
    # --------------------------------------------------------

    collator = RxNormTrainCollator(
        tokenizer=tokenizer,
        concepts=concepts,
        rxcui_to_index=
            rxcui_to_index,
        query_max_length=
            args.query_max_length,
        doc_max_length=
            args.doc_max_length,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            args.train_batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collator,
    )

    # --------------------------------------------------------
    # ZERO-SHOT dense + hybrid gate
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 78
    )
    print(
        "ZERO-SHOT DENSE + HYBRID EVALUATION"
    )
    print(
        "=" * 78
    )

    zero_emb = (
        encode_concept_catalog(
            model=model,
            tokenizer=tokenizer,
            concepts=concepts,
            device=device,
            batch_size=
                args.encode_doc_batch_size,
            doc_max_length=
                args.doc_max_length,
        )
    )

    zero_dense_rows = dense_retrieve(
        model=model,
        tokenizer=tokenizer,
        valid_rows=valid_rows,
        concepts=concepts,
        concept_embeddings=
            zero_emb,
        device=device,
        query_batch_size=
            args.eval_query_batch_size,
        query_max_length=
            args.query_max_length,
        top_k=
            DENSE_SAVE_TOP_K,
    )

    zero_dense_metrics = (
        metrics_from_ranked_rows(
            zero_dense_rows
        )
    )

    print_metrics(
        "ZERO_SHOT_DENSE",
        zero_dense_metrics,
    )

    zero_alpha, (
        zero_hybrid_metrics
    ), zero_hybrid_rows = (
        tune_hybrid_alpha(
            lexical_rows=
                lexical_valid_rows,
            dense_rows=
                zero_dense_rows,
            concepts=concepts,
            rxcui_to_index=
                rxcui_to_index,
        )
    )

    print(
        f"\n[ZERO_SHOT HYBRID BEST] "
        f"alpha={zero_alpha:.2f}"
    )

    print_metrics(
        "ZERO_SHOT_HYBRID",
        zero_hybrid_metrics,
    )

    save_json(
        {
            "dense":
                zero_dense_metrics,
            "hybrid_alpha":
                zero_alpha,
            "hybrid":
                zero_hybrid_metrics,
        },
        output_root
        / "zero_shot_dense_metrics.json",
    )

    # Epoch 0 may remain best.
    best_epoch = 0
    best_recall = float(
        zero_hybrid_metrics[
            f"recall@{SELECTION_K}"
        ]
    )
    best_mrr = float(
        zero_hybrid_metrics[
            "mrr"
        ]
    )
    best_alpha = float(
        zero_alpha
    )

    no_improvement = 0

    save_best_bundle(
        model=model,
        tokenizer=tokenizer,
        directory=
            output_root / "best",
        epoch=0,
        dense_metrics=
            zero_dense_metrics,
        hybrid_alpha=
            zero_alpha,
        hybrid_metrics=
            zero_hybrid_metrics,
        hybrid_rows=
            zero_hybrid_rows,
        concepts=concepts,
        embeddings=
            zero_emb,
        args=args,
        lexical_metrics=
            lexical_metrics,
        zero_shot_metrics={
            "dense":
                zero_dense_metrics,
            "hybrid_alpha":
                zero_alpha,
            "hybrid":
                zero_hybrid_metrics,
        },
    )

    del zero_emb
    gc.collect()

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    no_decay = (
        "bias",
        "LayerNorm.weight",
        "layer_norm.weight",
    )

    trainable_named = [
        (n, p)
        for n, p
        in model.named_parameters()
        if p.requires_grad
    ]

    optimizer_groups = [
        {
            "params": [
                p
                for n, p
                in trainable_named
                if not any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay":
                args.weight_decay,
        },
        {
            "params": [
                p
                for n, p
                in trainable_named
                if any(
                    x in n
                    for x in no_decay
                )
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = AdamW(
        optimizer_groups,
        lr=args.lr,
    )

    updates_per_epoch = math.ceil(
        len(train_loader)
        / args.grad_accum_steps
    )

    total_steps = (
        updates_per_epoch
        * args.epochs
    )

    warmup_steps = int(
        total_steps
        * args.warmup_ratio
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=
                warmup_steps,
            num_training_steps=
                total_steps,
        )
    )

    amp_enabled = bool(
        USE_AMP
        and device.type == "cuda"
    )

    try:
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=amp_enabled,
        )
    except Exception:
        scaler = (
            torch.cuda.amp.GradScaler(
                enabled=amp_enabled
            )
        )

    print(
        "\nTraining setup"
    )
    print(
        f"  unique train pairs   : "
        f"{len(train_ds):,}"
    )
    print(
        f"  batch                : "
        f"{args.train_batch_size}"
    )
    print(
        f"  grad accum           : "
        f"{args.grad_accum_steps}"
    )
    print(
        f"  effective q batch    : "
        f"{args.train_batch_size * args.grad_accum_steps}"
    )
    print(
        f"  explicit hard neg/q  : "
        f"{args.hard_negatives_per_query}"
    )
    print(
        f"  temperature          : "
        f"{args.temperature}"
    )
    print(
        f"  LR                   : "
        f"{args.lr}"
    )
    print(
        f"  epochs               : "
        f"{args.epochs}"
    )
    print(
        f"  total opt steps      : "
        f"{total_steps}"
    )
    print(
        f"  warmup steps         : "
        f"{warmup_steps}"
    )
    print(
        f"  current best epoch   : 0"
    )
    print(
        f"  current best R@20    : "
        f"{best_recall:.6f}"
    )
    print(
        f"  current best MRR     : "
        f"{best_mrr:.6f}"
    )

    history = []

    # --------------------------------------------------------
    # Fine-tuning
    # --------------------------------------------------------

    for epoch in range(
        1,
        args.epochs + 1,
    ):
        t0 = time.time()

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        running_loss = 0.0
        running_explicit = 0.0
        running_inbatch = 0.0
        running_exp_acc = 0.0
        running_ib_acc = 0.0

        for step, batch in enumerate(
            train_loader,
            1,
        ):
            q_batch = move_batch(
                batch[
                    "query_batch"
                ],
                device,
            )

            d_batch = move_batch(
                batch[
                    "doc_batch"
                ],
                device,
            )

            candidate_count = int(
                batch[
                    "candidate_count"
                ]
            )

            positive_rxcuis = (
                batch[
                    "positive_rxcuis"
                ]
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                q_emb = encode_batch(
                    model,
                    q_batch,
                )

                d_emb = encode_batch(
                    model,
                    d_batch,
                )

                loss_dict = retrieval_loss(
                    query_emb=q_emb,
                    doc_emb=d_emb,
                    candidate_count=
                        candidate_count,
                    positive_rxcuis=
                        positive_rxcuis,
                    temperature=
                        args.temperature,
                )

                full_loss = (
                    loss_dict["loss"]
                )

                loss = (
                    full_loss
                    / args.grad_accum_steps
                )

            scaler.scale(
                loss
            ).backward()

            running_loss += float(
                full_loss
                .detach()
                .cpu()
            )

            running_explicit += float(
                loss_dict[
                    "explicit_loss"
                ]
                .detach()
                .cpu()
            )

            running_inbatch += float(
                loss_dict[
                    "inbatch_loss"
                ]
                .detach()
                .cpu()
            )

            running_exp_acc += float(
                loss_dict[
                    "explicit_acc"
                ]
                .detach()
                .cpu()
            )

            running_ib_acc += float(
                loss_dict[
                    "inbatch_acc"
                ]
                .detach()
                .cpu()
            )

            should_update = (
                step
                % args.grad_accum_steps
                == 0
                or step
                == len(train_loader)
            )

            if should_update:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    [
                        p
                        for p
                        in model.parameters()
                        if p.requires_grad
                    ],
                    MAX_GRAD_NORM,
                )

                scaler.step(
                    optimizer
                )
                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

            if (
                step % 100 == 0
                or step
                == len(train_loader)
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"step "
                    f"{step:04d}/"
                    f"{len(train_loader):04d} | "
                    f"loss="
                    f"{running_loss/step:.5f} | "
                    f"exp_acc="
                    f"{running_exp_acc/step:.4f} | "
                    f"ib_acc="
                    f"{running_ib_acc/step:.4f} | "
                    f"lr="
                    f"{scheduler.get_last_lr()[0]:.3e}"
                )

        train_stats = {
            "loss":
                running_loss
                / len(train_loader),
            "explicit_loss":
                running_explicit
                / len(train_loader),
            "inbatch_loss":
                running_inbatch
                / len(train_loader),
            "explicit_accuracy":
                running_exp_acc
                / len(train_loader),
            "inbatch_accuracy":
                running_ib_acc
                / len(train_loader),
        }

        # ----------------------------------------------------
        # Full-catalog dense validation
        # ----------------------------------------------------

        print(
            f"\nEncoding RxNorm concept catalog "
            f"for epoch {epoch}..."
        )

        concept_emb = (
            encode_concept_catalog(
                model=model,
                tokenizer=tokenizer,
                concepts=concepts,
                device=device,
                batch_size=
                    args.encode_doc_batch_size,
                doc_max_length=
                    args.doc_max_length,
            )
        )

        dense_rows = dense_retrieve(
            model=model,
            tokenizer=tokenizer,
            valid_rows=valid_rows,
            concepts=concepts,
            concept_embeddings=
                concept_emb,
            device=device,
            query_batch_size=
                args.eval_query_batch_size,
            query_max_length=
                args.query_max_length,
            top_k=
                DENSE_SAVE_TOP_K,
        )

        dense_metrics = (
            metrics_from_ranked_rows(
                dense_rows
            )
        )

        print_metrics(
            f"DENSE_EPOCH_{epoch}",
            dense_metrics,
        )

        alpha, (
            hybrid_metrics
        ), hybrid_rows = (
            tune_hybrid_alpha(
                lexical_rows=
                    lexical_valid_rows,
                dense_rows=
                    dense_rows,
                concepts=concepts,
                rxcui_to_index=
                    rxcui_to_index,
            )
        )

        print(
            f"\n[EPOCH {epoch} HYBRID BEST] "
            f"alpha={alpha:.2f}"
        )

        print_metrics(
            f"HYBRID_EPOCH_{epoch}",
            hybrid_metrics,
        )

        epoch_record = {
            "epoch":
                epoch,
            "train":
                train_stats,
            "dense":
                dense_metrics,
            "hybrid_alpha":
                alpha,
            "hybrid":
                hybrid_metrics,
            "seconds":
                time.time() - t0,
        }

        history.append(
            epoch_record
        )

        save_json(
            history,
            output_root
            / "training_history.json",
        )

        save_model_only(
            model=model,
            tokenizer=tokenizer,
            directory=
                output_root
                / "last",
            epoch=epoch,
            args=args,
        )

        current_recall = float(
            hybrid_metrics[
                f"recall@{SELECTION_K}"
            ]
        )

        current_mrr = float(
            hybrid_metrics["mrr"]
        )

        recall_improved = (
            current_recall
            > best_recall
            + MIN_DELTA
        )

        recall_tied = (
            abs(
                current_recall
                - best_recall
            )
            <= MIN_DELTA
        )

        mrr_improved = (
            current_mrr
            > best_mrr
            + MIN_DELTA
        )

        improved = (
            recall_improved
            or (
                recall_tied
                and mrr_improved
            )
        )

        if improved:
            best_epoch = epoch
            best_recall = (
                current_recall
            )
            best_mrr = (
                current_mrr
            )
            best_alpha = (
                float(alpha)
            )
            no_improvement = 0

            print(
                "\n*** NEW BEST *** "
                f"epoch={epoch} | "
                f"Hybrid Recall@20="
                f"{best_recall:.6f} | "
                f"MRR="
                f"{best_mrr:.6f} | "
                f"alpha="
                f"{best_alpha:.2f}"
            )

            save_best_bundle(
                model=model,
                tokenizer=tokenizer,
                directory=
                    output_root
                    / "best",
                epoch=epoch,
                dense_metrics=
                    dense_metrics,
                hybrid_alpha=
                    alpha,
                hybrid_metrics=
                    hybrid_metrics,
                hybrid_rows=
                    hybrid_rows,
                concepts=concepts,
                embeddings=
                    concept_emb,
                args=args,
                lexical_metrics=
                    lexical_metrics,
                zero_shot_metrics={
                    "dense":
                        zero_dense_metrics,
                    "hybrid_alpha":
                        zero_alpha,
                    "hybrid":
                        zero_hybrid_metrics,
                },
            )

        else:
            no_improvement += 1

            print(
                f"\nNo improvement: "
                f"{no_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        print(
            f"\nEpoch {epoch} done | "
            f"best_epoch={best_epoch} | "
            f"best_R@20="
            f"{best_recall:.6f} | "
            f"best_MRR="
            f"{best_mrr:.6f} | "
            f"best_alpha="
            f"{best_alpha:.2f}\n"
        )

        del concept_emb
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

        if (
            no_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "Early stopping triggered."
            )
            break

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print("=" * 78)
    print(
        "PHASE 4 TRAINING COMPLETE"
    )
    print("=" * 78)

    print(
        f"Best epoch       : "
        f"{best_epoch}"
    )

    if best_epoch == 0:
        print(
            "Best dense model : "
            "ZERO-SHOT multilingual E5"
        )
    else:
        print(
            "Best dense model : "
            "fine-tuned multilingual E5"
        )

    print(
        f"Best Hybrid R@20 : "
        f"{best_recall:.6f}"
    )

    print(
        f"Best Hybrid MRR  : "
        f"{best_mrr:.6f}"
    )

    print(
        f"Best alpha       : "
        f"{best_alpha:.2f}"
    )

    print(
        f"Best checkpoint  : "
        f"{output_root / 'best'}"
    )

    print(
        "\nInference artifacts:"
    )

    print(
        f"  model/tokenizer : "
        f"{output_root / 'best'}"
    )

    print(
        f"  embeddings      : "
        f"{output_root / 'best' / 'rxnorm_concept_embeddings.npy'}"
    )

    print(
        f"  metadata        : "
        f"{output_root / 'best' / 'rxnorm_concept_metadata.jsonl'}"
    )

    print(
        f"  hybrid config   : "
        f"{output_root / 'best' / 'hybrid_config.json'}"
    )

    print(
        f"  overrides       : "
        f"{output_root / 'organizer_overrides.json'}"
    )


if __name__ == "__main__":
    main()

[INFO] Ignoring unknown Jupyter/Colab args: ['-f', '/root/.local/share/jupyter/runtime/kernel-d05d07bd-8c74-49a0-903b-0a89a05fe25b.json']
PHASE 4 - RXNORM HYBRID DRUG LINKER
train       : /content/drive/MyDrive/data/rxnorm_link_train.jsonl
valid       : /content/drive/MyDrive/data/rxnorm_link_valid_concept.jsonl
rxnorm.csv  : /content/drive/MyDrive/data/rxnorm.csv
model       : intfloat/multilingual-e5-base
output      : /content/drive/MyDrive/output_phase4_rxnorm
device      : cuda
GPU         : Tesla T4
[RXNORM] unique term rows=160,822
[RXNORM] unique concepts/RXCUI=82,455
[TRAIN] raw_rows=30,000 | unique_queries=27,497 | unique_pairs=27,585 | positive_RXCUI=8,567
[VALID] raw_rows=4,000 | unique_pairs=3,610 | heldout_RXCUI=964
[OVERRIDES] drug mention variants=24

LEXICAL BASELINE
[LEXICAL] exact keys=316,546
[LEXICAL] fitting char-TFIDF on 160,822 RxNorm terms...
[LEXICAL] TFIDF matrix=(160822, 95874) | nnz=15,747,138
[LEXICAL] 250/3,610
[LEXICAL] 500/3,610
[LEXICAL] 750/3,610
[LEX

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[FREEZE] bottom layers 3/12 frozen
[MODEL] gradient checkpointing enabled
[PARAMS] total=278,043,648 | trainable=256,780,032 | 92.35%

ZERO-SHOT DENSE + HYBRID EVALUATION
[DENSE KB] 96/82,455
[DENSE KB] 4,896/82,455
[DENSE KB] 9,696/82,455
[DENSE KB] 14,496/82,455
[DENSE KB] 19,296/82,455
[DENSE KB] 24,096/82,455
[DENSE KB] 28,896/82,455
[DENSE KB] 33,696/82,455
[DENSE KB] 38,496/82,455
[DENSE KB] 43,296/82,455
[DENSE KB] 48,096/82,455
[DENSE KB] 52,896/82,455
[DENSE KB] 57,696/82,455
[DENSE KB] 62,496/82,455
[DENSE KB] 67,296/82,455
[DENSE KB] 72,096/82,455
[DENSE KB] 76,896/82,455
[DENSE KB] 81,696/82,455

[ZERO_SHOT_DENSE]
  Recall@1  = 0.412465
  Recall@5  = 0.703601
  Recall@10 = 0.755679
  Recall@20 = 0.793629
  MRR       = 0.543432
[HYBRID alpha=0.0] R@20=0.985319 | MRR=0.719701
[HYBRID alpha=0.1] R@20=0.991413 | MRR=0.748278
[HYBRID alpha=0.2] R@20=0.993629 | MRR=0.748528
[HYBRID alpha=0.3] R@20=0.993906 | MRR=0.728341
[HYBRID alpha=0.4] R@20=0.993629 | MRR=0.706011
[HYBRID alp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS] optional index not saved: No module named 'faiss'

Training setup
  unique train pairs   : 27,585
  batch                : 64
  grad accum           : 2
  effective q batch    : 128
  explicit hard neg/q  : 4
  temperature          : 0.05
  LR                   : 1e-05
  epochs               : 4
  total opt steps      : 864
  warmup steps         : 86
  current best epoch   : 0
  current best R@20    : 0.993906
  current best MRR     : 0.728341


...

Epoch 01 | step 0100/0432 | loss=0.89204 | exp_acc=0.9825 | ib_acc=0.9816 | lr=5.814e-06
Epoch 01 | step 0200/0432 | loss=0.46092 | exp_acc=0.9892 | ib_acc=0.9895 | lr=9.820e-06
Epoch 01 | step 0300/0432 | loss=0.31091 | exp_acc=0.9916 | ib_acc=0.9927 | lr=9.177e-06
Epoch 01 | step 0400/0432 | loss=0.23549 | exp_acc=0.9929 | ib_acc=0.9943 | lr=8.535e-06
Epoch 01 | step 0432/0432 | loss=0.21856 | exp_acc=0.9934 | ib_acc=0.9948 | lr=8.329e-06

Encoding RxNorm concept catalog for epoch 1...
[DENSE KB] 96/82,455
[DENSE KB] 4,896/82,455
[DENSE KB] 9,696/82,455
[DENSE KB] 14,496/82,455
[DENSE KB] 19,296/82,455
[DENSE KB] 24,096/82,455
[DENSE KB] 28,896/82,455
[DENSE KB] 33,696/82,455
[DENSE KB] 38,496/82,455
[DENSE KB] 43,296/82,455
[DENSE KB] 48,096/82,455
[DENSE KB] 52,896/82,455
[DENSE KB] 57,696/82,455
[DENSE KB] 62,496/82,455
[DENSE KB] 67,296/82,455
[DENSE KB] 72,096/82,455
[DENSE KB] 76,896/82,455
[DENSE KB] 81,696/82,455

[DENSE_EPOCH_1]
  Recall@1  = 0.825485
  Recall@5  = 0.993352


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


*** NEW BEST *** epoch=1 | Hybrid Recall@20=1.000000 | MRR=0.895514 | alpha=0.70


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS] optional index not saved: No module named 'faiss'

Epoch 1 done | best_epoch=1 | best_R@20=1.000000 | best_MRR=0.895514 | best_alpha=0.70



In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
PHASE 5 — End-to-End Viettel AI Race Clinical NLP Inference
============================================================
Integrates:
  Phase 1: ViHealthBERT NER (sliding-window, raw character offsets)
  Phase 2: ViHealthBERT assertion classifier
  Phase 3: multilingual-E5 ICD-10 retriever
  Phase 4: multilingual-E5 + lexical RxNorm hybrid linker
           (automatic lexical fallback if Phase-4 best bundle is not ready)

Competition output:
  output/1.json ... output/100.json
Each JSON is a list of entity dictionaries.
Positions are [start, end), i.e. Python slicing convention.

Designed for Colab/Drive paths produced by the four training scripts.
This COLAB edition can be pasted into one cell and run without CLI arguments.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import re
import shutil
import sys
import time
import unicodedata
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np


# -----------------------------------------------------------------------------
# Defaults matching the actual four training runs
# -----------------------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_ROOT = DRIVE_ROOT / "data"
PHASE1_DIR = DRIVE_ROOT / "output" / "best"
PHASE2_DIR = DRIVE_ROOT / "output_phase2_assertion" / "best"
PHASE3_DIR = DRIVE_ROOT / "output_phase3_icd_retriever" / "best"
PHASE4_ROOT = DRIVE_ROOT / "output_phase4_rxnorm"
PHASE4_DIR = PHASE4_ROOT / "best"
RXNORM_CSV = DATA_ROOT / "rxnorm.csv"

# Colab-direct defaults. When this file is pasted into a Colab cell and Run is
# pressed, no CLI arguments are required.
DEFAULT_INPUT_ZIP = DRIVE_ROOT / "input_turn2_vong1.zip"
DEFAULT_INPUT_ZIP_DATA = DATA_ROOT / "input_turn2_vong1.zip"
DEFAULT_INPUT_DIR = DRIVE_ROOT / "input"

ENTITY_TYPES = [
    "TRIỆU_CHỨNG",
    "TÊN_XÉT_NGHIỆM",
    "KẾT_QUẢ_XÉT_NGHIỆM",
    "CHẨN_ĐOÁN",
    "THUỐC",
]
ASSERTION_TYPES = {"TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC"}
ASSERTION_LABELS = ["isNegated", "isFamily", "isHistorical"]
TYPE_TOKEN = {
    "TRIỆU_CHỨNG": "<TYPE_TRIEU_CHUNG>",
    "CHẨN_ĐOÁN": "<TYPE_CHAN_DOAN>",
    "THUỐC": "<TYPE_THUOC>",
}
ENT_START = "<ENT>"
ENT_END = "</ENT>"

TYPE_TO_TAG = {
    "TRIỆU_CHỨNG": "TRIEU_CHUNG",
    "TÊN_XÉT_NGHIỆM": "TEN_XET_NGHIEM",
    "KẾT_QUẢ_XÉT_NGHIỆM": "KET_QUA_XET_NGHIEM",
    "CHẨN_ĐOÁN": "CHAN_DOAN",
    "THUỐC": "THUOC",
}
TAG_TO_TYPE = {v: k for k, v in TYPE_TO_TAG.items()}
EXPECTED_NER_LABELS = ["O"] + [
    x
    for et in ENTITY_TYPES
    for x in (f"B-{TYPE_TO_TAG[et]}", f"I-{TYPE_TO_TAG[et]}")
]

SURFACE_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


# -----------------------------------------------------------------------------
# Basic helpers
# -----------------------------------------------------------------------------
def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else " ".join(s.split())


def numeric_sort_key(p: Path):
    try:
        return (0, int(p.stem))
    except Exception:
        return (1, p.name)


def choose_device(requested: str, torch_mod):
    if requested == "auto":
        return torch_mod.device("cuda" if torch_mod.cuda.is_available() else "cpu")
    return torch_mod.device(requested)


def require_ml_runtime():
    """Lazy import so --preflight can run without transformers installed."""
    try:
        import torch
        import torch.nn.functional as F
        from transformers import (
            AutoModel,
            AutoModelForSequenceClassification,
            AutoModelForTokenClassification,
            AutoTokenizer,
        )
    except Exception as exc:
        raise RuntimeError(
            "ML runtime is unavailable. Install transformers + sentencepiece and run "
            "in Colab/Kaggle where the checkpoints are mounted. Original error: "
            f"{exc}"
        ) from exc
    return torch, F, AutoModel, AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer


def surface_pieces(text: str) -> List[Dict[str, Any]]:
    return [
        {"text": m.group(0), "start": int(m.start()), "end": int(m.end())}
        for m in SURFACE_RE.finditer(text)
    ]


def _bos_eos(tokenizer) -> Tuple[int, int]:
    bos = tokenizer.bos_token_id
    eos = tokenizer.eos_token_id
    if bos is None:
        bos = tokenizer.cls_token_id
    if eos is None:
        eos = tokenizer.sep_token_id
    if bos is None or eos is None:
        raise RuntimeError("Tokenizer must provide BOS/CLS and EOS/SEP IDs.")
    return int(bos), int(eos)


def _tokenize_piece(tokenizer, piece: str) -> List[int]:
    ids = tokenizer.encode(piece, add_special_tokens=False)
    if not ids:
        if tokenizer.unk_token_id is None:
            raise RuntimeError(f"No token IDs for piece: {piece!r}")
        ids = [int(tokenizer.unk_token_id)]
    return [int(x) for x in ids]


# -----------------------------------------------------------------------------
# Phase 1 — Sliding-window NER
# -----------------------------------------------------------------------------
class SlidingNER:
    def __init__(
        self,
        checkpoint: Path,
        device: str = "auto",
        max_length: int = 256,
        overlap_tokens: int = 64,
        batch_size: int = 12,
    ):
        (
            self.torch,
            self.F,
            _AutoModel,
            _AutoSeq,
            AutoToken,
            AutoTokenizer,
        ) = require_ml_runtime()
        self.device = choose_device(device, self.torch)
        self.checkpoint = Path(checkpoint)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.checkpoint, use_fast=False, local_files_only=True
        )
        self.model = AutoToken.from_pretrained(
            self.checkpoint, local_files_only=True
        ).to(self.device)
        self.model.eval()

        config_cap = int(getattr(self.model.config, "max_position_embeddings", max_length + 2)) - 2
        self.max_length = max(8, min(int(max_length), config_cap))
        self.content_limit = self.max_length - 2
        self.overlap_tokens = max(0, min(int(overlap_tokens), self.content_limit // 2))
        self.batch_size = max(1, int(batch_size))
        self.bos_id, self.eos_id = _bos_eos(self.tokenizer)
        self.pad_id = int(self.tokenizer.pad_token_id)

        raw_id2label = getattr(self.model.config, "id2label", {}) or {}
        id2label = {int(k): str(v) for k, v in raw_id2label.items()}
        if set(id2label.values()) >= set(EXPECTED_NER_LABELS):
            self.id2label = id2label
        elif int(getattr(self.model.config, "num_labels", 0)) == len(EXPECTED_NER_LABELS):
            self.id2label = {i: x for i, x in enumerate(EXPECTED_NER_LABELS)}
        else:
            raise RuntimeError(
                "Phase-1 checkpoint label mapping does not match the 5 trained entity types."
            )

    def _make_windows(self, piece_token_ids: List[List[int]]) -> List[Tuple[int, int]]:
        n = len(piece_token_ids)
        windows: List[Tuple[int, int]] = []
        start = 0
        while start < n:
            total = 0
            end = start
            while end < n:
                plen = len(piece_token_ids[end])
                if plen > self.content_limit:
                    # Extremely pathological single surface token; retain a truncated view.
                    plen = self.content_limit
                if end > start and total + plen > self.content_limit:
                    break
                if end == start and plen > self.content_limit:
                    end += 1
                    total = self.content_limit
                    break
                if total + plen > self.content_limit:
                    break
                total += plen
                end += 1
            if end <= start:
                end = start + 1
            windows.append((start, end))
            if end >= n:
                break

            # Move next start backward from end to preserve ~overlap_tokens context.
            back = end
            overlap = 0
            while back > start + 1 and overlap < self.overlap_tokens:
                back -= 1
                overlap += min(len(piece_token_ids[back]), self.content_limit)
            next_start = max(start + 1, back)
            start = next_start
        return windows

    def _build_window(
        self,
        piece_token_ids: List[List[int]],
        start: int,
        end: int,
    ) -> Dict[str, Any]:
        content: List[int] = []
        first_positions: List[Tuple[int, int]] = []  # (global_piece_idx, token_position)
        for pidx in range(start, end):
            ids = piece_token_ids[pidx]
            if len(ids) > self.content_limit:
                ids = ids[: self.content_limit]
            if len(content) + len(ids) > self.content_limit:
                break
            first_positions.append((pidx, 1 + len(content)))
            content.extend(ids)
        return {
            "input_ids": [self.bos_id] + content + [self.eos_id],
            "first_positions": first_positions,
        }

    def predict(self, text: str) -> List[Dict[str, Any]]:
        pieces = surface_pieces(text)
        if not pieces:
            return []
        token_ids = [_tokenize_piece(self.tokenizer, p["text"]) for p in pieces]
        windows = [self._build_window(token_ids, s, e) for s, e in self._make_windows(token_ids)]

        n_labels = int(self.model.config.num_labels)
        logit_sum = np.zeros((len(pieces), n_labels), dtype=np.float64)
        vote_count = np.zeros(len(pieces), dtype=np.int32)

        for b0 in range(0, len(windows), self.batch_size):
            chunk = windows[b0 : b0 + self.batch_size]
            max_len = max(len(x["input_ids"]) for x in chunk)
            input_ids = []
            attention = []
            for x in chunk:
                ids = x["input_ids"]
                pad_n = max_len - len(ids)
                input_ids.append(ids + [self.pad_id] * pad_n)
                attention.append([1] * len(ids) + [0] * pad_n)
            batch = {
                "input_ids": self.torch.tensor(input_ids, dtype=self.torch.long, device=self.device),
                "attention_mask": self.torch.tensor(attention, dtype=self.torch.long, device=self.device),
            }
            with self.torch.no_grad():
                with self.torch.autocast(
                    device_type=self.device.type,
                    dtype=self.torch.float16,
                    enabled=self.device.type == "cuda",
                ):
                    logits = self.model(**batch).logits
            logits = logits.detach().float().cpu().numpy()
            for i, x in enumerate(chunk):
                for pidx, token_pos in x["first_positions"]:
                    logit_sum[pidx] += logits[i, token_pos]
                    vote_count[pidx] += 1

        avg = logit_sum / np.clip(vote_count[:, None], 1, None)
        pred_ids = avg.argmax(axis=1).tolist()
        labels = [self.id2label[int(i)] for i in pred_ids]

        entities: List[Dict[str, Any]] = []
        current_type: Optional[str] = None
        current_start: Optional[int] = None
        current_end: Optional[int] = None

        def close_current():
            nonlocal current_type, current_start, current_end
            if current_type is not None and current_start is not None and current_end is not None:
                entities.append(
                    {
                        "text": text[current_start:current_end],
                        "type": current_type,
                        "position": [int(current_start), int(current_end)],
                    }
                )
            current_type = current_start = current_end = None

        for label, piece in zip(labels, pieces):
            if label == "O" or "-" not in label:
                close_current()
                continue
            prefix, tag = label.split("-", 1)
            etype = TAG_TO_TYPE.get(tag)
            if etype is None:
                close_current()
                continue
            if prefix == "B" or current_type != etype:
                close_current()
                current_type = etype
                current_start = int(piece["start"])
                current_end = int(piece["end"])
            else:
                current_end = int(piece["end"])
        close_current()

        # Exact raw-text invariant and deterministic order.
        entities = [
            e
            for e in entities
            if 0 <= e["position"][0] < e["position"][1] <= len(text)
            and text[e["position"][0] : e["position"][1]] == e["text"]
        ]
        entities.sort(key=lambda e: (e["position"][0], e["position"][1], e["type"]))
        return entities


# -----------------------------------------------------------------------------
# Phase 2 — Assertions
# -----------------------------------------------------------------------------
def _encode_no_special(tokenizer, text: str) -> List[int]:
    if not text:
        return []
    return [int(x) for x in tokenizer.encode(text, add_special_tokens=False)]


def _allocate_context_budget(prefix_len: int, suffix_len: int, budget: int) -> Tuple[int, int]:
    if budget <= 0:
        return 0, 0
    n_prefix = min(prefix_len, budget // 2)
    n_suffix = min(suffix_len, budget - n_prefix)
    remaining = budget - n_prefix - n_suffix
    while remaining > 0:
        left_avail = prefix_len - n_prefix
        right_avail = suffix_len - n_suffix
        if left_avail <= 0 and right_avail <= 0:
            break
        if left_avail >= right_avail and left_avail > 0:
            x = min(left_avail, remaining)
            n_prefix += x
        else:
            x = min(right_avail, remaining)
            n_suffix += x
        remaining -= x
    return n_prefix, n_suffix


class AssertionClassifier:
    def __init__(self, checkpoint: Path, device: str = "auto", max_length: int = 256, batch_size: int = 32):
        (
            self.torch,
            self.F,
            _AutoModel,
            AutoSeq,
            _AutoToken,
            AutoTokenizer,
        ) = require_ml_runtime()
        self.device = choose_device(device, self.torch)
        self.checkpoint = Path(checkpoint)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.checkpoint, use_fast=False, local_files_only=True
        )
        self.model = AutoSeq.from_pretrained(self.checkpoint, local_files_only=True).to(self.device)
        self.model.eval()
        config_cap = int(getattr(self.model.config, "max_position_embeddings", max_length + 2)) - 2
        self.max_length = max(8, min(int(max_length), config_cap))
        self.batch_size = max(1, int(batch_size))
        thresholds = load_json(self.checkpoint / "thresholds.json", {}) or {}
        self.thresholds = np.array([float(thresholds.get(x, 0.5)) for x in ASSERTION_LABELS], dtype=np.float32)

    def _encode(self, text: str, entity: Dict[str, Any]) -> Dict[str, List[int]]:
        s, e = map(int, entity["position"])
        prefix = text[:s]
        ent_text = text[s:e]
        suffix = text[e:]
        prefix_ids = _encode_no_special(self.tokenizer, prefix)
        entity_ids = _encode_no_special(self.tokenizer, ent_text)
        suffix_ids = _encode_no_special(self.tokenizer, suffix)
        type_ids = _encode_no_special(self.tokenizer, TYPE_TOKEN[entity["type"]])
        ent_start_ids = _encode_no_special(self.tokenizer, ENT_START)
        ent_end_ids = _encode_no_special(self.tokenizer, ENT_END)
        n_outer = int(self.tokenizer.num_special_tokens_to_add(pair=False))
        content_limit = self.max_length - n_outer
        mandatory = len(type_ids) + len(ent_start_ids) + len(entity_ids) + len(ent_end_ids)
        if mandatory > content_limit:
            # Preserve the entity, trimming its subword view only as a last-resort guard.
            room = max(1, content_limit - len(type_ids) - len(ent_start_ids) - len(ent_end_ids))
            entity_ids = entity_ids[:room]
            mandatory = len(type_ids) + len(ent_start_ids) + len(entity_ids) + len(ent_end_ids)
        budget = max(0, content_limit - mandatory)
        npre, nsuf = _allocate_context_budget(len(prefix_ids), len(suffix_ids), budget)
        content = (
            type_ids
            + (prefix_ids[-npre:] if npre else [])
            + ent_start_ids
            + entity_ids
            + ent_end_ids
            + (suffix_ids[:nsuf] if nsuf else [])
        )
        ids = self.tokenizer.build_inputs_with_special_tokens(content)
        return {"input_ids": ids, "attention_mask": [1] * len(ids)}

    def predict(self, text: str, entities: List[Dict[str, Any]]) -> Dict[int, List[str]]:
        eligible = [(i, e) for i, e in enumerate(entities) if e["type"] in ASSERTION_TYPES]
        out: Dict[int, List[str]] = {}
        if not eligible:
            return out
        features = [(idx, self._encode(text, ent)) for idx, ent in eligible]
        for b0 in range(0, len(features), self.batch_size):
            chunk = features[b0 : b0 + self.batch_size]
            padded = self.tokenizer.pad(
                [x[1] for x in chunk], padding=True, return_tensors="pt"
            )
            padded = {k: v.to(self.device) for k, v in padded.items()}
            with self.torch.no_grad():
                with self.torch.autocast(
                    device_type=self.device.type,
                    dtype=self.torch.float16,
                    enabled=self.device.type == "cuda",
                ):
                    logits = self.model(**padded).logits
            probs = self.torch.sigmoid(logits).detach().float().cpu().numpy()
            for row_i, (entity_idx, _) in enumerate(chunk):
                labels = [
                    label
                    for j, label in enumerate(ASSERTION_LABELS)
                    if float(probs[row_i, j]) >= float(self.thresholds[j])
                ]
                out[int(entity_idx)] = labels
        return out


# -----------------------------------------------------------------------------
# Shared E5 encoder helper
# -----------------------------------------------------------------------------
def _average_pool(torch_mod, hidden, attention_mask):
    mask = attention_mask[..., None].bool()
    hidden = hidden.masked_fill(~mask, 0.0)
    denom = attention_mask.sum(dim=1)[..., None].clamp(min=1)
    return hidden.sum(dim=1) / denom


class DenseCatalogLinker:
    def __init__(
        self,
        checkpoint: Path,
        embeddings_name: str,
        metadata_name: str,
        id_field: str,
        device: str = "auto",
        query_max_length: int = 64,
        query_prefix: str = "query: ",
    ):
        (
            self.torch,
            self.F,
            AutoModel,
            _AutoSeq,
            _AutoToken,
            AutoTokenizer,
        ) = require_ml_runtime()
        self.device = choose_device(device, self.torch)
        self.checkpoint = Path(checkpoint)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.checkpoint, local_files_only=True
        )
        self.model = AutoModel.from_pretrained(
            self.checkpoint, local_files_only=True
        ).to(self.device)
        self.model.eval()
        self.embeddings = np.load(self.checkpoint / embeddings_name, mmap_mode="r")
        self.metadata = read_jsonl(self.checkpoint / metadata_name)
        if len(self.metadata) != int(self.embeddings.shape[0]):
            raise RuntimeError(
                f"Catalog metadata/embedding mismatch in {self.checkpoint}: "
                f"{len(self.metadata)} vs {self.embeddings.shape[0]}"
            )
        self.id_field = id_field
        self.query_max_length = int(query_max_length)
        self.query_prefix = str(query_prefix)

    def _encode_queries(self, queries: List[str]) -> np.ndarray:
        tokens = self.tokenizer(
            [self.query_prefix + q for q in queries],
            padding=True,
            truncation=True,
            max_length=self.query_max_length,
            return_tensors="pt",
        )
        tokens = {k: v.to(self.device) for k, v in tokens.items()}
        with self.torch.no_grad():
            with self.torch.autocast(
                device_type=self.device.type,
                dtype=self.torch.float16,
                enabled=self.device.type == "cuda",
            ):
                hidden = self.model(**tokens).last_hidden_state
                emb = _average_pool(self.torch, hidden, tokens["attention_mask"])
                emb = self.F.normalize(emb, p=2, dim=1)
        return emb.detach().float().cpu().numpy().astype(np.float32)

    def retrieve(self, queries: List[str], top_k: int = 20, batch_size: int = 64) -> List[List[Dict[str, Any]]]:
        all_out: List[List[Dict[str, Any]]] = []
        catalog = np.asarray(self.embeddings, dtype=np.float32)
        k = max(1, min(int(top_k), int(catalog.shape[0])))
        for b0 in range(0, len(queries), batch_size):
            q = self._encode_queries(queries[b0 : b0 + batch_size])
            scores = q @ catalog.T
            for row in scores:
                if k < len(row):
                    idx = np.argpartition(-row, k - 1)[:k]
                    idx = idx[np.argsort(-row[idx])]
                else:
                    idx = np.argsort(-row)
                all_out.append(
                    [
                        {
                            "id": str(self.metadata[int(i)][self.id_field]),
                            "score": float(row[int(i)]),
                            "meta": self.metadata[int(i)],
                        }
                        for i in idx[:k]
                    ]
                )
        return all_out


# -----------------------------------------------------------------------------
# Phase 3 — ICD-10
# -----------------------------------------------------------------------------
class ICDLinker(DenseCatalogLinker):
    def __init__(self, checkpoint: Path, device="auto"):
        cfg = load_json(Path(checkpoint) / "retrieval_config.json", {}) or {}
        qmax = int(cfg.get("query_max_length", 64))
        prefix = cfg.get("model_prefixes", {}).get("query", cfg.get("query_prefix", "query: "))
        super().__init__(
            checkpoint=checkpoint,
            embeddings_name="icd_embeddings.npy",
            metadata_name="icd_metadata.jsonl",
            id_field="code",
            device=device,
            query_max_length=qmax,
            query_prefix=prefix,
        )

    def link(self, mentions: List[str], output_k: int = 1) -> List[List[str]]:
        rows = self.retrieve(mentions, top_k=max(output_k, 20))
        return [[x["id"] for x in row[:output_k]] for row in rows]


# -----------------------------------------------------------------------------
# Phase 4 — RxNorm lexical + dense hybrid
# -----------------------------------------------------------------------------
ROUTE_FREQ_PATTERNS = [
    r"\bpo\b", r"\boral(?:ly)?\b", r"\biv\b", r"\bim\b", r"\bsc\b", r"\bsq\b",
    r"\bsubq\b", r"\bprn\b", r"\bdaily\b", r"\bonce daily\b", r"\bbid\b", r"\btid\b",
    r"\bqid\b", r"\bqam\b", r"\bqpm\b", r"\bqhs\b", r"\bq\d+h\b",
    r"\bevery \d+ hours?\b", r"\bat bedtime\b", r"\bas needed\b",
]
TTY_PRIORITY = {
    "SCD": 0, "SBD": 1, "PSN": 2, "PIN": 3, "MIN": 4, "IN": 5, "BN": 6,
    "GPCK": 7, "BPCK": 8, "SCDC": 9, "SBDC": 10, "SCDF": 11, "SBDF": 12,
}


def normalize_unicode(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s)).lower().replace("μg", "mcg").replace("µg", "mcg")
    return re.sub(r"\s+", " ", s).strip()


def normalize_rx_text(s: str) -> str:
    s = normalize_unicode(s)
    s = re.sub(r"\bmilligrams?\b", "mg", s)
    s = re.sub(r"\bmilligram\b", "mg", s)
    s = re.sub(r"\bmicrograms?\b", "mcg", s)
    s = re.sub(r"\bgrams?\b", "g", s)
    s = re.sub(r"\bmilliliters?\b", "ml", s)
    s = re.sub(r"\s*/\s*", "/", s)
    s = re.sub(r"[^a-z0-9\.\-/\s\[\]\(\)%]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def strip_schedule_route(s: str) -> str:
    s = normalize_rx_text(s)
    for pat in ROUTE_FREQ_PATTERNS:
        s = re.sub(pat, " ", s, flags=re.IGNORECASE)
    s = re.sub(r"[:;,]+$", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def lexical_query_variants(s: str) -> List[str]:
    out, seen = [], set()
    for x in (normalize_rx_text(s), strip_schedule_route(s)):
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out


class RxNormLexical:
    def __init__(self, rxnorm_csv: Path, overrides_json: Optional[Path] = None):
        try:
            import pandas as pd
            from sklearn.feature_extraction.text import TfidfVectorizer
        except Exception as exc:
            raise RuntimeError("Phase-4 lexical fallback requires pandas + scikit-learn") from exc
        self.pd = pd
        df = pd.read_csv(rxnorm_csv, dtype=str).fillna("")
        required = {"normalized_name", "rxcui", "display_name", "tty"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"rxnorm.csv missing columns: {sorted(missing)}")
        df["rxcui"] = df["rxcui"].astype(str).str.strip()
        df["display_name"] = df["display_name"].astype(str).str.strip()
        df["tty"] = df["tty"].astype(str).str.strip()
        df["lexical_text"] = [normalize_rx_text(a if clean_text(a) else b) for a, b in zip(df["normalized_name"], df["display_name"])]
        df = df[(df.rxcui != "") & (df.display_name != "") & (df.lexical_text != "")].drop_duplicates(["rxcui", "lexical_text"]).reset_index(drop=True)
        self.df = df

        grouped: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
        for row in df.to_dict(orient="records"):
            grouped[str(row["rxcui"])].append(row)
        concepts = []
        for rxcui, rows in grouped.items():
            rows.sort(key=lambda x: (TTY_PRIORITY.get(clean_text(x.get("tty")), 999), -len(clean_text(x.get("display_name")))))
            p = rows[0]
            concepts.append({"rxcui": rxcui, "display_name": p["display_name"], "tty": p["tty"]})
        concepts.sort(key=lambda x: (0, int(x["rxcui"])) if x["rxcui"].isdigit() else (1, x["rxcui"]))
        self.concepts = concepts
        self.rxcui_to_index = {x["rxcui"]: i for i, x in enumerate(concepts)}
        self.term_to_concept = np.array([self.rxcui_to_index[str(x)] for x in df["rxcui"]], dtype=np.int32)

        exact: Dict[str, set] = defaultdict(set)
        for row in df.to_dict(orient="records"):
            for key in {normalize_rx_text(row["display_name"]), normalize_rx_text(row["lexical_text"]), strip_schedule_route(row["display_name"])}:
                if key:
                    exact[key].add(str(row["rxcui"]))
        self.exact_map = {k: sorted(v) for k, v in exact.items()}

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=1, lowercase=False, dtype=np.float32, norm="l2"
        )
        self.term_matrix = self.vectorizer.fit_transform(df["lexical_text"].astype(str).tolist()).tocsr()
        self.overrides = load_json(overrides_json, {}) if overrides_json and Path(overrides_json).exists() else {}

    def override_candidates(self, query: str) -> Optional[List[str]]:
        if not self.overrides:
            return None
        for key in (normalize_unicode(query), normalize_rx_text(query), strip_schedule_route(query)):
            item = self.overrides.get(key)
            if item:
                vals = item.get("candidates", []) if isinstance(item, dict) else item
                if vals:
                    return [str(x) for x in vals]
        return None

    def retrieve_one(self, query: str, top_k: int = 100) -> List[Dict[str, Any]]:
        variants = lexical_query_variants(query)
        exact_rxcuis = set()
        for v in variants:
            exact_rxcuis.update(self.exact_map.get(v, []))
        q_text = variants[-1] if variants else normalize_rx_text(query)
        q_vec = self.vectorizer.transform([q_text])
        scores = (q_vec @ self.term_matrix.T).toarray()[0]
        term_top_n = min(max(int(top_k) * 8, 100), len(scores))
        if term_top_n < len(scores):
            idx = np.argpartition(-scores, term_top_n - 1)[:term_top_n]
            idx = idx[np.argsort(-scores[idx])]
        else:
            idx = np.argsort(-scores)
        concept_score: Dict[int, float] = {}
        for term_idx in idx:
            cidx = int(self.term_to_concept[int(term_idx)])
            score = float(scores[int(term_idx)])
            if cidx not in concept_score or score > concept_score[cidx]:
                concept_score[cidx] = score
        for rxcui in exact_rxcuis:
            cidx = self.rxcui_to_index[rxcui]
            concept_score[cidx] = max(concept_score.get(cidx, 0.0), 1.05)
        ranked = sorted(concept_score.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [
            {
                "rxcui": self.concepts[cidx]["rxcui"],
                "display_name": self.concepts[cidx]["display_name"],
                "tty": self.concepts[cidx]["tty"],
                "score": float(score),
                "exact": self.concepts[cidx]["rxcui"] in exact_rxcuis,
            }
            for cidx, score in ranked
        ]


class RxNormLinker:
    def __init__(
        self,
        phase4_root: Path,
        rxnorm_csv: Path,
        device: str = "auto",
        mode: str = "auto",
    ):
        self.phase4_root = Path(phase4_root)
        self.best_dir = self.phase4_root / "best"
        self.mode_requested = mode
        self.lexical = RxNormLexical(
            rxnorm_csv=Path(rxnorm_csv),
            overrides_json=self.phase4_root / "organizer_overrides.json",
        )
        required_hybrid = [
            self.best_dir / "model.safetensors",
            self.best_dir / "rxnorm_concept_embeddings.npy",
            self.best_dir / "rxnorm_concept_metadata.jsonl",
            self.best_dir / "hybrid_config.json",
        ]
        hybrid_ready = all(p.exists() for p in required_hybrid)
        if mode == "hybrid" and not hybrid_ready:
            raise FileNotFoundError("Phase-4 hybrid requested but its best bundle is incomplete.")
        self.mode = "hybrid" if (mode == "hybrid" or (mode == "auto" and hybrid_ready)) else "lexical"
        self.dense = None
        self.hybrid_alpha = 0.5
        if self.mode == "hybrid":
            cfg = load_json(self.best_dir / "hybrid_config.json", {}) or {}
            self.hybrid_alpha = float(cfg.get("hybrid_alpha", 0.5))
            self.dense = DenseCatalogLinker(
                checkpoint=self.best_dir,
                embeddings_name="rxnorm_concept_embeddings.npy",
                metadata_name="rxnorm_concept_metadata.jsonl",
                id_field="rxcui",
                device=device,
                query_max_length=int(cfg.get("query_max_length", 96)),
                query_prefix=str(cfg.get("query_prefix", "query: ")),
            )

    @staticmethod
    def _norm_scores(cands: List[Dict[str, Any]], id_key: str) -> Dict[str, float]:
        if not cands:
            return {}
        vals = np.array([float(x["score"]) for x in cands], dtype=np.float32)
        lo, hi = float(vals.min()), float(vals.max())
        norm = np.ones_like(vals) if hi - lo < 1e-8 else (vals - lo) / (hi - lo)
        return {str(x[id_key]): float(s) for x, s in zip(cands, norm)}

    def link(self, mentions: List[str], output_k: int = 1) -> List[List[str]]:
        result: List[List[str]] = []
        dense_rows = None
        if self.mode == "hybrid" and self.dense is not None:
            dense_rows = self.dense.retrieve(mentions, top_k=100)
        for i, mention in enumerate(mentions):
            override = self.lexical.override_candidates(mention)
            if override:
                result.append(override[:output_k])
                continue
            lex = self.lexical.retrieve_one(mention, top_k=100)
            if self.mode != "hybrid" or dense_rows is None:
                result.append([str(x["rxcui"]) for x in lex[:output_k]])
                continue
            dense = dense_rows[i]
            lex_map = self._norm_scores(lex, "rxcui")
            dense_map = {x["id"]: s for x, s in zip(dense, self._norm_score_values(dense))}
            union = set(lex_map) | set(dense_map)
            scored = [
                (
                    self.hybrid_alpha * dense_map.get(c, 0.0)
                    + (1.0 - self.hybrid_alpha) * lex_map.get(c, 0.0),
                    dense_map.get(c, 0.0),
                    lex_map.get(c, 0.0),
                    c,
                )
                for c in union
            ]
            scored.sort(reverse=True)
            result.append([x[3] for x in scored[:output_k]])
        return result

    @staticmethod
    def _norm_score_values(cands: List[Dict[str, Any]]) -> List[float]:
        if not cands:
            return []
        vals = np.array([float(x["score"]) for x in cands], dtype=np.float32)
        lo, hi = float(vals.min()), float(vals.max())
        if hi - lo < 1e-8:
            return [1.0] * len(vals)
        return [float(x) for x in ((vals - lo) / (hi - lo))]


# -----------------------------------------------------------------------------
# Full pipeline
# -----------------------------------------------------------------------------
class ClinicalPipeline:
    def __init__(self, args):
        self.args = args
        self.ner = SlidingNER(
            checkpoint=Path(args.phase1_dir),
            device=args.device,
            max_length=args.ner_max_length,
            overlap_tokens=args.ner_overlap_tokens,
            batch_size=args.ner_batch_size,
        )
        self.assertion = AssertionClassifier(
            checkpoint=Path(args.phase2_dir),
            device=args.device,
            max_length=args.assertion_max_length,
            batch_size=args.assertion_batch_size,
        )
        self.icd = ICDLinker(Path(args.phase3_dir), device=args.device)
        self.rxnorm = RxNormLinker(
            phase4_root=Path(args.phase4_root),
            rxnorm_csv=Path(args.rxnorm_csv),
            device=args.device,
            mode=args.phase4_mode,
        )

    def predict(self, text: str) -> List[Dict[str, Any]]:
        entities = self.ner.predict(text)
        assertion_map = self.assertion.predict(text, entities)

        diag_idx = [i for i, e in enumerate(entities) if e["type"] == "CHẨN_ĐOÁN"]
        drug_idx = [i for i, e in enumerate(entities) if e["type"] == "THUỐC"]
        diag_candidates = self.icd.link(
            [entities[i]["text"] for i in diag_idx], output_k=self.args.icd_k
        ) if diag_idx else []
        drug_candidates = self.rxnorm.link(
            [entities[i]["text"] for i in drug_idx], output_k=self.args.rxnorm_k
        ) if drug_idx else []
        diag_map = {idx: c for idx, c in zip(diag_idx, diag_candidates)}
        drug_map = {idx: c for idx, c in zip(drug_idx, drug_candidates)}

        output = []
        for i, e in enumerate(entities):
            s, t = map(int, e["position"])
            if text[s:t] != e["text"]:
                raise RuntimeError(f"Output span invariant failed: {e}")
            row = {
                "text": e["text"],
                "type": e["type"],
            }
            if e["type"] == "CHẨN_ĐOÁN":
                row["candidates"] = diag_map.get(i, [])
            elif e["type"] == "THUỐC":
                row["candidates"] = drug_map.get(i, [])
            if e["type"] in ASSERTION_TYPES:
                row["assertions"] = assertion_map.get(i, [])
            row["position"] = [s, t]
            output.append(row)
        return output


# -----------------------------------------------------------------------------
# Input/output / preflight
# -----------------------------------------------------------------------------
def extract_input_zip(zip_path: Path, temp_root: Path) -> Path:
    temp_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_root)
    candidates = [p for p in temp_root.rglob("*.txt") if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"No .txt files in {zip_path}")
    parents = defaultdict(int)
    for p in candidates:
        parents[p.parent] += 1
    return max(parents, key=parents.get)


def discover_inputs(input_dir: Path) -> List[Path]:
    files = sorted([p for p in input_dir.glob("*.txt") if p.is_file()], key=numeric_sort_key)
    return files


def input_stats(files: List[Path]) -> Dict[str, Any]:
    rows = []
    for p in files:
        text = p.read_text(encoding="utf-8", errors="replace")
        rows.append({"file": p.name, "chars": len(text), "words": len(text.split()), "lines": text.count("\n") + 1})
    if not rows:
        return {"count": 0, "files": []}
    chars = [x["chars"] for x in rows]
    words = [x["words"] for x in rows]
    return {
        "count": len(rows),
        "numeric_names_complete": sorted([p.stem for p in files if p.stem.isdigit()], key=int) == [str(i) for i in range(1, len(files) + 1)],
        "chars": {"min": int(min(chars)), "median": float(np.median(chars)), "max": int(max(chars))},
        "words": {"min": int(min(words)), "median": float(np.median(words)), "max": int(max(words))},
        "files": rows,
    }


def check_bundle(path: Path, required: Sequence[str]) -> Dict[str, Any]:
    return {
        "path": str(path),
        "exists": path.exists(),
        "required": {name: (path / name).exists() for name in required},
        "ready": path.exists() and all((path / name).exists() for name in required),
    }


def preflight_report(args, files: List[Path]) -> Dict[str, Any]:
    phase1 = check_bundle(Path(args.phase1_dir), ["config.json", "model.safetensors", "tokenizer_config.json"])
    phase2 = check_bundle(Path(args.phase2_dir), ["config.json", "model.safetensors", "tokenizer_config.json", "thresholds.json"])
    phase3 = check_bundle(Path(args.phase3_dir), ["config.json", "model.safetensors", "tokenizer_config.json", "icd_embeddings.npy", "icd_metadata.jsonl", "retrieval_config.json"])
    p4best = Path(args.phase4_root) / "best"
    phase4 = check_bundle(p4best, ["config.json", "model.safetensors", "rxnorm_concept_embeddings.npy", "rxnorm_concept_metadata.jsonl", "hybrid_config.json"])
    rx_csv = Path(args.rxnorm_csv)
    p4_override = Path(args.phase4_root) / "organizer_overrides.json"
    stats = input_stats(files)
    return {
        "input": stats,
        "expected_input_count": int(args.expect_files),
        "input_count_ok": stats["count"] == int(args.expect_files),
        "phase1": phase1,
        "phase2": phase2,
        "phase3": phase3,
        "phase4_hybrid": phase4,
        "phase4_lexical_fallback": {
            "rxnorm_csv": str(rx_csv),
            "rxnorm_csv_exists": rx_csv.exists(),
            "organizer_overrides": str(p4_override),
            "organizer_overrides_exists": p4_override.exists(),
            "ready": rx_csv.exists(),
        },
        "phase4_mode_requested": args.phase4_mode,
        "notes": [
            "Long test documents require sliding-window Phase-1 NER; this pipeline does not truncate each document to its first 256 tokens.",
            "When the Phase-4 hybrid best bundle is absent and phase4_mode=auto, RxNorm uses organizer overrides + exact/char-TFIDF lexical fallback.",
            "Competition JSON is emitted only for predictions; run_report.json is kept outside output.zip.",
        ],
    }


def validate_output(text: str, entities: List[Dict[str, Any]]) -> None:
    for e in entities:
        if e.get("type") not in ENTITY_TYPES:
            raise ValueError(f"Invalid type: {e}")
        if "text" not in e or "position" not in e:
            raise ValueError(f"Missing text/position: {e}")
        s, t = map(int, e["position"])
        if not (0 <= s < t <= len(text)) or text[s:t] != e["text"]:
            raise ValueError(f"Invalid raw character span: {e}")
        if e["type"] in ASSERTION_TYPES:
            unknown = set(e.get("assertions", [])) - set(ASSERTION_LABELS)
            if unknown:
                raise ValueError(f"Unknown assertions: {unknown}")
        elif "assertions" in e:
            raise ValueError(f"Assertions must not be emitted for {e['type']}")
        if e["type"] in {"CHẨN_ĐOÁN", "THUỐC"}:
            if not isinstance(e.get("candidates", []), list):
                raise ValueError(f"candidates must be list: {e}")
        elif "candidates" in e:
            raise ValueError(f"Candidates must not be emitted for {e['type']}")


def make_output_zip(output_dir: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(output_dir.glob("*.json"), key=numeric_sort_key):
            zf.write(p, arcname=f"output/{p.name}")


def parse_args():
    p = argparse.ArgumentParser(description="Phase-5 end-to-end clinical NLP inference")

    # Colab-safe: input is no longer a required CLI argument. If neither option
    # is supplied, main() resolves a sensible Drive default automatically.
    src = p.add_mutually_exclusive_group(required=False)
    src.add_argument("--input_dir", type=str, default=None)
    src.add_argument("--input_zip", type=str, default=None)
    p.add_argument("--output_dir", type=str, default="/content/phase5_submission/output")
    p.add_argument("--output_zip", type=str, default="/content/phase5_submission/output.zip")
    p.add_argument("--report_json", type=str, default="/content/phase5_submission/run_report.json")
    p.add_argument("--expect_files", type=int, default=100)

    p.add_argument("--phase1_dir", type=str, default=str(PHASE1_DIR))
    p.add_argument("--phase2_dir", type=str, default=str(PHASE2_DIR))
    p.add_argument("--phase3_dir", type=str, default=str(PHASE3_DIR))
    p.add_argument("--phase4_root", type=str, default=str(PHASE4_ROOT))
    p.add_argument("--rxnorm_csv", type=str, default=str(RXNORM_CSV))
    p.add_argument("--phase4_mode", choices=["auto", "hybrid", "lexical"], default="auto")

    p.add_argument("--device", type=str, default="auto")
    p.add_argument("--ner_max_length", type=int, default=256)
    p.add_argument("--ner_overlap_tokens", type=int, default=64)
    p.add_argument("--ner_batch_size", type=int, default=12)
    p.add_argument("--assertion_max_length", type=int, default=256)
    p.add_argument("--assertion_batch_size", type=int, default=32)
    p.add_argument("--icd_k", type=int, default=1)
    p.add_argument("--rxnorm_k", type=int, default=1)
    p.add_argument("--preflight_only", action="store_true")
    p.add_argument("--overwrite", action="store_true")

    # IPython/Colab injects arguments such as ``-f <kernel.json>``. Using
    # parse_known_args prevents those notebook-only arguments from terminating
    # the pipeline with SystemExit.
    args, unknown = p.parse_known_args()
    if unknown:
        print("[INFO] Ignoring Colab/Jupyter arguments:", unknown)
    return args


def resolve_default_input(args):
    """Resolve input when running the whole script directly inside Colab."""
    if args.input_zip:
        return "zip", Path(args.input_zip)
    if args.input_dir:
        return "dir", Path(args.input_dir)

    # Prefer the exact file used in this project, but support the common Data
    # location too so the user can move the ZIP without editing this script.
    for candidate in (DEFAULT_INPUT_ZIP, DEFAULT_INPUT_ZIP_DATA):
        if candidate.is_file():
            print(f"[COLAB] Auto-detected input ZIP: {candidate}")
            return "zip", candidate

    if DEFAULT_INPUT_DIR.is_dir():
        print(f"[COLAB] Auto-detected input directory: {DEFAULT_INPUT_DIR}")
        return "dir", DEFAULT_INPUT_DIR

    raise FileNotFoundError(
        "No test input was found. Expected one of:\n"
        f"  {DEFAULT_INPUT_ZIP}\n"
        f"  {DEFAULT_INPUT_ZIP_DATA}\n"
        f"  {DEFAULT_INPUT_DIR}\n\n"
        "Upload/move input_turn2_vong1.zip to one of those Drive paths, "
        "or set --input_zip / --input_dir when running as a script."
    )


def main():
    args = parse_args()

    # Friendly Colab check before loading any large model.
    if not DRIVE_ROOT.exists():
        raise FileNotFoundError(
            "Google Drive is not mounted at /content/drive/MyDrive. Run first:\n"
            "from google.colab import drive\n"
            "drive.mount('/content/drive')"
        )

    input_kind, input_path = resolve_default_input(args)
    temp_root = Path("/content/phase5_input_extract")
    if input_kind == "zip":
        input_dir = extract_input_zip(input_path, temp_root)
        args.input_zip = str(input_path)
    else:
        input_dir = input_path
        args.input_dir = str(input_path)
    files = discover_inputs(input_dir)
    report = preflight_report(args, files)
    report["input_dir"] = str(input_dir)
    report["started_at_unix"] = time.time()
    save_json(report, Path(args.report_json))

    print(json.dumps({
        "input_count": report["input"]["count"],
        "input_count_ok": report["input_count_ok"],
        "phase1_ready": report["phase1"]["ready"],
        "phase2_ready": report["phase2"]["ready"],
        "phase3_ready": report["phase3"]["ready"],
        "phase4_hybrid_ready": report["phase4_hybrid"]["ready"],
        "phase4_lexical_ready": report["phase4_lexical_fallback"]["ready"],
    }, ensure_ascii=False, indent=2))

    if not report["input_count_ok"]:
        raise RuntimeError(f"Expected {args.expect_files} .txt files, found {len(files)}")
    if args.preflight_only:
        return

    # Phase 1-3 are mandatory trained components. Phase 4 can use the explicit lexical fallback.
    for key in ("phase1", "phase2", "phase3"):
        if not report[key]["ready"]:
            raise FileNotFoundError(f"{key} bundle is incomplete: {report[key]}")
    if args.phase4_mode == "hybrid" and not report["phase4_hybrid"]["ready"]:
        raise FileNotFoundError("Phase-4 hybrid best bundle is incomplete.")
    if args.phase4_mode in {"auto", "lexical"} and not (
        report["phase4_hybrid"]["ready"] or report["phase4_lexical_fallback"]["ready"]
    ):
        raise FileNotFoundError("Neither Phase-4 hybrid nor lexical fallback is available.")

    output_dir = Path(args.output_dir)
    if output_dir.exists() and args.overwrite:
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pipeline = ClinicalPipeline(args)
    report["effective_phase4_mode"] = pipeline.rxnorm.mode
    failures = []
    entity_counts = []

    for n, path in enumerate(files, 1):
        text = path.read_text(encoding="utf-8", errors="replace")
        try:
            entities = pipeline.predict(text)
            validate_output(text, entities)
            save_json(entities, output_dir / f"{path.stem}.json")
            entity_counts.append(len(entities))
            print(f"[{n:03d}/{len(files):03d}] {path.name}: {len(entities)} entities")
        except Exception as exc:
            failures.append({"file": path.name, "error": repr(exc)})
            print(f"[ERROR] {path.name}: {exc}", file=sys.stderr)
            raise

    make_output_zip(output_dir, Path(args.output_zip))
    report["completed"] = True
    report["failures"] = failures
    report["prediction_files"] = len(list(output_dir.glob("*.json")))
    report["entity_counts"] = {
        "total": int(sum(entity_counts)),
        "min": int(min(entity_counts)) if entity_counts else 0,
        "median": float(np.median(entity_counts)) if entity_counts else 0.0,
        "max": int(max(entity_counts)) if entity_counts else 0,
    }
    report["output_dir"] = str(output_dir)
    report["output_zip"] = str(args.output_zip)
    report["finished_at_unix"] = time.time()
    save_json(report, Path(args.report_json))
    print(f"Saved submission: {args.output_zip}")


if __name__ == "__main__":
    main()

[INFO] Ignoring Colab/Jupyter arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-d05d07bd-8c74-49a0-903b-0a89a05fe25b.json']
[COLAB] Auto-detected input directory: /content/drive/MyDrive/input
{
  "input_count": 100,
  "input_count_ok": true,
  "phase1_ready": true,
  "phase2_ready": true,
  "phase3_ready": true,
  "phase4_hybrid_ready": true,
  "phase4_lexical_ready": true
}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[001/100] 1.txt: 208 entities
[002/100] 2.txt: 151 entities
[003/100] 3.txt: 129 entities
[004/100] 4.txt: 121 entities
[005/100] 5.txt: 113 entities
[006/100] 6.txt: 108 entities
[007/100] 7.txt: 139 entities
[008/100] 8.txt: 119 entities
[009/100] 9.txt: 128 entities
[010/100] 10.txt: 107 entities
[011/100] 11.txt: 88 entities
[012/100] 12.txt: 80 entities
[013/100] 13.txt: 148 entities
[014/100] 14.txt: 61 entities
[015/100] 15.txt: 72 entities
[016/100] 16.txt: 141 entities
[017/100] 17.txt: 158 entities
[018/100] 18.txt: 102 entities
[019/100] 19.txt: 65 entities
[020/100] 20.txt: 138 entities
[021/100] 21.txt: 92 entities
[022/100] 22.txt: 89 entities
[023/100] 23.txt: 90 entities
[024/100] 24.txt: 96 entities
[025/100] 25.txt: 93 entities
[026/100] 26.txt: 81 entities
[027/100] 27.txt: 97 entities
[028/100] 28.txt: 59 entities
[029/100] 29.txt: 70 entities
[030/100] 30.txt: 65 entities
[031/100] 31.txt: 70 entities
[032/100] 32.txt: 84 entities
[033/100] 33.txt: 70 entities
[034